In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd, numpy as np, os, glob

DATA_DIR = '/content/drive/MyDrive/UNSW-NB15'
cols = ['srcip','sport','dstip','dsport','proto','state','dur','sbytes','dbytes','sttl','dttl',
        'sloss','dloss','service','Sload','Dload','Spkts','Dpkts','swin','dwin','stcpb','dtcpb',
        'smeansz','dmeansz','trans_depth','res_bdy_len','Sjit','Djit','Stime','Ltime','Sintpkt',
        'Dintpkt','tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl','ct_flw_http_mthd',
        'is_ftp_login','ct_ftp_cmd','ct_srv_src','ct_srv_dst','ct_dst_ltm','ct_src_ltm',
        'ct_src_dport_ltm','ct_dst_sport_ltm','ct_dst_src_ltm','attack_cat','Label']
assert len(cols) == 49

# search the folder and every subfolder (e.g. "CSV Files")
files = sorted(glob.glob(os.path.join(DATA_DIR, '**', 'UNSW-NB15_[1-4].csv'), recursive=True))
print('Found:', files)
assert len(files) == 4, 'Expected 4 files UNSW-NB15_1..4.csv under ' + DATA_DIR

df = pd.concat([pd.read_csv(f, header=None, names=cols, low_memory=False) for f in files],
               ignore_index=True)
print('Rows:', len(df), '| columns:', df.shape[1])

# Clean labels
df['attack_cat'] = (df['attack_cat'].fillna('Normal').astype(str).str.strip()
                    .replace({'Backdoor': 'Backdoors', '': 'Normal'}))
df['Label'] = pd.to_numeric(df['Label'], errors='coerce').fillna(0).astype(int)
df['Stime'] = pd.to_numeric(df['Stime'], errors='coerce')
df = df.dropna(subset=['Stime']).sort_values('Stime', kind='mergesort').reset_index(drop=True)

# Time range
t0, t1 = pd.to_datetime(df['Stime'].min(), unit='s'), pd.to_datetime(df['Stime'].max(), unit='s')
print('Time range (UTC):', t0, '->', t1)
print('\nClass counts:\n', df['attack_cat'].value_counts())
print('\nAttack share:', round(df['Label'].mean(), 4))

# Rho per chronological block (5 equal-count blocks; train = all earlier blocks)
K = 5
df['block'] = pd.qcut(np.arange(len(df)), K, labels=False)
print('\nBlock time spans:')
for k in range(K):
    b = df[df.block == k]
    print(k+1, pd.to_datetime(b.Stime.min(), unit='s'), '->', pd.to_datetime(b.Stime.max(), unit='s'),
          '| rows', len(b), '| attack share', round(b.Label.mean(), 4))

rows = []
for k in range(1, K):
    seen = set(df[df.block < k]['attack_cat'])
    te = df[df.block == k]
    novel = ~te['attack_cat'].isin(seen)
    att = te['Label'] == 1
    rho = novel.mean()
    rho_A = (novel & att).sum() / max(att.sum(), 1)
    rows.append(dict(test_block=k+1, rho=round(rho, 4), rho_A=round(rho_A, 4),
                     novel_classes=sorted(set(te['attack_cat']) - seen)))
print('\nNovel-class fraction per chronological split:')
print(pd.DataFrame(rows).to_string(index=False))

Found: ['/content/drive/MyDrive/UNSW-NB15/CSV Files/UNSW-NB15_1.csv', '/content/drive/MyDrive/UNSW-NB15/CSV Files/UNSW-NB15_2.csv', '/content/drive/MyDrive/UNSW-NB15/CSV Files/UNSW-NB15_3.csv', '/content/drive/MyDrive/UNSW-NB15/CSV Files/UNSW-NB15_4.csv']
Rows: 2540047 | columns: 49
Time range (UTC): 2015-01-22 11:49:37 -> 2015-02-18 12:21:08

Class counts:
 attack_cat
Normal            2218764
Generic            215481
Exploits            44525
Fuzzers             24246
DoS                 16353
Reconnaissance      13987
Analysis             2677
Backdoors            2329
Shellcode            1511
Worms                 174
Name: count, dtype: int64

Attack share: 0.1265

Block time spans:
1 2015-01-22 11:49:37 -> 2015-01-22 17:08:30 | rows 508010 | attack share 0.0437
2 2015-01-22 17:08:30 -> 2015-01-22 23:34:56 | rows 508009 | attack share 0.0
3 2015-01-22 23:34:56 -> 2015-02-18 04:27:43 | rows 508009 | attack share 0.1708
4 2015-02-18 04:27:43 -> 2015-02-18 08:22:37 | rows 508009 | 

In [ ]:
import pandas as pd, numpy as np, os, glob, time, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, recall_score, accuracy_score, roc_auc_score, confusion_matrix
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from scipy import stats

DATA_DIR = '/content/drive/MyDrive/UNSW-NB15'
CACHE = os.path.join(DATA_DIR, 'unsw_clean.parquet')
RES   = os.path.join(DATA_DIR, 'unsw_results.csv')
SEEDS = list(range(42, 52))
N_TRAIN, N_TEST = 200_000, 100_000

# ---------- load / clean (cached) ----------
if os.path.exists(CACHE):
    df = pd.read_parquet(CACHE)
    print('Loaded cache:', df.shape)
else:
    cols = ['srcip','sport','dstip','dsport','proto','state','dur','sbytes','dbytes','sttl','dttl',
            'sloss','dloss','service','Sload','Dload','Spkts','Dpkts','swin','dwin','stcpb','dtcpb',
            'smeansz','dmeansz','trans_depth','res_bdy_len','Sjit','Djit','Stime','Ltime','Sintpkt',
            'Dintpkt','tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl','ct_flw_http_mthd',
            'is_ftp_login','ct_ftp_cmd','ct_srv_src','ct_srv_dst','ct_dst_ltm','ct_src_ltm',
            'ct_src_dport_ltm','ct_dst_sport_ltm','ct_dst_src_ltm','attack_cat','Label']
    files = sorted(glob.glob(os.path.join(DATA_DIR, '**', 'UNSW-NB15_[1-4].csv'), recursive=True))
    assert len(files) == 4
    df = pd.concat([pd.read_csv(f, header=None, names=cols, low_memory=False) for f in files],
                   ignore_index=True)
    df['attack_cat'] = (df['attack_cat'].fillna('Normal').astype(str).str.strip()
                        .replace({'Backdoor': 'Backdoors', '': 'Normal'}))
    df['Label'] = pd.to_numeric(df['Label'], errors='coerce').fillna(0).astype('int8')
    df['Stime'] = pd.to_numeric(df['Stime'], errors='coerce')
    df = df.dropna(subset=['Stime']).sort_values('Stime', kind='mergesort').reset_index(drop=True)
    df = df.drop(columns=['srcip','dstip','sport','dsport','Ltime'])   # identifiers / leakage
    for c in ['proto','state','service']:
        df[c] = df[c].astype(str).str.strip()
    num = [c for c in df.columns if c not in ['proto','state','service','attack_cat','Label','Stime']]
    for c in num:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')
    df.to_parquet(CACHE)
    print('Cleaned and cached:', df.shape)

CAT = ['proto', 'state', 'service']
NUM = [c for c in df.columns if c not in CAT + ['Stime', 'attack_cat', 'Label']]

# ---------- helpers ----------
def strat_sample(d, n, seed):
    frac = min(1.0, n / len(d))
    return d.groupby('attack_cat', group_keys=False).sample(frac=frac, random_state=seed)

def make_xy(tr, te):
    maps = {c: {v: i for i, v in enumerate(sorted(tr[c].unique()))} for c in CAT}
    def enc(d):
        X = d[NUM].copy()
        for c in CAT:
            X[c] = d[c].map(maps[c]).fillna(-1).astype('float32')
        return X.replace([np.inf, -np.inf], np.nan)
    Xtr, Xte = enc(tr), enc(te)
    med = Xtr.median()
    Xtr, Xte = Xtr.fillna(med), Xte.fillna(med)
    keep = [c for c in Xtr.columns if Xtr[c].var() >= 0.01]
    corr = Xtr[keep].corr().abs().values
    drop = set()
    for l in range(len(keep)):
        for j in range(l):
            if keep[j] not in drop and corr[j, l] >= 0.95:
                drop.add(keep[l]); break
    keep = [c for c in keep if c not in drop]
    return Xtr[keep].values, Xte[keep].values, len(keep)

def get_split(protocol, seed):
    if protocol == 'random':
        S = strat_sample(df, N_TRAIN + N_TEST, seed)
        tr, te = train_test_split(S, test_size=N_TEST / (N_TRAIN + N_TEST),
                                  stratify=S['attack_cat'], random_state=seed)
    else:
        cut = int(0.8 * len(df))
        tr = strat_sample(df.iloc[:cut], N_TRAIN, seed)
        te = strat_sample(df.iloc[cut:], N_TEST, seed)
    return tr, te

def models(seed):
    return {
        'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced',
                                               n_jobs=-1, random_state=seed),
        'LightGBM': LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8,
                                   subsample=0.8, class_weight='balanced', random_state=seed,
                                   n_jobs=-1, verbose=-1),
        'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, colsample_bytree=0.8, subsample=0.8,
                                 eval_metric='logloss', tree_method='hist', random_state=seed, n_jobs=-1),
    }

# ---------- run (resumable) ----------
res = pd.read_csv(RES) if os.path.exists(RES) else pd.DataFrame()
done = set(zip(res.seed, res.model, res.protocol)) if len(res) else set()

for seed in SEEDS:
    t0 = time.time()
    for protocol in ['random', 'temporal']:
        todo = [m for m in ['RandomForest', 'LightGBM', 'XGBoost'] if (seed, m, protocol) not in done]
        if not todo:
            continue
        tr, te = get_split(protocol, seed)
        rho = (~te['attack_cat'].isin(set(tr['attack_cat']))).mean()
        Xtr, Xte, nf = make_xy(tr, te)
        ytr, yte = tr['Label'].values, te['Label'].values
        for name, mdl in models(seed).items():
            if name not in todo:
                continue
            mdl.fit(Xtr, ytr)
            p = mdl.predict(Xte); pr = mdl.predict_proba(Xte)[:, 1]
            tn, fp, fn, tp = confusion_matrix(yte, p, labels=[0, 1]).ravel()
            row = dict(seed=seed, model=name, protocol=protocol, n_feat=nf, rho=round(float(rho), 5),
                       f1_macro=f1_score(yte, p, average='macro'), recall=recall_score(yte, p),
                       fpr=fp / max(fp + tn, 1), acc=accuracy_score(yte, p), auc=roc_auc_score(yte, pr))
            res = pd.concat([res, pd.DataFrame([row])], ignore_index=True)
            res.to_csv(RES, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows saved: {len(res)}')

# ---------- summary ----------
def cliffs(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return (sum(x > y for x in a for y in b) - sum(x < y for x in a for y in b)) / (len(a) * len(b))

out = []
for m in ['RandomForest', 'LightGBM', 'XGBoost']:
    for metric in ['f1_macro', 'recall', 'fpr', 'acc', 'auc']:
        r = res[(res.model == m) & (res.protocol == 'random')].set_index('seed')[metric].sort_index()
        t = res[(res.model == m) & (res.protocol == 'temporal')].set_index('seed')[metric].sort_index()
        common = r.index.intersection(t.index)
        if len(common) < 3: continue
        r, t = r.loc[common], t.loc[common]
        gap = (r - t) * 100
        n = len(gap); half = stats.t.ppf(0.975, n - 1) * gap.std(ddof=1) / np.sqrt(n)
        try: pw = stats.wilcoxon(r, t).pvalue
        except Exception: pw = np.nan
        out.append(dict(model=m, metric=metric, n=n, random=round(r.mean(), 4), temporal=round(t.mean(), 4),
                        gap_pp=round(gap.mean(), 2), ci_lo=round(gap.mean() - half, 2),
                        ci_hi=round(gap.mean() + half, 2), wilcoxon_p=round(pw, 5),
                        cliffs_d=round(cliffs(r, t), 3)))
print('\n=== Random vs temporal (gap = random - temporal, percentage points) ===')
print(pd.DataFrame(out).to_string(index=False))
print('\nMean rho (temporal protocol, sampled):', res[res.protocol == 'temporal'].rho.mean())
print('Features kept (typical):', res.n_feat.value_counts().to_dict())

Cleaned and cached: (2540047, 44)
seed 42 done in 77s | rows saved: 6
seed 43 done in 76s | rows saved: 12
seed 44 done in 72s | rows saved: 18
seed 45 done in 76s | rows saved: 24
seed 46 done in 75s | rows saved: 30
seed 47 done in 70s | rows saved: 36
seed 48 done in 77s | rows saved: 42
seed 49 done in 73s | rows saved: 48
seed 50 done in 70s | rows saved: 54
seed 51 done in 75s | rows saved: 60

=== Random vs temporal (gap = random - temporal, percentage points) ===
       model   metric  n  random  temporal  gap_pp  ci_lo  ci_hi  wilcoxon_p  cliffs_d
RandomForest f1_macro 10  0.9735    0.9716    0.19   0.12   0.26     0.00195      1.00
RandomForest   recall 10  0.9997    0.9996    0.01  -0.01   0.03     0.18555      0.34
RandomForest      fpr 10  0.0139    0.0228   -0.89  -0.93  -0.84     0.00195     -1.00
RandomForest      acc 10  0.9878    0.9816    0.63   0.59   0.66     0.00195      1.00
RandomForest      auc 10  0.9996    0.9993    0.03   0.03   0.04     0.00195      1.00
  

In [ ]:
import pandas as pd, numpy as np, os, time, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score
from lightgbm import LGBMClassifier
from scipy import stats

DATA_DIR = '/content/drive/MyDrive/UNSW-NB15'
CACHE = os.path.join(DATA_DIR, 'unsw_clean.parquet')
RES2  = os.path.join(DATA_DIR, 'unsw_dose_results.csv')
assert os.path.exists(CACHE), 'Run the previous experiment cell first (it creates the cache).'
SEEDS = list(range(42, 52))
N_TRAIN, N_TEST = 200_000, 100_000

df = pd.read_parquet(CACHE)
CAT = ['proto', 'state', 'service']
NUM = [c for c in df.columns if c not in CAT + ['Stime', 'attack_cat', 'Label']]

def strat_sample(d, n, seed):
    frac = min(1.0, n / len(d))
    return d.groupby('attack_cat', group_keys=False).sample(frac=frac, random_state=seed)

def make_xy(tr, te):
    maps = {c: {v: i for i, v in enumerate(sorted(tr[c].unique()))} for c in CAT}
    def enc(d):
        X = d[NUM].copy()
        for c in CAT:
            X[c] = d[c].map(maps[c]).fillna(-1).astype('float32')
        return X.replace([np.inf, -np.inf], np.nan)
    Xtr, Xte = enc(tr), enc(te)
    med = Xtr.median()
    Xtr, Xte = Xtr.fillna(med), Xte.fillna(med)
    keep = [c for c in Xtr.columns if Xtr[c].var() >= 0.01]
    corr = Xtr[keep].corr().abs().values
    drop = set()
    for l in range(len(keep)):
        for j in range(l):
            if keep[j] not in drop and corr[j, l] >= 0.95:
                drop.add(keep[l]); break
    keep = [c for c in keep if c not in drop]
    return Xtr[keep].values, Xte[keep].values

cut = int(0.8 * len(df))
pool_tr, pool_te = df.iloc[:cut], df.iloc[cut:]

# order attack families by frequency in the test period (rarest first)
fam_te = pool_te[pool_te.Label == 1]['attack_cat'].value_counts().sort_values()
fam_tr = set(pool_tr[pool_tr.Label == 1]['attack_cat'])
order = [f for f in fam_te.index if f in fam_tr]
levels = list(range(0, len(order) - 1))          # keep at least 2 attack families in training
print('Family removal order (rarest first):', order)
print('Levels (families removed):', levels)

res = pd.read_csv(RES2) if os.path.exists(RES2) else pd.DataFrame()
done = set(zip(res.seed, res.level)) if len(res) else set()

for seed in SEEDS:
    t0 = time.time()
    te = strat_sample(pool_te, N_TEST, seed)
    yte = te['Label'].values
    att = yte == 1
    for m in levels:
        if (seed, m) in done:
            continue
        H = set(order[:m])
        tr = strat_sample(pool_tr[~pool_tr['attack_cat'].isin(H)], N_TRAIN, seed)
        novel = te['attack_cat'].isin(H).values
        rho = novel.mean()
        Xtr, Xte = make_xy(tr, te)
        ytr = tr['Label'].values
        rows = []
        for name, mdl in [
            ('RandomForest', RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced',
                                                    n_jobs=-1, random_state=seed)),
            ('LightGBM', LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8,
                                        subsample=0.8, class_weight='balanced', random_state=seed,
                                        n_jobs=-1, verbose=-1))]:
            mdl.fit(Xtr, ytr)
            p = mdl.predict(Xte)
            seen_att, nov_att = att & ~novel, att & novel
            rows.append(dict(seed=seed, level=m, model=name, rho=rho,
                             f1_macro=f1_score(yte, p, average='macro'),
                             recall_all=recall_score(yte, p),
                             recall_seen=(p[seen_att] == 1).mean() if seen_att.sum() else np.nan,
                             recall_novel=(p[nov_att] == 1).mean() if nov_att.sum() else np.nan,
                             acc=np.nan, bound=np.nan))
        # 10-class closed-set RF -> check Proposition 1: accuracy <= 1 - rho
        classes = sorted(tr['attack_cat'].unique()); cmap = {c: i for i, c in enumerate(classes)}
        mc = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced',
                                    n_jobs=-1, random_state=seed)
        mc.fit(Xtr, tr['attack_cat'].map(cmap).values)
        true = te['attack_cat'].map(cmap).fillna(-1).values
        acc = (mc.predict(Xte) == true).mean()
        rows.append(dict(seed=seed, level=m, model='RF-multiclass', rho=rho, f1_macro=np.nan,
                         recall_all=np.nan, recall_seen=np.nan, recall_novel=np.nan,
                         acc=acc, bound=1 - rho))
        res = pd.concat([res, pd.DataFrame(rows)], ignore_index=True)
        res.to_csv(RES2, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows: {len(res)}')

# ---------- summary ----------
print('\n=== Binary models: effect of removing families from training ===')
for mname in ['RandomForest', 'LightGBM']:
    d = res[res.model == mname]
    base = d[d.level == 0].set_index('seed')['f1_macro']
    out = []
    for m in levels:
        dm = d[d.level == m].set_index('seed')
        common = base.index.intersection(dm.index)
        gap = (base.loc[common] - dm.loc[common, 'f1_macro']) * 100
        n = len(gap)
        h = stats.t.ppf(0.975, n - 1) * gap.std(ddof=1) / np.sqrt(n) if n > 2 else np.nan
        out.append(dict(level=m, rho=round(dm.rho.mean(), 4), f1_macro=round(dm.f1_macro.mean(), 4),
                        drop_pp=round(gap.mean(), 2), ci=f'{gap.mean()-h:.2f} to {gap.mean()+h:.2f}',
                        recall_seen=round(dm.recall_seen.mean(), 4), recall_novel=round(dm.recall_novel.mean(), 4)))
    print('\n', mname); print(pd.DataFrame(out).to_string(index=False))
    r, p = stats.spearmanr(d.rho, d.f1_macro)
    print(f'Spearman(rho, F1-macro) over all runs: {r:.3f} (p={p:.3g})')

print('\n=== Proposition 1 check: 10-class RF accuracy vs bound 1 - rho ===')
mc = res[res.model == 'RF-multiclass']
summ = mc.groupby('level').agg(rho=('rho', 'mean'), acc=('acc', 'mean'), bound=('bound', 'mean'),
                              max_excess=('acc', lambda s: 0)).round(4)
summ['max_excess'] = mc.assign(ex=mc.acc - mc.bound).groupby('level').ex.max().round(5)
print(summ.to_string())
print('Runs violating acc <= 1 - rho:', int((mc.acc > mc.bound + 1e-9).sum()), 'of', len(mc))

Family removal order (rarest first): ['Worms', 'Shellcode', 'Analysis', 'Backdoors', 'Reconnaissance', 'DoS', 'Fuzzers', 'Exploits', 'Generic']
Levels (families removed): [0, 1, 2, 3, 4, 5, 6, 7]
seed 42 done in 447s | rows: 24
seed 43 done in 432s | rows: 48
seed 44 done in 446s | rows: 72
seed 45 done in 436s | rows: 96
seed 46 done in 440s | rows: 120
seed 47 done in 430s | rows: 144
seed 48 done in 437s | rows: 168
seed 49 done in 430s | rows: 192
seed 50 done in 431s | rows: 216
seed 51 done in 422s | rows: 240

=== Binary models: effect of removing families from training ===

 RandomForest
 level    rho  f1_macro  drop_pp             ci  recall_seen  recall_novel
     0 0.0000    0.9716     0.00   0.00 to 0.00       0.9996           NaN
     1 0.0001    0.9716    -0.00  -0.00 to 0.00       0.9996        0.9700
     2 0.0010    0.9716     0.00  -0.00 to 0.01       0.9996        0.9959
     3 0.0023    0.9727    -0.11 -0.14 to -0.09       0.9993        0.9583
     4 0.0037    0.972

In [ ]:
# ===== VARIANT A (new results file): temporal baseline at other chronological cut points =====
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, os, time, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score, confusion_matrix
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from scipy import stats

DATA_DIR = '/content/drive/MyDrive/UNSW-NB15'
CACHE = os.path.join(DATA_DIR, 'unsw_clean.parquet')
BASE  = os.path.join(DATA_DIR, 'unsw_results.csv')        # original 80/20 results (read only)
RES3  = os.path.join(DATA_DIR, 'unsw_cut_results.csv')    # new file
SEEDS = list(range(42, 52)); CUTS = [0.6, 0.7, 0.9]
N_TRAIN, N_TEST = 200_000, 100_000
df = pd.read_parquet(CACHE)
CAT = ['proto', 'state', 'service']
NUM = [c for c in df.columns if c not in CAT + ['Stime', 'attack_cat', 'Label']]

def strat_sample(d, n, seed):
    frac = min(1.0, n / len(d))
    return d.groupby('attack_cat', group_keys=False).sample(frac=frac, random_state=seed)

def make_xy(tr, te):
    maps = {c: {v: i for i, v in enumerate(sorted(tr[c].unique()))} for c in CAT}
    def enc(d):
        X = d[NUM].copy()
        for c in CAT:
            X[c] = d[c].map(maps[c]).fillna(-1).astype('float32')
        return X.replace([np.inf, -np.inf], np.nan)
    Xtr, Xte = enc(tr), enc(te)
    med = Xtr.median()
    Xtr, Xte = Xtr.fillna(med), Xte.fillna(med)
    keep = [c for c in Xtr.columns if Xtr[c].var() >= 0.01]
    corr = Xtr[keep].corr().abs().values
    drop = set()
    for l in range(len(keep)):
        for j in range(l):
            if keep[j] not in drop and corr[j, l] >= 0.95:
                drop.add(keep[l]); break
    keep = [c for c in keep if c not in drop]
    return Xtr[keep].values, Xte[keep].values

def models(seed):
    return {
        'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed),
        'LightGBM': LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8,
                                   class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1),
        'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, colsample_bytree=0.8, subsample=0.8, eval_metric='logloss',
                                 tree_method='hist', random_state=seed, n_jobs=-1)}

res = pd.read_csv(RES3) if os.path.exists(RES3) else pd.DataFrame()
done = set(zip(res.seed, res.model, res.cut)) if len(res) else set()
for seed in SEEDS:
    t0 = time.time()
    for cutf in CUTS:
        todo = [m for m in ['RandomForest', 'LightGBM', 'XGBoost'] if (seed, m, cutf) not in done]
        if not todo:
            continue
        cut = int(cutf * len(df))
        tr = strat_sample(df.iloc[:cut], N_TRAIN, seed); te = strat_sample(df.iloc[cut:], N_TEST, seed)
        rho = (~te['attack_cat'].isin(set(tr['attack_cat']))).mean()
        Xtr, Xte = make_xy(tr, te)
        ytr, yte = tr['Label'].values, te['Label'].values
        for name, mdl in models(seed).items():
            if name not in todo:
                continue
            mdl.fit(Xtr, ytr); p = mdl.predict(Xte)
            tn, fp, fn, tp = confusion_matrix(yte, p, labels=[0, 1]).ravel()
            row = dict(seed=seed, model=name, cut=cutf, rho=float(rho), attack_share_test=float(yte.mean()),
                       f1_macro=f1_score(yte, p, average='macro'), recall=recall_score(yte, p), fpr=fp / max(fp + tn, 1))
            res = pd.concat([res, pd.DataFrame([row])], ignore_index=True)
            res.to_csv(RES3, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows: {len(res)}')

base = pd.read_csv(BASE)
out = []
for m in ['RandomForest', 'LightGBM', 'XGBoost']:
    r = base[(base.model == m) & (base.protocol == 'random')].set_index('seed')['f1_macro'].sort_index()
    for cutf in [0.6, 0.7, 0.8, 0.9]:
        if cutf == 0.8:
            t = base[(base.model == m) & (base.protocol == 'temporal')].set_index('seed')['f1_macro'].sort_index()
        else:
            t = res[(res.model == m) & (res.cut == cutf)].set_index('seed')['f1_macro'].sort_index()
        common = r.index.intersection(t.index)
        if len(common) < 3: continue
        gap = (r.loc[common] - t.loc[common]) * 100
        n = len(gap); h = stats.t.ppf(0.975, n - 1) * gap.std(ddof=1) / np.sqrt(n)
        out.append(dict(model=m, cut=cutf, n=n, f1_random=round(r.loc[common].mean(), 4), f1_temporal=round(t.loc[common].mean(), 4),
                        gap_pp=round(gap.mean(), 2), ci=f'{gap.mean()-h:.2f} to {gap.mean()+h:.2f}',
                        wilcoxon_p=round(stats.wilcoxon(r.loc[common], t.loc[common]).pvalue, 5)))
print('\n=== VARIANT A: F1-macro gap (random - temporal), by chronological cut ===')
print(pd.DataFrame(out).to_string(index=False))
print('\nMean rho by cut:', res.groupby('cut').rho.mean().round(5).to_dict())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
seed 42 done in 125s | rows: 9
seed 43 done in 109s | rows: 18
seed 44 done in 106s | rows: 27
seed 45 done in 105s | rows: 36
seed 46 done in 110s | rows: 45
seed 47 done in 109s | rows: 54
seed 48 done in 110s | rows: 63
seed 49 done in 108s | rows: 72
seed 50 done in 106s | rows: 81
seed 51 done in 103s | rows: 90

=== VARIANT A: F1-macro gap (random - temporal), by chronological cut ===
       model  cut  n  f1_random  f1_temporal  gap_pp            ci  wilcoxon_p
RandomForest  0.6 10     0.9735       0.9728    0.07  0.01 to 0.14     0.01367
RandomForest  0.7 10     0.9735       0.9718    0.18  0.12 to 0.23     0.00195
RandomForest  0.8 10     0.9735       0.9716    0.19  0.12 to 0.26     0.00195
RandomForest  0.9 10     0.9735       0.9731    0.04 -0.02 to 0.11     0.13086
    LightGBM  0.6 10     0.9756       0.9739    0.17  0.10 to 0.23     0.00195
   

In [ ]:
# ===== VARIANT B (new results file): hold out the LARGEST attack families first, to reach a wide rho range =====
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, os, time, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score
from lightgbm import LGBMClassifier
from scipy import stats

DATA_DIR = '/content/drive/MyDrive/UNSW-NB15'
CACHE = os.path.join(DATA_DIR, 'unsw_clean.parquet')
PREV  = os.path.join(DATA_DIR, 'unsw_dose_results.csv')        # original dose results (read only; level 0 = baseline)
RES4  = os.path.join(DATA_DIR, 'unsw_dose_large_results.csv')  # new file
SEEDS = list(range(42, 47))                                    # 5 seeds (same seeds/test samples as the original run)
N_TRAIN, N_TEST = 200_000, 100_000
df = pd.read_parquet(CACHE)
CAT = ['proto', 'state', 'service']
NUM = [c for c in df.columns if c not in CAT + ['Stime', 'attack_cat', 'Label']]

def strat_sample(d, n, seed):
    frac = min(1.0, n / len(d))
    return d.groupby('attack_cat', group_keys=False).sample(frac=frac, random_state=seed)

def make_xy(tr, te):
    maps = {c: {v: i for i, v in enumerate(sorted(tr[c].unique()))} for c in CAT}
    def enc(d):
        X = d[NUM].copy()
        for c in CAT:
            X[c] = d[c].map(maps[c]).fillna(-1).astype('float32')
        return X.replace([np.inf, -np.inf], np.nan)
    Xtr, Xte = enc(tr), enc(te)
    med = Xtr.median()
    Xtr, Xte = Xtr.fillna(med), Xte.fillna(med)
    keep = [c for c in Xtr.columns if Xtr[c].var() >= 0.01]
    corr = Xtr[keep].corr().abs().values
    drop = set()
    for l in range(len(keep)):
        for j in range(l):
            if keep[j] not in drop and corr[j, l] >= 0.95:
                drop.add(keep[l]); break
    keep = [c for c in keep if c not in drop]
    return Xtr[keep].values, Xte[keep].values

cut = int(0.8 * len(df))
pool_tr, pool_te = df.iloc[:cut], df.iloc[cut:]
fam_te = pool_te[pool_te.Label == 1]['attack_cat'].value_counts()          # descending
fam_tr = set(pool_tr[pool_tr.Label == 1]['attack_cat'])
order = [f for f in fam_te.index if f in fam_tr]                           # LARGEST first
levels = [1, 2, 3, 4]
print('Removal order (largest first):', order[:5], '| share of test flows:', (fam_te / len(pool_te)).round(4).head(5).to_dict())

res = pd.read_csv(RES4) if os.path.exists(RES4) else pd.DataFrame()
done = set(zip(res.seed, res.level)) if len(res) else set()
for seed in SEEDS:
    t0 = time.time()
    te = strat_sample(pool_te, N_TEST, seed)
    yte = te['Label'].values; att = yte == 1
    for m in levels:
        if (seed, m) in done:
            continue
        H = set(order[:m])
        tr = strat_sample(pool_tr[~pool_tr['attack_cat'].isin(H)], N_TRAIN, seed)
        novel = te['attack_cat'].isin(H).values
        rho = novel.mean()
        Xtr, Xte = make_xy(tr, te)
        ytr = tr['Label'].values
        rows = []
        for name, mdl in [
            ('RandomForest', RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed)),
            ('LightGBM', LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8,
                                        class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1))]:
            mdl.fit(Xtr, ytr); p = mdl.predict(Xte)
            seen_att, nov_att = att & ~novel, att & novel
            rows.append(dict(seed=seed, level=m, model=name, removed='+'.join(order[:m]), rho=rho,
                             attack_share_train=float(ytr.mean()), f1_macro=f1_score(yte, p, average='macro'),
                             recall_all=recall_score(yte, p),
                             recall_seen=(p[seen_att] == 1).mean() if seen_att.sum() else np.nan,
                             recall_novel=(p[nov_att] == 1).mean() if nov_att.sum() else np.nan,
                             fpr=((p == 1) & (yte == 0)).sum() / max((yte == 0).sum(), 1), acc=np.nan, bound=np.nan))
        classes = sorted(tr['attack_cat'].unique()); cmap = {c: i for i, c in enumerate(classes)}
        mc = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed)
        mc.fit(Xtr, tr['attack_cat'].map(cmap).values)
        acc = (mc.predict(Xte) == te['attack_cat'].map(cmap).fillna(-1).values).mean()
        rows.append(dict(seed=seed, level=m, model='RF-multiclass', removed='+'.join(order[:m]), rho=rho,
                         attack_share_train=float(ytr.mean()), f1_macro=np.nan, recall_all=np.nan, recall_seen=np.nan,
                         recall_novel=np.nan, fpr=np.nan, acc=acc, bound=1 - rho))
        res = pd.concat([res, pd.DataFrame(rows)], ignore_index=True)
        res.to_csv(RES4, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows: {len(res)}')

prev = pd.read_csv(PREV)
print('\n=== VARIANT B: binary models, largest families removed first ===')
for mname in ['RandomForest', 'LightGBM']:
    base = prev[(prev.model == mname) & (prev.level == 0)].set_index('seed')
    out = []
    for m in levels:
        d = res[(res.model == mname) & (res.level == m)].set_index('seed')
        common = d.index.intersection(base.index)
        if len(common) < 3: continue
        gap = (base.loc[common, 'f1_macro'] - d.loc[common, 'f1_macro']) * 100
        n = len(gap); h = stats.t.ppf(0.975, n - 1) * gap.std(ddof=1) / np.sqrt(n)
        out.append(dict(level=m, removed=d.removed.iloc[0], rho=round(d.rho.mean(), 4), f1_macro=round(d.f1_macro.mean(), 4),
                        drop_pp=round(gap.mean(), 2), ci=f'{gap.mean()-h:.2f} to {gap.mean()+h:.2f}',
                        recall_seen=round(d.recall_seen.mean(), 4), recall_novel=round(d.recall_novel.mean(), 4),
                        fpr=round(d.fpr.mean(), 4)))
    print('\n', mname, '(baseline F1-macro =', round(base.f1_macro.mean(), 4), ')'); print(pd.DataFrame(out).to_string(index=False))
mc = res[res.model == 'RF-multiclass']
mb = prev[(prev.model == 'RF-multiclass') & (prev.level == 0)].set_index('seed')['acc']
print('\n=== VARIANT B: 10-class RF accuracy vs bound 1 - rho (baseline acc at rho=0: %.4f) ===' % mb.mean())
s = mc.groupby('level').agg(removed=('removed', 'first'), rho=('rho', 'mean'), acc=('acc', 'mean'), bound=('bound', 'mean')).round(4)
s['max_excess'] = mc.assign(ex=mc.acc - mc.bound).groupby('level').ex.max().round(5)
print(s.to_string())
print('Runs violating acc <= 1 - rho:', int((mc.acc > mc.bound + 1e-9).sum()), 'of', len(mc))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Removal order (largest first): ['Generic', 'Exploits', 'Fuzzers', 'DoS', 'Reconnaissance'] | share of test flows: {'Generic': 0.1347, 'Exploits': 0.0253, 'Fuzzers': 0.0132, 'DoS': 0.0102, 'Reconnaissance': 0.008}
seed 42 done in 216s | rows: 12
seed 43 done in 211s | rows: 24
seed 44 done in 216s | rows: 36
seed 45 done in 210s | rows: 48
seed 46 done in 215s | rows: 60

=== VARIANT B: binary models, largest families removed first ===

 RandomForest (baseline F1-macro = 0.9716 )
 level                      removed    rho  f1_macro  drop_pp             ci  recall_seen  recall_novel    fpr
     1                      Generic 0.1347    0.9075     6.40   3.74 to 9.07       0.9980        0.7247 0.0230
     2             Generic+Exploits 0.1600    0.9164     5.52  0.39 to 10.64       0.9969        0.8022 0.0229
     3     Generic+Exploits+Fuzzers 0.1732    0.7652  

In [ ]:
# ===== VARIANT B-ext (resumes the Variant B results file): complete seeds 47-51 so Variant B has 10 seeds =====
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, os, time, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score
from lightgbm import LGBMClassifier
from scipy import stats

DATA_DIR = '/content/drive/MyDrive/UNSW-NB15'
CACHE = os.path.join(DATA_DIR, 'unsw_clean.parquet')
PREV  = os.path.join(DATA_DIR, 'unsw_dose_results.csv')        # original dose results (read only; level 0 = baseline)
RES4  = os.path.join(DATA_DIR, 'unsw_dose_large_results.csv')  # Variant B file (already has seeds 42-46; new seeds appended)
SEEDS = list(range(42, 52))
N_TRAIN, N_TEST = 200_000, 100_000
df = pd.read_parquet(CACHE)
CAT = ['proto', 'state', 'service']
NUM = [c for c in df.columns if c not in CAT + ['Stime', 'attack_cat', 'Label']]

def strat_sample(d, n, seed):
    frac = min(1.0, n / len(d))
    return d.groupby('attack_cat', group_keys=False).sample(frac=frac, random_state=seed)

def make_xy(tr, te):
    maps = {c: {v: i for i, v in enumerate(sorted(tr[c].unique()))} for c in CAT}
    def enc(d):
        X = d[NUM].copy()
        for c in CAT:
            X[c] = d[c].map(maps[c]).fillna(-1).astype('float32')
        return X.replace([np.inf, -np.inf], np.nan)
    Xtr, Xte = enc(tr), enc(te)
    med = Xtr.median()
    Xtr, Xte = Xtr.fillna(med), Xte.fillna(med)
    keep = [c for c in Xtr.columns if Xtr[c].var() >= 0.01]
    corr = Xtr[keep].corr().abs().values
    drop = set()
    for l in range(len(keep)):
        for j in range(l):
            if keep[j] not in drop and corr[j, l] >= 0.95:
                drop.add(keep[l]); break
    keep = [c for c in keep if c not in drop]
    return Xtr[keep].values, Xte[keep].values

cut = int(0.8 * len(df))
pool_tr, pool_te = df.iloc[:cut], df.iloc[cut:]
fam_te = pool_te[pool_te.Label == 1]['attack_cat'].value_counts()
fam_tr = set(pool_tr[pool_tr.Label == 1]['attack_cat'])
order = [f for f in fam_te.index if f in fam_tr]
levels = [1, 2, 3, 4]

res = pd.read_csv(RES4) if os.path.exists(RES4) else pd.DataFrame()
done = set(zip(res.seed, res.level)) if len(res) else set()
for seed in SEEDS:
    t0 = time.time()
    te = strat_sample(pool_te, N_TEST, seed)
    yte = te['Label'].values; att = yte == 1
    for m in levels:
        if (seed, m) in done:
            continue
        H = set(order[:m])
        tr = strat_sample(pool_tr[~pool_tr['attack_cat'].isin(H)], N_TRAIN, seed)
        novel = te['attack_cat'].isin(H).values
        rho = novel.mean()
        Xtr, Xte = make_xy(tr, te)
        ytr = tr['Label'].values
        rows = []
        for name, mdl in [
            ('RandomForest', RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed)),
            ('LightGBM', LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8,
                                        class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1))]:
            mdl.fit(Xtr, ytr); p = mdl.predict(Xte)
            seen_att, nov_att = att & ~novel, att & novel
            rows.append(dict(seed=seed, level=m, model=name, removed='+'.join(order[:m]), rho=rho,
                             attack_share_train=float(ytr.mean()), f1_macro=f1_score(yte, p, average='macro'),
                             recall_all=recall_score(yte, p),
                             recall_seen=(p[seen_att] == 1).mean() if seen_att.sum() else np.nan,
                             recall_novel=(p[nov_att] == 1).mean() if nov_att.sum() else np.nan,
                             fpr=((p == 1) & (yte == 0)).sum() / max((yte == 0).sum(), 1), acc=np.nan, bound=np.nan))
        classes = sorted(tr['attack_cat'].unique()); cmap = {c: i for i, c in enumerate(classes)}
        mc = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed)
        mc.fit(Xtr, tr['attack_cat'].map(cmap).values)
        acc = (mc.predict(Xte) == te['attack_cat'].map(cmap).fillna(-1).values).mean()
        rows.append(dict(seed=seed, level=m, model='RF-multiclass', removed='+'.join(order[:m]), rho=rho,
                         attack_share_train=float(ytr.mean()), f1_macro=np.nan, recall_all=np.nan, recall_seen=np.nan,
                         recall_novel=np.nan, fpr=np.nan, acc=acc, bound=1 - rho))
        res = pd.concat([res, pd.DataFrame(rows)], ignore_index=True)
        res.to_csv(RES4, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows: {len(res)}')

prev = pd.read_csv(PREV)
print('\n=== VARIANT B (10 seeds): binary models, largest families removed first ===')
for mname in ['RandomForest', 'LightGBM']:
    base = prev[(prev.model == mname) & (prev.level == 0)].set_index('seed')
    out = []
    for m in levels:
        d = res[(res.model == mname) & (res.level == m)].set_index('seed')
        common = d.index.intersection(base.index)
        gap = (base.loc[common, 'f1_macro'] - d.loc[common, 'f1_macro']) * 100
        n = len(gap); h = stats.t.ppf(0.975, n - 1) * gap.std(ddof=1) / np.sqrt(n)
        out.append(dict(level=m, n=n, rho=round(d.rho.mean(), 4), f1_macro=round(d.f1_macro.mean(), 4),
                        drop_pp=round(gap.mean(), 2), ci=f'{gap.mean()-h:.2f} to {gap.mean()+h:.2f}',
                        recall_seen=round(d.recall_seen.mean(), 4), recall_novel=round(d.recall_novel.mean(), 4),
                        attack_share_train=round(d.attack_share_train.mean(), 4)))
    print('\n', mname, '(baseline F1-macro =', round(base.f1_macro.mean(), 4), ')'); print(pd.DataFrame(out).to_string(index=False))
mc = res[res.model == 'RF-multiclass']
print('\n=== 10-class RF accuracy vs bound 1 - rho ===')
s = mc.groupby('level').agg(n=('acc', 'size'), rho=('rho', 'mean'), acc=('acc', 'mean'), bound=('bound', 'mean')).round(4)
s['max_excess'] = mc.assign(ex=mc.acc - mc.bound).groupby('level').ex.max().round(5)
print(s.to_string())
print('Runs violating acc <= 1 - rho:', int((mc.acc > mc.bound + 1e-9).sum()), 'of', len(mc))


Mounted at /content/drive
seed 42 done in 0s | rows: 60
seed 43 done in 0s | rows: 60
seed 44 done in 0s | rows: 60
seed 45 done in 0s | rows: 60
seed 46 done in 0s | rows: 60
seed 47 done in 221s | rows: 72
seed 48 done in 196s | rows: 84
seed 49 done in 186s | rows: 96
seed 50 done in 184s | rows: 108
seed 51 done in 195s | rows: 120

=== VARIANT B (10 seeds): binary models, largest families removed first ===

 RandomForest (baseline F1-macro = 0.9716 )
 level  n    rho  f1_macro  drop_pp             ci  recall_seen  recall_novel  attack_share_train
     1 10 0.1347    0.9110     6.06   4.26 to 7.86       0.9982        0.7394              0.0399
     2 10 0.1600    0.9076     6.40   4.12 to 8.68       0.9977        0.7701              0.0235
     3 10 0.1732    0.7612    21.05 19.45 to 22.64       0.9967        0.3764              0.0141
     4 10 0.1834    0.7294    24.22 22.12 to 26.33       0.9910        0.3446              0.0081

 LightGBM (baseline F1-macro = 0.9733 )
 level  n

In [ ]:
# ===== VARIANT C (new results file): class-balance control - hold the TRAINING attack share fixed while removing families =====
import pandas as pd, numpy as np, os, time, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score
from lightgbm import LGBMClassifier
from scipy import stats

DATA_DIR = '/content/drive/MyDrive/UNSW-NB15'
CACHE = os.path.join(DATA_DIR, 'unsw_clean.parquet')
PREV  = os.path.join(DATA_DIR, 'unsw_dose_results.csv')        # baseline (level 0), read only
VARB  = os.path.join(DATA_DIR, 'unsw_dose_large_results.csv')  # Variant B (unbalanced), read only
RES5  = os.path.join(DATA_DIR, 'unsw_dose_balanced_results.csv')  # new file
SEEDS = list(range(42, 52))
N_TRAIN, N_TEST = 200_000, 100_000
df = pd.read_parquet(CACHE)
CAT = ['proto', 'state', 'service']
NUM = [c for c in df.columns if c not in CAT + ['Stime', 'attack_cat', 'Label']]

def strat_sample(d, n, seed):
    frac = min(1.0, n / len(d))
    return d.groupby('attack_cat', group_keys=False).sample(frac=frac, random_state=seed)

def make_xy(tr, te):
    maps = {c: {v: i for i, v in enumerate(sorted(tr[c].unique()))} for c in CAT}
    def enc(d):
        X = d[NUM].copy()
        for c in CAT:
            X[c] = d[c].map(maps[c]).fillna(-1).astype('float32')
        return X.replace([np.inf, -np.inf], np.nan)
    Xtr, Xte = enc(tr), enc(te)
    med = Xtr.median()
    Xtr, Xte = Xtr.fillna(med), Xte.fillna(med)
    keep = [c for c in Xtr.columns if Xtr[c].var() >= 0.01]
    corr = Xtr[keep].corr().abs().values
    drop = set()
    for l in range(len(keep)):
        for j in range(l):
            if keep[j] not in drop and corr[j, l] >= 0.95:
                drop.add(keep[l]); break
    keep = [c for c in keep if c not in drop]
    return Xtr[keep].values, Xte[keep].values

cut = int(0.8 * len(df))
pool_tr, pool_te = df.iloc[:cut], df.iloc[cut:]
fam_te = pool_te[pool_te.Label == 1]['attack_cat'].value_counts()
fam_tr = set(pool_tr[pool_tr.Label == 1]['attack_cat'])
order = [f for f in fam_te.index if f in fam_tr]
levels = [1, 2, 3, 4]
ben_pool = pool_tr[pool_tr.Label == 0]

res = pd.read_csv(RES5) if os.path.exists(RES5) else pd.DataFrame()
done = set(zip(res.seed, res.level)) if len(res) else set()
for seed in SEEDS:
    t0 = time.time()
    te = strat_sample(pool_te, N_TEST, seed)
    yte = te['Label'].values; att = yte == 1
    s0 = strat_sample(pool_tr, N_TRAIN, seed)['Label'].mean()      # baseline training attack share for this seed
    for m in levels:
        if (seed, m) in done:
            continue
        H = set(order[:m])
        att_pool = pool_tr[(pool_tr.Label == 1) & (~pool_tr['attack_cat'].isin(H))]
        n_att = min(int(round(s0 * N_TRAIN)), len(att_pool))
        n_ben = int(round(n_att * (1 - s0) / s0))                  # keeps attack share = s0 exactly
        tr_att = att_pool.groupby('attack_cat', group_keys=False).sample(frac=n_att / len(att_pool), random_state=seed)
        tr_ben = ben_pool.sample(n=min(n_ben, len(ben_pool)), random_state=seed)
        tr = pd.concat([tr_att, tr_ben])
        novel = te['attack_cat'].isin(H).values
        rho = novel.mean()
        Xtr, Xte = make_xy(tr, te)
        ytr = tr['Label'].values
        for name, mdl in [
            ('RandomForest', RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed)),
            ('LightGBM', LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8,
                                        class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1))]:
            mdl.fit(Xtr, ytr); p = mdl.predict(Xte)
            seen_att, nov_att = att & ~novel, att & novel
            row = dict(seed=seed, level=m, model=name, removed='+'.join(order[:m]), rho=rho, n_train=len(tr),
                       attack_share_train=float(ytr.mean()), s0=float(s0), f1_macro=f1_score(yte, p, average='macro'),
                       recall_seen=(p[seen_att] == 1).mean() if seen_att.sum() else np.nan,
                       recall_novel=(p[nov_att] == 1).mean() if nov_att.sum() else np.nan,
                       fpr=((p == 1) & (yte == 0)).sum() / max((yte == 0).sum(), 1))
            res = pd.concat([res, pd.DataFrame([row])], ignore_index=True)
        res.to_csv(RES5, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows: {len(res)}')

prev = pd.read_csv(PREV); vb = pd.read_csv(VARB)
print('\n=== VARIANT C: training attack share held fixed (balanced) vs Variant B (share not controlled) ===')
for mname in ['RandomForest', 'LightGBM']:
    base = prev[(prev.model == mname) & (prev.level == 0)].set_index('seed')['f1_macro']
    out = []
    for m in levels:
        d = res[(res.model == mname) & (res.level == m)].set_index('seed')
        b = vb[(vb.model == mname) & (vb.level == m)].set_index('seed')
        common = d.index.intersection(base.index)
        gap = (base.loc[common] - d.loc[common, 'f1_macro']) * 100
        gapb = (base.loc[b.index.intersection(base.index)] - b.loc[b.index.intersection(base.index), 'f1_macro']) * 100
        n = len(gap); h = stats.t.ppf(0.975, n - 1) * gap.std(ddof=1) / np.sqrt(n)
        out.append(dict(level=m, n=n, rho=round(d.rho.mean(), 4), share_train=round(d.attack_share_train.mean(), 4),
                        n_train=int(d.n_train.mean()), drop_pp_balanced=round(gap.mean(), 2),
                        ci=f'{gap.mean()-h:.2f} to {gap.mean()+h:.2f}', drop_pp_varB=round(gapb.mean(), 2),
                        recall_novel=round(d.recall_novel.mean(), 4), recall_seen=round(d.recall_seen.mean(), 4)))
    print('\n', mname, '(baseline F1-macro =', round(base.mean(), 4), ')'); print(pd.DataFrame(out).to_string(index=False))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
seed 42 done in 109s | rows: 8
seed 43 done in 107s | rows: 16
seed 44 done in 106s | rows: 24
seed 45 done in 104s | rows: 32
seed 46 done in 108s | rows: 40
seed 47 done in 105s | rows: 48
seed 48 done in 106s | rows: 56
seed 49 done in 109s | rows: 64
seed 50 done in 104s | rows: 72
seed 51 done in 105s | rows: 80

=== VARIANT C: training attack share held fixed (balanced) vs Variant B (share not controlled) ===

 RandomForest (baseline F1-macro = 0.9716 )
 level  n    rho  share_train  n_train  drop_pp_balanced             ci  drop_pp_varB  recall_novel  recall_seen
     1 10 0.1347       0.1093   200002              0.96   0.18 to 1.73          6.06        0.9568       0.9993
     2 10 0.1600       0.1093   200002              0.47   0.16 to 0.77          6.40        0.9821       0.9991
     3 10 0.1732       0.1093   200002             16.84 13.44 to 20

In [ ]:
# ===== CELL 9: Mechanism experiment (leave-one-family-out + feature-space distance) =====
import pandas as pd, numpy as np, os, time, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from scipy import stats
for nm in ['pool_tr','pool_te','strat_sample','make_xy','SEEDS','N_TRAIN','N_TEST','DATA_DIR']:
    assert nm in globals(), f'{nm} missing: re-run the earlier cells (load + helper definitions) first'
RESM = os.path.join(DATA_DIR, 'unsw_mech_results.csv')
FAMS = ['Generic','Exploits','Fuzzers','DoS','Reconnaissance','Analysis','Backdoors','Shellcode','Worms']
def models(seed):
    return [('RandomForest', RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed)),
            ('LightGBM', LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1)),
            ('XGBoost', XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, eval_metric='logloss', verbosity=0))]
res = pd.read_csv(RESM) if os.path.exists(RESM) else pd.DataFrame()
done = set(zip(res.seed, res.family)) if len(res) else set()
for seed in SEEDS:
    t0 = time.time()
    base_tr = strat_sample(pool_tr, N_TRAIN, seed)
    for fam in FAMS:
        if (seed, fam) in done: continue
        ev = pool_te[pool_te['attack_cat'] == fam]
        if len(ev) > 5000: ev = ev.sample(5000, random_state=seed)
        if len(ev) < 5: continue
        tr = strat_sample(pool_tr[pool_tr['attack_cat'] != fam], N_TRAIN, seed)
        Xtr, Xev = make_xy(tr, ev); ytr = tr['Label'].values
        Xb, Xev_b = make_xy(base_tr, ev); yb = base_tr['Label'].values
        sc = StandardScaler().fit(np.asarray(Xtr, dtype=float))
        Z = sc.transform(np.asarray(Xtr, dtype=float)); Ze = sc.transform(np.asarray(Xev, dtype=float))
        rng = np.random.RandomState(seed)
        ia = np.where(ytr == 1)[0]; inn = np.where(ytr == 0)[0]
        ia = rng.choice(ia, min(len(ia), 20000), replace=False); inn = rng.choice(inn, min(len(inn), 20000), replace=False)
        ee = rng.choice(len(Ze), min(len(Ze), 1000), replace=False)
        d_att = NearestNeighbors(n_neighbors=1).fit(Z[ia]).kneighbors(Ze[ee])[0].ravel()
        d_nor = NearestNeighbors(n_neighbors=1).fit(Z[inn]).kneighbors(Ze[ee])[0].ravel()
        fam_in_base = base_tr['attack_cat'].values == fam
        scb = StandardScaler().fit(np.asarray(Xb, dtype=float))
        Zb = scb.transform(np.asarray(Xb, dtype=float))[fam_in_base]; Zeb = scb.transform(np.asarray(Xev_b, dtype=float))
        d_own = NearestNeighbors(n_neighbors=1).fit(Zb[:20000]).kneighbors(Zeb[ee])[0].ravel() if fam_in_base.sum() > 0 else np.array([np.nan])
        geo = dict(d_att=np.median(d_att), d_nor=np.median(d_nor), ratio=np.median(d_nor / np.maximum(d_att, 1e-9)),
                   frac_closer_att=float((d_att < d_nor).mean()), d_own=np.nanmedian(d_own))
        rows = []
        for (name, m), (_, mb) in zip(models(seed), models(seed)):
            m.fit(Xtr, ytr); rec_loo = float((m.predict(Xev) == 1).mean())
            mb.fit(Xb, yb); rec_base = float((mb.predict(Xev_b) == 1).mean())
            rows.append(dict(seed=seed, family=fam, model=name, n_eval=len(ev), recall_loo=rec_loo, recall_base=rec_base, **geo))
        res = pd.concat([res, pd.DataFrame(rows)], ignore_index=True); res.to_csv(RESM, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows {len(res)}')
g = res.groupby(['model','family']).agg(recall_loo=('recall_loo','mean'), recall_base=('recall_base','mean'), ratio=('ratio','mean'),
        frac_closer_att=('frac_closer_att','mean'), d_own=('d_own','mean'), n=('seed','nunique')).reset_index()
print('\n=== Leave-one-family-out recall and geometry (means over seeds) ===')
for mdl in ['RandomForest','LightGBM','XGBoost']:
    print('\n', mdl); print(g[g.model==mdl].sort_values('recall_loo').drop(columns='model').round(4).to_string(index=False))
print('\n=== Spearman across the 9 families: held-out recall vs geometry ===')
for mdl in ['RandomForest','LightGBM','XGBoost']:
    s = g[g.model==mdl]
    for col in ['ratio','frac_closer_att','d_own']:
        r, p = stats.spearmanr(s[col], s['recall_loo']); print(f'{mdl:12s} {col:16s} rho={r:+.2f} p={p:.3f}')


seed 42 done in 540s | rows 27
seed 43 done in 531s | rows 54
seed 44 done in 536s | rows 81
seed 45 done in 546s | rows 108
seed 46 done in 553s | rows 135
seed 47 done in 523s | rows 162
seed 48 done in 548s | rows 189
seed 49 done in 543s | rows 216
seed 50 done in 528s | rows 243
seed 51 done in 539s | rows 270

=== Leave-one-family-out recall and geometry (means over seeds) ===

 RandomForest
        family  recall_loo  recall_base  ratio  frac_closer_att  d_own  n
       Fuzzers      0.5410       0.9956 1.2469           0.6613 0.4385 10
       Generic      0.7420       1.0000 1.3511           0.8541 0.1448 10
      Analysis      0.9330       1.0000 4.9094           0.9485 0.5363 10
         Worms      0.9846       0.9808 4.1050           0.9865 1.5365 10
           DoS      0.9986       0.9993 5.0649           0.9902 0.3491 10
      Exploits      0.9993       0.9995 2.7768           0.9716 0.4417 10
Reconnaissance      1.0000       1.0000 2.4381           0.9620 0.2130 10
     Ba

In [ ]:
# ===== CELL 10: UNSW-NB15 random vs temporal WITH SMOTE (paper's binary rule: k=max(1,min(5,n_min-1)), target=max(n_min,min(30000,n_maj))) =====
import pandas as pd, numpy as np, os, time, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from scipy import stats
for nm in ['pool_tr','pool_te','strat_sample','make_xy','SEEDS','N_TRAIN','N_TEST','DATA_DIR']:
    assert nm in globals(), f'{nm} missing: re-run the earlier cells first'
RESS = os.path.join(DATA_DIR, 'unsw_smote_results.csv')
def models(seed):
    return [('RandomForest', RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed)),
            ('LightGBM', LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1)),
            ('XGBoost', XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, eval_metric='logloss', verbosity=0))]
def smote_fit(Xtr, ytr, seed):
    sc = StandardScaler().fit(np.asarray(Xtr, dtype=float)); Z = sc.transform(np.asarray(Xtr, dtype=float))
    n_min, n_maj = int((ytr == 1).sum()), int((ytr == 0).sum())
    target = max(n_min, min(30000, n_maj)); k = max(1, min(5, n_min - 1))
    if target > n_min:
        Z, y = SMOTE(sampling_strategy={1: target}, k_neighbors=k, random_state=seed).fit_resample(Z, ytr)
    else: y = ytr
    return sc, Z, y, target - n_min if target > n_min else 0
allp = pd.concat([pool_tr, pool_te])
res = pd.read_csv(RESS) if os.path.exists(RESS) else pd.DataFrame()
done = set(zip(res.seed, res.protocol)) if len(res) else set()
for seed in SEEDS:
    t0 = time.time()
    rtr, rte = train_test_split(allp, test_size=0.2, stratify=allp['Label'], random_state=seed)
    for proto, (ptr, pte) in {'random': (rtr, rte), 'temporal': (pool_tr, pool_te)}.items():
        if (seed, proto) in done: continue
        tr = strat_sample(ptr, N_TRAIN, seed); te = strat_sample(pte, N_TEST, seed)
        Xtr, Xte = make_xy(tr, te); ytr, yte = tr['Label'].values, te['Label'].values
        sc, Z, y, n_syn = smote_fit(Xtr, ytr, seed); Zte = sc.transform(np.asarray(Xte, dtype=float))
        rows = []
        for name, m in models(seed):
            m.fit(Z, y); p = m.predict(Zte)
            rows.append(dict(seed=seed, protocol=proto, model=name, n_synth=n_syn, f1_macro=f1_score(yte, p, average='macro'),
                             recall=recall_score(yte, p), fpr=((p == 1) & (yte == 0)).sum() / max((yte == 0).sum(), 1)))
        res = pd.concat([res, pd.DataFrame(rows)], ignore_index=True); res.to_csv(RESS, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows {len(res)}')
print('\n=== UNSW-NB15 with SMOTE: random vs temporal (80/20), 10 seeds ===')
for mdl in ['RandomForest','LightGBM','XGBoost']:
    a = res[(res.model==mdl)&(res.protocol=='random')].sort_values('seed'); b = res[(res.model==mdl)&(res.protocol=='temporal')].sort_values('seed')
    d = (a.f1_macro.values - b.f1_macro.values) * 100; n = len(d); se = d.std(ddof=1) / np.sqrt(n); h = stats.t.ppf(0.975, n-1) * se
    try: p = stats.wilcoxon(d, method='exact').pvalue
    except Exception: p = np.nan
    cd = np.mean([np.sign(x - y) for x in a.f1_macro.values for y in b.f1_macro.values])
    print(f'{mdl:12s} random={a.f1_macro.mean():.4f} temporal={b.f1_macro.mean():.4f} gap={d.mean():.2f} pp CI [{d.mean()-h:.2f}, {d.mean()+h:.2f}] wilcoxon p={p:.5f} cliff={cd:.3f} | recall gap={(a.recall.values-b.recall.values).mean()*100:.2f} pp, FPR diff={(b.fpr.values-a.fpr.values).mean()*100:.2f} pp | synth added={b.n_synth.mean():.0f}')


seed 42 done in 97s | rows 6
seed 43 done in 93s | rows 12
seed 44 done in 89s | rows 18
seed 45 done in 88s | rows 24
seed 46 done in 88s | rows 30
seed 47 done in 86s | rows 36
seed 48 done in 88s | rows 42
seed 49 done in 88s | rows 48
seed 50 done in 87s | rows 54
seed 51 done in 84s | rows 60

=== UNSW-NB15 with SMOTE: random vs temporal (80/20), 10 seeds ===
RandomForest random=0.9736 temporal=0.9716 gap=0.19 pp CI [0.14, 0.25] wilcoxon p=0.00195 cliff=1.000 | recall gap=0.01 pp, FPR diff=0.89 pp | synth added=8131
LightGBM     random=0.9758 temporal=0.9737 gap=0.21 pp CI [0.16, 0.26] wilcoxon p=0.00195 cliff=1.000 | recall gap=0.06 pp, FPR diff=0.82 pp | synth added=8131
XGBoost      random=0.9828 temporal=0.9806 gap=0.22 pp CI [0.18, 0.26] wilcoxon p=0.00195 cliff=1.000 | recall gap=-0.37 pp, FPR diff=0.45 pp | synth added=8131


In [ ]:
# ===== CELL 11: UNSW-NB15 family removal (largest first) WITH SMOTE: uncontrolled vs balanced training attack share =====
import pandas as pd, numpy as np, os, time, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from scipy import stats
for nm in ['pool_tr','pool_te','strat_sample','make_xy','SEEDS','N_TRAIN','N_TEST','DATA_DIR']:
    assert nm in globals(), f'{nm} missing: re-run the earlier cells first'
RESR = os.path.join(DATA_DIR, 'unsw_smote_removal_results.csv')
ORDER = ['Generic','Exploits','Fuzzers','DoS']
def models(seed):
    return [('RandomForest', RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed)),
            ('LightGBM', LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1))]
def smote_fit(Xtr, ytr, seed):
    sc = StandardScaler().fit(np.asarray(Xtr, dtype=float)); Z = sc.transform(np.asarray(Xtr, dtype=float))
    n_min, n_maj = int((ytr == 1).sum()), int((ytr == 0).sum())
    target = max(n_min, min(30000, n_maj)); k = max(1, min(5, n_min - 1))
    if target > n_min: Z, y = SMOTE(sampling_strategy={1: target}, k_neighbors=k, random_state=seed).fit_resample(Z, ytr)
    else: y = ytr
    return sc, Z, y
res = pd.read_csv(RESR) if os.path.exists(RESR) else pd.DataFrame()
done = set(zip(res.seed, res.level, res.variant)) if len(res) else set()
for seed in SEEDS:
    t0 = time.time()
    te = strat_sample(pool_te, N_TEST, seed); yte = te['Label'].values
    base_tr = strat_sample(pool_tr, N_TRAIN, seed); base_share = float(base_tr['Label'].mean())
    for level in range(0, len(ORDER) + 1):
        H = set(ORDER[:level]); novel = te['attack_cat'].isin(H).values; rho = float(novel.mean())
        pr = pool_tr[~pool_tr['attack_cat'].isin(H)]
        for variant in (['uncontrolled'] if level == 0 else ['uncontrolled', 'balanced']):
            if (seed, level, variant) in done: continue
            if variant == 'uncontrolled': tr = strat_sample(pr, N_TRAIN, seed)
            else:
                att, nor = pr[pr['Label'] == 1], pr[pr['Label'] == 0]
                n_att = min(len(att), int(round(base_share * N_TRAIN))); n_nor = min(len(nor), int(round(n_att * (1 - base_share) / base_share)))
                tr = pd.concat([strat_sample(att, n_att, seed), strat_sample(nor, n_nor, seed)])
            Xtr, Xte = make_xy(tr, te); ytr = tr['Label'].values
            sc, Z, y = smote_fit(Xtr, ytr, seed); Zte = sc.transform(np.asarray(Xte, dtype=float))
            rows = []
            for name, m in models(seed):
                m.fit(Z, y); p = m.predict(Zte)
                rows.append(dict(seed=seed, level=level, variant=variant, removed='+'.join(ORDER[:level]), model=name, rho=rho,
                                 n_train=len(tr), share_train=float(ytr.mean()), share_after_smote=float(y.mean()),
                                 f1_macro=f1_score(yte, p, average='macro'),
                                 recall_novel=(p[novel & (yte == 1)] == 1).mean() if (novel & (yte == 1)).sum() else np.nan))
            res = pd.concat([res, pd.DataFrame(rows)], ignore_index=True); res.to_csv(RESR, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows {len(res)}')
print('\n=== UNSW-NB15 family removal WITH SMOTE: F1-macro drop vs level-0 baseline (pp), 10 seeds ===')
for mdl in ['RandomForest','LightGBM']:
    b0 = res[(res.model==mdl)&(res.level==0)].sort_values('seed').f1_macro.values
    print(f'\n{mdl} (baseline F1-macro = {b0.mean():.4f})')
    for variant in ['uncontrolled','balanced']:
        for level in range(1, len(ORDER) + 1):
            s = res[(res.model==mdl)&(res.level==level)&(res.variant==variant)].sort_values('seed')
            if len(s) != len(b0): continue
            d = (b0 - s.f1_macro.values) * 100; n = len(d); h = stats.t.ppf(0.975, n-1) * d.std(ddof=1) / np.sqrt(n)
            print(f'{variant:12s} L{level} rho={s.rho.mean():.4f} n_train={s.n_train.mean():.0f} share_after_smote={s.share_after_smote.mean():.3f} drop={d.mean():.2f} CI [{d.mean()-h:.2f}, {d.mean()+h:.2f}] recall_novel={s.recall_novel.mean():.3f}')


seed 42 done in 342s | rows 18
seed 43 done in 330s | rows 36
seed 44 done in 330s | rows 54
seed 45 done in 335s | rows 72
seed 46 done in 333s | rows 90
seed 47 done in 327s | rows 108
seed 48 done in 331s | rows 126
seed 49 done in 331s | rows 144
seed 50 done in 324s | rows 162
seed 51 done in 325s | rows 180

=== UNSW-NB15 family removal WITH SMOTE: F1-macro drop vs level-0 baseline (pp), 10 seeds ===

RandomForest (baseline F1-macro = 0.9716)
uncontrolled L1 rho=0.1347 n_train=200000 share_after_smote=0.135 drop=2.54 CI [1.06, 4.02] recall_novel=0.887
uncontrolled L2 rho=0.1600 n_train=200001 share_after_smote=0.133 drop=2.18 CI [0.31, 4.06] recall_novel=0.919
uncontrolled L3 rho=0.1732 n_train=200000 share_after_smote=0.132 drop=17.99 CI [14.03, 21.96] recall_novel=0.448
uncontrolled L4 rho=0.1834 n_train=199999 share_after_smote=0.131 drop=21.52 CI [19.20, 23.85] recall_novel=0.395
balanced     L1 rho=0.1347 n_train=200002 share_after_smote=0.144 drop=0.68 CI [0.41, 0.94] recal

In [ ]:
# ===== CELL 16: CIC-IDS 2017 pipeline (from the paper's Sections 3.3/4.1) + validation against the published seed-42 numbers =====
import pandas as pd, numpy as np, os, time, gc, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score, accuracy_score
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
CIC = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017/preprocessed/all_days_merged.csv'
OUT = '/content/drive/MyDrive/UNSW-NB15'   # new result files go next to the earlier ones
if 'cic' not in globals():
    cic = pd.read_csv(CIC); cic.columns = [c.strip() for c in cic.columns]
    cic['day'] = cic['day'].str.lower(); cic['Label'] = cic['Label'].str.strip()
    for c in cic.columns:
        if cic[c].dtype == 'float64': cic[c] = cic[c].astype('float32')
FEATS = [c for c in cic.columns if c not in ('Label', 'day', 'Destination Port')]
TRAIN_DAYS = ['monday', 'tuesday', 'wednesday', 'thursday']
print('rows', len(cic), 'features before filters', len(FEATS), '| days', cic['day'].value_counts().to_dict())
def prop_sample(df, n, seed):
    frac = n / len(df); parts = []
    for _, g in df.groupby('Label'):
        k = int(frac * len(g))
        if k > 0: parts.append(g.sample(n=k, random_state=seed))
    return pd.concat(parts)
def make_partitions(proto, seed):
    if proto == 'temporal':
        return prop_sample(cic[cic['day'].isin(TRAIN_DAYS)], 199_999, seed), prop_sample(cic[cic['day'] == 'friday'], 99_999, seed)
    pooled = prop_sample(cic, 199_999, seed)
    return train_test_split(pooled, test_size=0.2, stratify=(pooled['Label'] != 'BENIGN'), random_state=seed)
def prepare(tr, te, smote=True, seed=42):
    Xtr, Xte = tr[FEATS].astype('float64'), te[FEATS].astype('float64')
    vt = VarianceThreshold(0.01).fit(Xtr); cols = Xtr.columns[vt.get_support()]
    Xtr, Xte = Xtr[cols], Xte[cols]
    cm = Xtr.corr().abs(); up = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool))
    drop = [c for c in up.columns if (up[c] > 0.95).any()]; Xtr, Xte = Xtr.drop(columns=drop), Xte.drop(columns=drop)
    sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
    ytr, yte = (tr['Label'] != 'BENIGN').astype(int).values, (te['Label'] != 'BENIGN').astype(int).values
    n_syn = 0
    if smote:
        n_min, n_maj = int(ytr.sum()), int((ytr == 0).sum()); target = max(n_min, min(30000, n_maj)); k = max(1, min(5, n_min - 1))
        if target > n_min:
            Ztr, ytr = SMOTE(sampling_strategy={1: target}, k_neighbors=k, random_state=seed).fit_resample(Ztr, ytr); n_syn = target - n_min
    return Ztr, ytr, Zte, yte, Xtr.shape[1], n_syn
def metrics(yte, p): return dict(f1_macro=f1_score(yte, p, average='macro'), recall=recall_score(yte, p), fpr=((p == 1) & (yte == 0)).sum() / max((yte == 0).sum(), 1))
PAPER_BASE = {'RandomForest': (0.9950, 0.4399), 'LightGBM': (0.9984, 0.3734), 'XGBoost': (0.9987, 0.4960)}
def base_models(seed):
    return {'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed),
            'LightGBM': LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1),
            'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, eval_metric='logloss', verbosity=0)}
t0 = time.time(); val = {}
for proto in ['random', 'temporal']:
    tr, te = make_partitions(proto, 42); Ztr, ytr, Zte, yte, nf, ns = prepare(tr, te, True, 42)
    print(f'{proto:8s} train={len(tr)} test={len(te)} attack_test={yte.sum()} features={nf} smote_added={ns}')
    for name, m in base_models(42).items():
        m.fit(Ztr, ytr); val[(name, proto)] = metrics(yte, m.predict(Zte))['f1_macro']
print('\nVALIDATION (seed 42), F1-macro: mine vs paper')
for name, (pr, pt) in PAPER_BASE.items():
    print(f'{name:12s} random {val[(name,"random")]:.4f} (paper {pr:.4f}) | temporal {val[(name,"temporal")]:.4f} (paper {pt:.4f})')
print(f'elapsed {time.time()-t0:.0f}s')


rows 2827876 features before filters 77 | days {'friday': 702718, 'wednesday': 691406, 'monday': 529481, 'thursday': 458626, 'tuesday': 445645}
random   train=159993 test=39999 attack_test=7871 features=43 smote_added=0
temporal train=199993 test=99997 attack_test=41094 features=43 smote_added=4806

VALIDATION (seed 42), F1-macro: mine vs paper
RandomForest random 0.9901 (paper 0.9950) | temporal 0.4294 (paper 0.4399)
LightGBM     random 0.9978 (paper 0.9984) | temporal 0.3721 (paper 0.3734)
XGBoost      random 0.9979 (paper 0.9987) | temporal 0.4488 (paper 0.4960)
elapsed 125s


In [ ]:
# ===== CELL 17: CIC-IDS 2017 hyperparameter sweep: does the random-vs-temporal gap depend on trees / depth? (5 seeds) =====
import pandas as pd, numpy as np, os, time
from scipy import stats
for nm in ['cic', 'make_partitions', 'prepare', 'metrics']: assert nm in globals(), 'run CELL 16 first'
RESW = os.path.join(OUT, 'cic_sweep_results.csv'); SW_SEEDS = [42, 43, 44, 45, 46]
CONFIGS = [('RandomForest', 'n100_d10 (paper)', dict(n_estimators=100, max_depth=10)), ('RandomForest', 'n50_d10', dict(n_estimators=50, max_depth=10)),
           ('RandomForest', 'n200_d10', dict(n_estimators=200, max_depth=10)), ('RandomForest', 'n100_d6', dict(n_estimators=100, max_depth=6)),
           ('RandomForest', 'n100_d14', dict(n_estimators=100, max_depth=14)), ('RandomForest', 'n100_dNone', dict(n_estimators=100, max_depth=None)),
           ('LightGBM', 'n100_d8 (paper)', dict(n_estimators=100, max_depth=8)), ('LightGBM', 'n50_d8', dict(n_estimators=50, max_depth=8)),
           ('LightGBM', 'n200_d8', dict(n_estimators=200, max_depth=8)), ('LightGBM', 'n100_d4', dict(n_estimators=100, max_depth=4)),
           ('LightGBM', 'n100_d12', dict(n_estimators=100, max_depth=12)),
           ('XGBoost', 'n100_d6 (paper)', dict(n_estimators=100, max_depth=6)), ('XGBoost', 'n50_d6', dict(n_estimators=50, max_depth=6)),
           ('XGBoost', 'n200_d6', dict(n_estimators=200, max_depth=6)), ('XGBoost', 'n100_d3', dict(n_estimators=100, max_depth=3)),
           ('XGBoost', 'n100_d9', dict(n_estimators=100, max_depth=9))]
def build(model, kw, seed):
    if model == 'RandomForest': return RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=seed, **kw)
    if model == 'LightGBM': return LGBMClassifier(num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1, **kw)
    return XGBClassifier(subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, eval_metric='logloss', verbosity=0, **kw)
res = pd.read_csv(RESW) if os.path.exists(RESW) else pd.DataFrame()
done = set(zip(res.seed, res.config, res.model)) if len(res) else set()
for seed in SW_SEEDS:
    t0 = time.time(); data = {}
    todo = [(m, c, kw) for m, c, kw in CONFIGS if (seed, c, m) not in done]
    if not todo: continue
    for proto in ['random', 'temporal']:
        tr, te = make_partitions(proto, seed); Ztr, ytr, Zte, yte, nf, ns = prepare(tr, te, True, seed); data[proto] = (Ztr, ytr, Zte, yte)
    rows = []
    for m, c, kw in todo:
        r = dict(seed=seed, model=m, config=c)
        for proto in ['random', 'temporal']:
            Ztr, ytr, Zte, yte = data[proto]; mdl = build(m, kw, seed); mdl.fit(Ztr, ytr); r['f1_' + proto] = metrics(yte, mdl.predict(Zte))['f1_macro']
        rows.append(r)
        res = pd.concat([res, pd.DataFrame([r])], ignore_index=True); res.to_csv(RESW, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows {len(res)}')
res['gap'] = (res.f1_random - res.f1_temporal) * 100
print('\n=== CIC-IDS 2017: random-minus-temporal F1-macro gap (pp) by configuration, mean over seeds [95% CI] ===')
for m in ['RandomForest', 'LightGBM', 'XGBoost']:
    print('\n', m)
    for c in [c for mm, c, _ in CONFIGS if mm == m]:
        g = res[(res.model == m) & (res.config == c)].gap.values; n = len(g)
        if n < 2: continue
        h = stats.t.ppf(0.975, n - 1) * g.std(ddof=1) / np.sqrt(n)
        f = res[(res.model == m) & (res.config == c)]
        print(f'  {c:18s} n={n} gap={g.mean():6.2f} [{g.mean()-h:6.2f}, {g.mean()+h:6.2f}]  random={f.f1_random.mean():.4f} temporal={f.f1_temporal.mean():.4f}')


seed 42 done in 666s | rows 16
seed 43 done in 644s | rows 32
seed 44 done in 621s | rows 48
seed 45 done in 633s | rows 64
seed 46 done in 628s | rows 80

=== CIC-IDS 2017: random-minus-temporal F1-macro gap (pp) by configuration, mean over seeds [95% CI] ===

 RandomForest
  n100_d10 (paper)   n=5 gap= 56.19 [ 54.58,  57.80]  random=0.9901 temporal=0.4282
  n50_d10            n=5 gap= 55.50 [ 54.32,  56.69]  random=0.9885 temporal=0.4335
  n200_d10           n=5 gap= 56.23 [ 54.58,  57.88]  random=0.9908 temporal=0.4285
  n100_d6            n=5 gap= 35.25 [ 32.55,  37.95]  random=0.9485 temporal=0.5960
  n100_d14           n=5 gap= 57.79 [ 53.66,  61.93]  random=0.9954 temporal=0.4174
  n100_dNone         n=5 gap= 57.46 [ 53.78,  61.14]  random=0.9970 temporal=0.4224

 LightGBM
  n100_d8 (paper)    n=5 gap= 61.53 [ 59.90,  63.16]  random=0.9973 temporal=0.3820
  n50_d8             n=5 gap= 62.38 [ 62.27,  62.49]  random=0.9949 temporal=0.3711
  n200_d8            n=5 gap= 60.69 [ 58.

In [ ]:
# ===== CELL 18: CIC-IDS 2017 MLP, random vs temporal, 10 seeds (paper Table 2 settings) =====
import pandas as pd, numpy as np, os, time
from scipy import stats
from sklearn.neural_network import MLPClassifier
for nm in ['cic', 'make_partitions', 'prepare', 'metrics']: assert nm in globals(), 'run CELL 16 first'
RESM = os.path.join(OUT, 'cic_mlp_results.csv'); ML_SEEDS = list(range(42, 52))
res = pd.read_csv(RESM) if os.path.exists(RESM) else pd.DataFrame()
done = set(zip(res.seed, res.protocol)) if len(res) else set()
for seed in ML_SEEDS:
    t0 = time.time()
    for proto in ['random', 'temporal']:
        if (seed, proto) in done: continue
        tr, te = make_partitions(proto, seed); Ztr, ytr, Zte, yte, nf, ns = prepare(tr, te, True, seed)
        m = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', solver='adam', alpha=1e-4, batch_size=256, learning_rate_init=1e-3,
                          max_iter=100, early_stopping=True, random_state=seed).fit(Ztr, ytr)
        r = dict(seed=seed, protocol=proto, n_iter=m.n_iter_, features=nf, **metrics(yte, m.predict(Zte)))
        res = pd.concat([res, pd.DataFrame([r])], ignore_index=True); res.to_csv(RESM, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows {len(res)}')
a = res[res.protocol == 'random'].sort_values('seed'); b = res[res.protocol == 'temporal'].sort_values('seed')
print('\n=== MLP, CIC-IDS 2017, 10 seeds ===')
for col in ['f1_macro', 'recall', 'fpr']:
    d = (a[col].values - b[col].values) * 100; n = len(d); h = stats.t.ppf(0.975, n - 1) * d.std(ddof=1) / np.sqrt(n)
    try: p = stats.wilcoxon(d, method='exact').pvalue
    except Exception: p = np.nan
    print(f'{col:9s} random={a[col].mean():.4f} temporal={b[col].mean():.4f} gap={d.mean():6.2f} pp CI [{d.mean()-h:.2f}, {d.mean()+h:.2f}] wilcoxon p={p:.5f}')
print('features:', sorted(res.features.unique()), '| mean epochs:', res.n_iter.mean())


seed 42 done in 148s | rows 2
seed 43 done in 173s | rows 4
seed 44 done in 115s | rows 6
seed 45 done in 177s | rows 8
seed 46 done in 116s | rows 10
seed 47 done in 112s | rows 12
seed 48 done in 138s | rows 14
seed 49 done in 114s | rows 16
seed 50 done in 182s | rows 18
seed 51 done in 194s | rows 20

=== MLP, CIC-IDS 2017, 10 seeds ===
f1_macro  random=0.9641 temporal=0.5866 gap= 37.75 pp CI [35.86, 39.63] wilcoxon p=0.00195
recall    random=0.9443 temporal=0.2446 gap= 69.97 pp CI [67.59, 72.35] wilcoxon p=0.00195
fpr       random=0.0147 temporal=0.0114 gap=  0.32 pp CI [-0.10, 0.75] wilcoxon p=0.23242
features: [np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44)] | mean epochs: 41.2


In [ ]:
# ===== CELL 19: measured multiclass temporal check of Proposition 1 on CIC-IDS 2017 (closed-set accuracy on Friday vs the ceiling 1 - rho) =====
import pandas as pd, numpy as np, os, time
from sklearn.preprocessing import LabelEncoder
for nm in ['cic', 'make_partitions', 'FEATS', 'VarianceThreshold', 'StandardScaler']: assert nm in globals(), 'run CELL 16 first'
RESX = os.path.join(OUT, 'cic_multiclass_temporal_results.csv'); MX_SEEDS = list(range(42, 52))
def build_mc(model, seed):
    if model == 'RandomForest': return RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed)
    if model == 'LightGBM': return LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1)
    return XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, eval_metric='mlogloss', verbosity=0)
res = pd.read_csv(RESX) if os.path.exists(RESX) else pd.DataFrame()
done = set(zip(res.seed, res.model)) if len(res) else set()
for seed in MX_SEEDS:
    t0 = time.time(); tr, te = make_partitions('temporal', seed)
    Xtr, Xte = tr[FEATS].astype('float64'), te[FEATS].astype('float64')
    vt = VarianceThreshold(0.01).fit(Xtr); cols = Xtr.columns[vt.get_support()]; Xtr, Xte = Xtr[cols], Xte[cols]
    cm_ = Xtr.corr().abs(); up = cm_.where(np.triu(np.ones(cm_.shape), k=1).astype(bool)); drop = [c for c in up.columns if (up[c] > 0.95).any()]
    Xtr, Xte = Xtr.drop(columns=drop), Xte.drop(columns=drop); sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
    le = LabelEncoder().fit(tr['Label']); ytr = le.transform(tr['Label']); known = te['Label'].isin(le.classes_).values
    rho = 1 - known.mean(); yte = np.where(known, le.transform(te['Label'].where(known, le.classes_[0])), -1)
    for name in ['RandomForest', 'LightGBM', 'XGBoost']:
        if (seed, name) in done: continue
        m = build_mc(name, seed).fit(Ztr, ytr); p = m.predict(Zte); acc = float((p == yte).mean())
        r = dict(seed=seed, model=name, rho=rho, accuracy=acc, ceiling=1 - rho, slack=1 - rho - acc, violated=bool(acc > 1 - rho + 1e-12), n_classes_train=len(le.classes_))
        res = pd.concat([res, pd.DataFrame([r])], ignore_index=True); res.to_csv(RESX, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows {len(res)}')
print('\n=== Multiclass temporal evaluation (train Mon-Thu classes, test Friday), 10 seeds ===')
print('unseen classes in Friday:', sorted(set(cic[cic.day == 'friday'].Label) - set(cic[cic.day.isin(TRAIN_DAYS)].Label)))
print(res.groupby('model').agg(rho=('rho', 'mean'), accuracy=('accuracy', 'mean'), ceiling=('ceiling', 'mean'), slack=('slack', 'mean'), violations=('violated', 'sum'), runs=('seed', 'count')).round(4).to_string())


seed 42 done in 21s | rows 30
seed 43 done in 11s | rows 30
seed 44 done in 12s | rows 30
seed 45 done in 6s | rows 30
seed 46 done in 7s | rows 30
seed 47 done in 7s | rows 30
seed 48 done in 8s | rows 30
seed 49 done in 6s | rows 30
seed 50 done in 7s | rows 30
seed 51 done in 6s | rows 30

=== Multiclass temporal evaluation (train Mon-Thu classes, test Friday), 10 seeds ===
unseen classes in Friday: ['Bot', 'DDoS', 'PortScan']
                rho  accuracy  ceiling   slack  violations  runs
model                                                           
LightGBM      0.411    0.5888    0.589  0.0003           0    10
RandomForest  0.411    0.5726    0.589  0.0164           0    10
XGBoost       0.411    0.5886    0.589  0.0004           0    10


In [ ]:
# ===== CELL 22: CIC-DDoS2019 class totals per day and novel-class fraction rho in both time directions (read-only) =====
import pandas as pd, numpy as np, re, os
ROOT = '/content/drive/MyDrive/UNSW-NB15/CSV Files/CICIDS2019'
c = pd.read_csv(os.path.join(ROOT, 'ddos2019_label_counts.csv'))
def norm(s):
    s = re.sub(r'^drdos_', '', str(s).strip(), flags=re.I); s = re.sub(r'[-_ ]', '', s).lower()
    return {'benign': 'BENIGN', 'udplag': 'UDPLag', 'netbios': 'NetBIOS', 'mssql': 'MSSQL', 'ldap': 'LDAP', 'syn': 'Syn', 'udp': 'UDP', 'portmap': 'Portmap'}.get(s, s)
c['cls'] = c['label'].map(norm)
t = c.groupby(['day', 'label', 'cls']).n.sum().reset_index()
print('RAW LABELS BY DAY'); print(t.to_string(index=False))
tot = c.groupby(['day', 'cls']).n.sum().unstack(0).fillna(0).astype(int)
print('\nNORMALISED TOTALS'); print(tot.to_string())
print('\nrows per day:', tot.sum().to_dict())
print('time order from timestamps: 03-11 files = 2018-11-03 (earlier), 01-12 files = 2018-12-01 (later)')
for name, tr, te in [('A: train 01-12 -> test 03-11 (folder/paper order)', '01-12', '03-11'), ('B: train 03-11 -> test 01-12 (timestamp order)', '03-11', '01-12')]:
    seen = set(tot.index[tot[tr] > 0]); nte = tot[te]; nov = nte[[k for k in tot.index if k not in seen]].sum()
    att = nte.drop('BENIGN', errors='ignore'); nova = att[[k for k in att.index if k not in seen]].sum()
    print(f'\n{name}: novel classes in test = {[k for k in tot.index if k not in seen and nte[k] > 0]}')
    print(f'   rho (all flows) = {nov / nte.sum():.4f} | rho_A (attack flows) = {nova / att.sum():.4f} | ceiling 1-rho = {1 - nov / nte.sum():.4f} | test rows = {nte.sum()}')


RAW LABELS BY DAY
  day         label     cls        n
01-12        BENIGN  BENIGN    56863
01-12     DrDoS_DNS     dns  5071011
01-12    DrDoS_LDAP    LDAP  2179930
01-12   DrDoS_MSSQL   MSSQL  4522492
01-12     DrDoS_NTP     ntp  1202642
01-12 DrDoS_NetBIOS NetBIOS  4093279
01-12    DrDoS_SNMP    snmp  5159870
01-12    DrDoS_SSDP    ssdp  2610611
01-12     DrDoS_UDP     UDP  3134645
01-12           Syn     Syn  1582289
01-12          TFTP    tftp 20082580
01-12       UDP-lag  UDPLag   366461
01-12       WebDDoS webddos      439
03-11        BENIGN  BENIGN    56965
03-11          LDAP    LDAP  1915122
03-11         MSSQL   MSSQL  5787453
03-11       NetBIOS NetBIOS  3657497
03-11       Portmap Portmap   186960
03-11           Syn     Syn  4891500
03-11           UDP     UDP  3867155
03-11        UDPLag  UDPLag     1873

NORMALISED TOTALS
day         01-12    03-11
cls                       
BENIGN      56863    56965
LDAP      2179930  1915122
MSSQL     4522492  5787453
NetBIOS   4093

In [ ]:
# ===== CELL 23: CIC-DDoS2019 uniform sampled pools (about 1,000,000 flows per day), resumable per file =====
import pandas as pd, numpy as np, glob, os, time, re, zlib
ROOT = '/content/drive/MyDrive/UNSW-NB15/CSV Files/CICIDS2019'; PARTS = os.path.join(ROOT, 'pool_parts'); os.makedirs(PARTS, exist_ok=True)
DAY_ROWS = {'01-12': 50063112, '03-11': 20364525}; POOL = 1000000
DROP = {'Unnamed: 0', 'Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Timestamp', 'SimillarHTTP'}
def norm(s):
    s = re.sub(r'^drdos_', '', str(s).strip(), flags=re.I); s = re.sub(r'[-_ ]', '', s).lower()
    return {'benign': 'BENIGN', 'udplag': 'UDPLag', 'netbios': 'NetBIOS', 'mssql': 'MSSQL', 'ldap': 'LDAP', 'syn': 'Syn', 'udp': 'UDP', 'portmap': 'Portmap'}.get(s, s)
t0 = time.time()
files = sorted(f for f in glob.glob(os.path.join(ROOT, '**', '*.csv'), recursive=True) if not os.path.basename(f).startswith('.~lock') and 'ddos2019_' not in os.path.basename(f))
print(len(files), 'csv files')
for f in files:
    day = '01-12' if '01-12' in f else '03-11'
    part = os.path.join(PARTS, day + '_' + os.path.basename(f).replace('.csv', '.parquet'))
    if os.path.exists(part): continue
    p = POOL / DAY_ROWS[day]; rng = np.random.default_rng(zlib.crc32(part.encode())); keep = []
    for ch in pd.read_csv(f, chunksize=500000, low_memory=False):
        ch.columns = [c.strip() for c in ch.columns]
        s = ch[rng.random(len(ch)) < p]
        if len(s): keep.append(s.drop(columns=[c for c in DROP if c in s.columns]))
    d = pd.concat(keep, ignore_index=True); lab = d.pop('Label').map(norm).values
    d = d.apply(pd.to_numeric, errors='coerce').astype('float32'); d['Label'] = lab; d['day'] = day
    d.to_parquet(part); print(f'{day} {os.path.basename(f)} kept={len(d)} cols={d.shape[1]} | {time.time()-t0:.0f}s', flush=True)
for day in DAY_ROWS:
    parts = sorted(glob.glob(os.path.join(PARTS, day + '_*.parquet')))
    pool = pd.concat([pd.read_parquet(x) for x in parts], ignore_index=True)
    pool.to_parquet(os.path.join(ROOT, f'ddos2019_pool_{day}.parquet'))
    print(day, 'pool rows', len(pool), pool.Label.value_counts().to_dict())
print('done', round(time.time() - t0), 's')


18 csv files
01-12 DrDoS_DNS.csv kept=101385 cols=81 | 141s
01-12 DrDoS_LDAP.csv kept=43253 cols=81 | 189s
01-12 DrDoS_MSSQL.csv kept=90256 cols=81 | 317s
01-12 DrDoS_NTP.csv kept=24613 cols=81 | 349s
01-12 DrDoS_NetBIOS.csv kept=81254 cols=81 | 439s
01-12 DrDoS_SNMP.csv kept=103009 cols=81 | 552s
01-12 DrDoS_SSDP.csv kept=52475 cols=81 | 627s
01-12 DrDoS_UDP.csv kept=62672 cols=81 | 702s
01-12 Syn.csv kept=31661 cols=81 | 735s
01-12 TFTP.csv kept=401988 cols=81 | 1231s
01-12 UDPLag.csv kept=7533 cols=81 | 1241s
03-11 LDAP.csv kept=103473 cols=81 | 1290s
03-11 MSSQL.csv kept=284222 cols=81 | 1420s
03-11 NetBIOS.csv kept=169907 cols=81 | 1497s
03-11 Portmap.csv kept=9314 cols=81 | 1502s
03-11 Syn.csv kept=211853 cols=81 | 1602s
03-11 UDP.csv kept=185626 cols=81 | 1697s
03-11 UDPLag.csv kept=35323 cols=81 | 1715s
01-12 pool rows 1000099 {'tftp': 401462, 'snmp': 102970, 'dns': 101309, 'MSSQL': 90223, 'NetBIOS': 81222, 'UDP': 62636, 'ssdp': 52454, 'LDAP': 43214, 'Syn': 31657, 'ntp': 24293,

In [ ]:
# ===== CELL 24: CIC-DDoS2019 multiclass chronological test of Proposition 1 (A: train 01-12 test 03-11; B: train 03-11 test 01-12 = true time order) + random split =====
import pandas as pd, numpy as np, os, time, warnings; warnings.filterwarnings('ignore')
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
ROOT = '/content/drive/MyDrive/UNSW-NB15/CSV Files/CICIDS2019'; RES = os.path.join(ROOT, 'ddos2019_multiclass_results.csv')
NTR, NTE, SEEDS = 200000, 100000, list(range(42, 52))
pool = {d: pd.read_parquet(os.path.join(ROOT, f'ddos2019_pool_{d}.parquet')) for d in ('01-12', '03-11')}
FE = [c for c in pool['01-12'].columns if c not in ('Label', 'day')]
for d in pool:
    X = pool[d][FE].replace([np.inf, -np.inf], np.nan); ok = X.notna().all(axis=1).values
    print(d, 'pool rows', len(pool[d]), 'dropped (NaN/inf)', int((~ok).sum())); pool[d] = pool[d][ok].reset_index(drop=True)
def build(model, seed):
    if model == 'RandomForest': return RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed)
    if model == 'LightGBM': return LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1)
    return XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, eval_metric='mlogloss', verbosity=0)
def draw(d, n, seed): return pool[d].sample(n=min(n, len(pool[d])), random_state=seed)
def partitions(design, seed):
    if design == 'A_temporal': return draw('01-12', NTR, seed), draw('03-11', NTE, seed)
    if design == 'B_temporal': return draw('03-11', NTR, seed), draw('01-12', NTE, seed)
    allp = pd.concat([draw('01-12', 100000, seed), draw('03-11', 100000, seed)], ignore_index=True)
    vc = allp.Label.value_counts(); grp = allp.Label.where(allp.Label.map(vc) >= 5, '__rare__')
    if (grp == '__rare__').sum() < 2: grp = allp.Label.where(allp.Label.map(vc) >= 2, allp.Label.iloc[0])
    return train_test_split(allp, test_size=0.2, stratify=grp, random_state=seed)
res = pd.read_csv(RES) if os.path.exists(RES) else pd.DataFrame()
done = set(zip(res.seed, res.design, res.model)) if len(res) else set()
for seed in SEEDS:
    t0 = time.time()
    for design in ['B_temporal', 'A_temporal', 'random']:
        if all((seed, design, m) in done for m in ('RandomForest', 'LightGBM', 'XGBoost')): continue
        tr, te = partitions(design, seed)
        Xtr, Xte = tr[FE].astype('float64'), te[FE].astype('float64')
        vt = VarianceThreshold(0.01).fit(Xtr); cols = Xtr.columns[vt.get_support()]; Xtr, Xte = Xtr[cols], Xte[cols]
        cm_ = Xtr.corr().abs(); up = cm_.where(np.triu(np.ones(cm_.shape), k=1).astype(bool)); drop = [c for c in up.columns if (up[c] > 0.95).any()]
        Xtr, Xte = Xtr.drop(columns=drop), Xte.drop(columns=drop); sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
        le = LabelEncoder().fit(tr['Label']); ytr = le.transform(tr['Label']); known = te['Label'].isin(le.classes_).values
        rho = 1 - known.mean(); yte = np.where(known, le.transform(te['Label'].where(known, le.classes_[0])), -1)
        novel = sorted(set(te['Label']) - set(le.classes_))
        for name in ['RandomForest', 'LightGBM', 'XGBoost']:
            if (seed, design, name) in done: continue
            m = build(name, seed).fit(Ztr, ytr); p = m.predict(Zte); acc = float((p == yte).mean())
            r = dict(seed=seed, design=design, model=name, rho=rho, accuracy=acc, ceiling=1 - rho, slack=1 - rho - acc, violated=bool(acc > 1 - rho + 1e-12),
                     acc_known=float((p[known] == yte[known]).mean()), n_train=len(tr), n_test=len(te), n_features=Ztr.shape[1], n_classes_train=len(le.classes_), novel='+'.join(novel))
            res = pd.concat([res, pd.DataFrame([r])], ignore_index=True); res.to_csv(RES, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows {len(res)}', flush=True)
print('\n=== CIC-DDoS2019 multiclass (means over seeds) ===')
print(res.groupby(['design', 'model']).agg(rho=('rho', 'mean'), accuracy=('accuracy', 'mean'), ceiling=('ceiling', 'mean'), slack=('slack', 'mean'), acc_known=('acc_known', 'mean'), violations=('violated', 'sum'), runs=('seed', 'count')).round(4).to_string())


01-12 pool rows 1000099 dropped (NaN/inf) 27321
03-11 pool rows 999718 dropped (NaN/inf) 39428
seed 42 done in 277s | rows 9
seed 43 done in 271s | rows 18
seed 44 done in 269s | rows 27
seed 45 done in 270s | rows 36
seed 46 done in 276s | rows 45
seed 47 done in 270s | rows 54
seed 48 done in 267s | rows 63
seed 49 done in 264s | rows 72
seed 50 done in 262s | rows 81
seed 51 done in 271s | rows 90

=== CIC-DDoS2019 multiclass (means over seeds) ===
                            rho  accuracy  ceiling   slack  acc_known  violations  runs
design     model                                                                       
A_temporal LightGBM      0.0089    0.8041   0.9911  0.1870     0.8113           0    10
           RandomForest  0.0089    0.7806   0.9911  0.2104     0.7877           0    10
           XGBoost       0.0089    0.7643   0.9911  0.2268     0.7712           0    10
B_temporal LightGBM      0.6850    0.2532   0.3150  0.0619     0.8036           0    10
           Rando

In [ ]:
# ===== CELL 25: CIC-DDoS2019 why is accuracy low in direction A? confusion matrices + weighted vs unweighted models (5 seeds) =====
import pandas as pd, numpy as np, os, time, warnings; warnings.filterwarnings('ignore')
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
ROOT = '/content/drive/MyDrive/UNSW-NB15/CSV Files/CICIDS2019'
assert os.path.isdir(ROOT), 'Drive is not mounted: run the mount cell first'
RES2 = os.path.join(ROOT, 'ddos2019_why_results.csv'); CONF2 = os.path.join(ROOT, 'ddos2019_why_confusion.csv')
NTR, NTE, SEEDS = 200000, 100000, list(range(42, 47))
if 'pool' not in globals():
    pool = {d: pd.read_parquet(os.path.join(ROOT, f'ddos2019_pool_{d}.parquet')) for d in ('01-12', '03-11')}
    for d in pool:
        X = pool[d].drop(columns=['Label', 'day']).replace([np.inf, -np.inf], np.nan); pool[d] = pool[d][X.notna().all(axis=1).values].reset_index(drop=True)
FE = [c for c in pool['01-12'].columns if c not in ('Label', 'day')]
def draw(d, n, seed): return pool[d].sample(n=min(n, len(pool[d])), random_state=seed)
def partitions(design, seed):
    if design == 'A_temporal': return draw('01-12', NTR, seed), draw('03-11', NTE, seed)
    if design == 'B_temporal': return draw('03-11', NTR, seed), draw('01-12', NTE, seed)
    allp = pd.concat([draw('01-12', 100000, seed), draw('03-11', 100000, seed)], ignore_index=True)
    vc = allp.Label.value_counts(); grp = allp.Label.where(allp.Label.map(vc) >= 5, '__rare__')
    if (grp == '__rare__').sum() < 2: grp = allp.Label.where(allp.Label.map(vc) >= 2, allp.Label.iloc[0])
    return train_test_split(allp, test_size=0.2, stratify=grp, random_state=seed)
def prepare(tr, te):
    Xtr, Xte = tr[FE].astype('float64'), te[FE].astype('float64')
    vt = VarianceThreshold(0.01).fit(Xtr); cols = Xtr.columns[vt.get_support()]; Xtr, Xte = Xtr[cols], Xte[cols]
    cm_ = Xtr.corr().abs(); up = cm_.where(np.triu(np.ones(cm_.shape), k=1).astype(bool)); drop = [c for c in up.columns if (up[c] > 0.95).any()]
    Xtr, Xte = Xtr.drop(columns=drop), Xte.drop(columns=drop); sc = StandardScaler().fit(Xtr)
    return sc.transform(Xtr), sc.transform(Xte)
def fit_predict(model, weighted, seed, Ztr, ytr, Zte):
    cw = 'balanced' if weighted else None
    if model == 'RandomForest': m = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight=cw, n_jobs=-1, random_state=seed)
    elif model == 'LightGBM': m = LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight=cw, random_state=seed, n_jobs=-1, verbose=-1)
    else: m = XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, eval_metric='mlogloss', verbosity=0)
    if model == 'XGBoost' and weighted: m.fit(Ztr, ytr, sample_weight=compute_sample_weight('balanced', ytr))
    else: m.fit(Ztr, ytr)
    return m.predict(Zte)
res = pd.read_csv(RES2) if os.path.exists(RES2) else pd.DataFrame(); conf = pd.read_csv(CONF2) if os.path.exists(CONF2) else pd.DataFrame()
done = set(zip(res.seed, res.design, res.model, res.variant)) if len(res) else set()
for seed in SEEDS:
    t0 = time.time()
    for design in ['B_temporal', 'A_temporal', 'random']:
        todo = [(m, w) for m in ('RandomForest', 'LightGBM', 'XGBoost') for w in (False, True) if (seed, design, m, 'balanced' if w else 'unweighted') not in done]
        if not todo: continue
        tr, te = partitions(design, seed); Ztr, Zte = prepare(tr, te)
        le = LabelEncoder().fit(tr['Label']); ytr = le.transform(tr['Label']); known = te['Label'].isin(le.classes_).values
        rho = 1 - known.mean(); yte = np.where(known, le.transform(te['Label'].where(known, le.classes_[0])), -1)
        for model, w in todo:
            p = fit_predict(model, w, seed, Ztr, ytr, Zte); acc = float((p == yte).mean()); var = 'balanced' if w else 'unweighted'
            res = pd.concat([res, pd.DataFrame([dict(seed=seed, design=design, model=model, variant=var, rho=rho, accuracy=acc, acc_known=float((p[known] == yte[known]).mean()), violated=bool(acc > 1 - rho + 1e-12))])], ignore_index=True)
            cc = pd.DataFrame({'true': te['Label'].values, 'pred': le.classes_[p]}).value_counts().reset_index(name='n')
            cc.insert(0, 'variant', var); cc.insert(0, 'model', model); cc.insert(0, 'design', design); cc.insert(0, 'seed', seed)
            conf = pd.concat([conf, cc], ignore_index=True)
        res.to_csv(RES2, index=False); conf.to_csv(CONF2, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows {len(res)}', flush=True)
print('\n=== accuracy by design x model x variant (means over seeds) ===')
print(res.groupby(['design', 'model', 'variant']).agg(rho=('rho', 'mean'), accuracy=('accuracy', 'mean'), acc_known=('acc_known', 'mean'), violations=('violated', 'sum'), runs=('seed', 'count')).round(4).to_string())
for design, model, var in [('A_temporal', 'RandomForest', 'unweighted'), ('A_temporal', 'XGBoost', 'unweighted'), ('random', 'RandomForest', 'unweighted')]:
    g = conf[(conf.design == design) & (conf.model == model) & (conf.variant == var)].groupby(['true', 'pred']).n.sum().reset_index()
    tot = g.groupby('true').n.transform('sum'); g['share'] = g.n / tot
    rec = g[g.true == g.pred].set_index('true').share.round(3).to_dict()
    off = g[g.true != g.pred].sort_values('n', ascending=False).head(8)
    print(f'\n--- {design} {model} {var}: per-class recall {rec}')
    print(off.assign(share=off.share.round(3)).to_string(index=False))


seed 42 done in 0s | rows 90
seed 43 done in 0s | rows 90
seed 44 done in 0s | rows 90
seed 45 done in 0s | rows 90
seed 46 done in 0s | rows 90

=== accuracy by design x model x variant (means over seeds) ===
                                       rho  accuracy  acc_known  violations  runs
design     model        variant                                                  
A_temporal LightGBM     balanced    0.0087    0.8046     0.8117           0     5
                        unweighted  0.0087    0.1719     0.1734           0     5
           RandomForest balanced    0.0087    0.7772     0.7840           0     5
                        unweighted  0.0087    0.8555     0.8630           0     5
           XGBoost      balanced    0.0087    0.8322     0.8395           0     5
                        unweighted  0.0087    0.7782     0.7850           0     5
B_temporal LightGBM     balanced    0.6849    0.2539     0.8057           0     5
                        unweighted  0.6849    0.2046

In [ ]:
# ===== CELL 26: single-family hold-out on CIC-DDoS2019 (detection as "not predicted BENIGN" + feature-space proximity), 3 seeds =====
assert 'prepare' in globals() and 'pool' in globals(), 'run CELL 25 first'
from sklearn.neighbors import NearestNeighbors
from scipy import stats
RES3 = os.path.join(ROOT, 'ddos2019_holdout_results.csv'); SEEDS3 = [42, 43, 44]; NTR3 = 100000; NEV = 5000
allp = pd.concat([pool['01-12'], pool['03-11']], ignore_index=True)
FAMS = sorted(set(allp.Label) - {'BENIGN'}); print('families:', FAMS, '| pooled rows', len(allp), '| benign', int((allp.Label == 'BENIGN').sum()))
res3 = pd.read_csv(RES3) if os.path.exists(RES3) else pd.DataFrame()
done3 = set(zip(res3.seed, res3.family)) if len(res3) else set()
for seed in SEEDS3:
    t0 = time.time()
    for fam in FAMS:
        if (seed, fam) in done3: continue
        fam_rows = allp[allp.Label == fam]
        if len(fam_rows) < 5: continue
        ev = fam_rows.sample(n=min(NEV, len(fam_rows)), random_state=seed); R = allp.drop(ev.index)
        trs = {'base': R.sample(n=NTR3, random_state=seed), 'loo': R[R.Label != fam].sample(n=NTR3, random_state=seed)}
        out = {}
        for k, tr in trs.items():
            Ztr, Zev = prepare(tr, ev); le = LabelEncoder().fit(tr['Label']); ytr = le.transform(tr['Label'])
            out[k] = dict(Ztr=Ztr, Zev=Zev, le=le, ytr=ytr, labels=tr['Label'].values)
        o = out['loo']; rng = np.random.RandomState(seed)
        ia = np.where(o['labels'] != 'BENIGN')[0]; inn = np.where(o['labels'] == 'BENIGN')[0]
        ia = rng.choice(ia, min(len(ia), 20000), replace=False); ee = rng.choice(len(o['Zev']), min(len(o['Zev']), 1000), replace=False)
        d_att = NearestNeighbors(n_neighbors=1).fit(o['Ztr'][ia]).kneighbors(o['Zev'][ee])[0].ravel()
        d_nor = NearestNeighbors(n_neighbors=1).fit(o['Ztr'][inn]).kneighbors(o['Zev'][ee])[0].ravel()
        geo = dict(ratio=float(np.median(d_nor / np.maximum(d_att, 1e-9))), frac_closer_att=float((d_att < d_nor).mean()), n_benign_ref=len(inn))
        rows = []
        for model in ('RandomForest', 'LightGBM', 'XGBoost'):
            w = model != 'XGBoost'; r = dict(seed=seed, family=fam, model=model, n_eval=len(ev), **geo)
            for k in ('loo', 'base'):
                o = out[k]; p = o['le'].classes_[fit_predict(model, w, seed, o['Ztr'], o['ytr'], o['Zev'])]
                r['det_' + k] = float((p != 'BENIGN').mean()); r['exact_' + k] = float((p == fam).mean())
                if k == 'loo':
                    vc = pd.Series(p).value_counts(); r['top_pred'] = vc.index[0]; r['top_share'] = float(vc.iloc[0] / len(p))
            rows.append(r)
        res3 = pd.concat([res3, pd.DataFrame(rows)], ignore_index=True); res3.to_csv(RES3, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows {len(res3)}', flush=True)
g = res3.groupby(['model', 'family']).agg(det_loo=('det_loo', 'mean'), det_base=('det_base', 'mean'), exact_base=('exact_base', 'mean'), top_pred=('top_pred', lambda s: s.mode().iloc[0]), top_share=('top_share', 'mean'), ratio=('ratio', 'mean'), fc=('frac_closer_att', 'mean'), n_eval=('n_eval', 'mean')).reset_index()
print('\n=== single-family hold-out: share of held-out flows NOT predicted BENIGN (det_loo) vs when family is in training (det_base) ===')
for mdl in ('RandomForest', 'LightGBM', 'XGBoost'):
    print('\n', mdl); print(g[g.model == mdl].sort_values('det_loo').drop(columns='model').round(3).to_string(index=False))
print('\n=== Spearman across families: det_loo vs geometry ===')
for mdl in ('RandomForest', 'LightGBM', 'XGBoost'):
    s = g[g.model == mdl]
    for col in ('fc', 'ratio'):
        r_, p_ = stats.spearmanr(s[col], s['det_loo']); print(f'{mdl:12s} {col:6s} rho={r_:+.2f} p={p_:.3f} n={len(s)}')


families: ['LDAP', 'MSSQL', 'NetBIOS', 'Portmap', 'Syn', 'UDP', 'UDPLag', 'dns', 'ntp', 'snmp', 'ssdp', 'tftp', 'webddos'] | pooled rows 1933068 | benign 3923
seed 42 done in 1321s | rows 39
seed 43 done in 1306s | rows 78
seed 44 done in 1285s | rows 117

=== single-family hold-out: share of held-out flows NOT predicted BENIGN (det_loo) vs when family is in training (det_base) ===

 RandomForest
 family  det_loo  det_base  exact_base top_pred  top_share        ratio    fc  n_eval
    Syn    0.323     0.998       0.898   BENIGN      0.677 1.070520e+02 0.924  5000.0
webddos    0.333     0.333       0.000   BENIGN      0.667 4.430000e-01 0.222     6.0
    ntp    0.997     0.998       0.952      dns      0.409 2.117000e+00 0.968  5000.0
 UDPLag    0.998     0.998       0.750      Syn      0.844 1.458686e+07 0.687  5000.0
Portmap    1.000     1.000       0.813  NetBIOS      0.992 1.428151e+09 0.999  5000.0
    dns    1.000     1.000       0.498     LDAP      0.757 1.901623e+09 0.999  5000.

In [ ]:
# ===== CELL 27: CIC-DDoS2019 why does unweighted LightGBM collapse? diagnostic on seed 42 (A_temporal and random) =====
assert 'prepare' in globals() and 'partitions' in globals(), 'run CELL 25 first'
RES4 = os.path.join(ROOT, 'ddos2019_lgbm_diag_results.csv')
VARIANTS = {
    'default (paper setting, unweighted)': dict(),
    'learning_rate 0.05': dict(learning_rate=0.05),
    'learning_rate 0.02': dict(learning_rate=0.02),
    'reg_lambda 1, min_child_samples 50': dict(reg_lambda=1.0, min_child_samples=50),
    'min_sum_hessian 0.01, reg_lambda 1': dict(min_sum_hessian_in_leaf=0.01, reg_lambda=1.0),
    'num_leaves 15, lr 0.05, reg_lambda 1': dict(num_leaves=15, learning_rate=0.05, reg_lambda=1.0),
}
rows = []
for design in ['A_temporal', 'random']:
    tr, te = partitions(design, 42); Ztr, Zte = prepare(tr, te)
    le = LabelEncoder().fit(tr['Label']); ytr = le.transform(tr['Label']); known = te['Label'].isin(le.classes_).values
    yte = np.where(known, le.transform(te['Label'].where(known, le.classes_[0])), -1)
    for name, kw in VARIANTS.items():
        t0 = time.time(); kw = dict(kw)
        m = LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=kw.pop('num_leaves', 31), colsample_bytree=0.8, subsample=0.8, class_weight=None, random_state=42, n_jobs=-1, verbose=-1, **kw).fit(Ztr, ytr)
        ptr = m.predict(Ztr); p = m.predict(Zte); vc = pd.Series(le.classes_[p]).value_counts()
        rows.append(dict(design=design, variant=name, train_acc=float((ptr == ytr).mean()), test_acc=float((p == yte).mean()), n_pred_classes=int(vc.shape[0]), top_pred=vc.index[0], top_share=float(vc.iloc[0] / len(p)), secs=round(time.time() - t0)))
        print(rows[-1], flush=True)
    del Ztr, Zte
d = pd.DataFrame(rows); d.to_csv(RES4, index=False)
print('\n=== LightGBM unweighted diagnostic (seed 42) ===')
print(d.round(4).to_string(index=False))


{'design': 'A_temporal', 'variant': 'default (paper setting, unweighted)', 'train_acc': 0.252, 'test_acc': 0.2778, 'n_pred_classes': 11, 'top_pred': 'NetBIOS', 'top_share': 0.72384, 'secs': 75}
{'design': 'A_temporal', 'variant': 'learning_rate 0.05', 'train_acc': 0.83998, 'test_acc': 0.64591, 'n_pred_classes': 13, 'top_pred': 'NetBIOS', 'top_share': 0.27326, 'secs': 55}
{'design': 'A_temporal', 'variant': 'learning_rate 0.02', 'train_acc': 0.85845, 'test_acc': 0.6334, 'n_pred_classes': 13, 'top_pred': 'MSSQL', 'top_share': 0.27921, 'secs': 54}
{'design': 'A_temporal', 'variant': 'reg_lambda 1, min_child_samples 50', 'train_acc': 0.863885, 'test_acc': 0.84299, 'n_pred_classes': 12, 'top_pred': 'MSSQL', 'top_share': 0.27844, 'secs': 50}
{'design': 'A_temporal', 'variant': 'min_sum_hessian 0.01, reg_lambda 1', 'train_acc': 0.86403, 'test_acc': 0.84406, 'n_pred_classes': 12, 'top_pred': 'MSSQL', 'top_share': 0.27839, 'secs': 51}
{'design': 'A_temporal', 'variant': 'num_leaves 15, lr 0.05,

In [ ]:
# ===== CELL 28: extend the single-family hold-out to 8 seeds (42-49); resumes from the saved CSV, families with fewer than 100 flows are excluded from the summary =====
assert 'prepare' in globals() and 'pool' in globals() and 'fit_predict' in globals(), 'run CELL 25 first'
from sklearn.neighbors import NearestNeighbors
from scipy import stats
RES3 = os.path.join(ROOT, 'ddos2019_holdout_results.csv'); SEEDS3 = list(range(42, 50)); NTR3 = 100000; NEV = 5000
allp = pd.concat([pool['01-12'], pool['03-11']], ignore_index=True)
FAMS = sorted(set(allp.Label) - {'BENIGN'}); print('families:', FAMS, '| pooled rows', len(allp), '| benign', int((allp.Label == 'BENIGN').sum()))
res3 = pd.read_csv(RES3) if os.path.exists(RES3) else pd.DataFrame()
done3 = set(zip(res3.seed, res3.family)) if len(res3) else set()
for seed in SEEDS3:
    t0 = time.time()
    for fam in FAMS:
        if (seed, fam) in done3: continue
        fam_rows = allp[allp.Label == fam]
        if len(fam_rows) < 5: continue
        ev = fam_rows.sample(n=min(NEV, len(fam_rows)), random_state=seed); R = allp.drop(ev.index)
        trs = {'base': R.sample(n=NTR3, random_state=seed), 'loo': R[R.Label != fam].sample(n=NTR3, random_state=seed)}
        out = {}
        for k, tr in trs.items():
            Ztr, Zev = prepare(tr, ev); le = LabelEncoder().fit(tr['Label']); ytr = le.transform(tr['Label'])
            out[k] = dict(Ztr=Ztr, Zev=Zev, le=le, ytr=ytr, labels=tr['Label'].values)
        o = out['loo']; rng = np.random.RandomState(seed)
        ia = np.where(o['labels'] != 'BENIGN')[0]; inn = np.where(o['labels'] == 'BENIGN')[0]
        ia = rng.choice(ia, min(len(ia), 20000), replace=False); ee = rng.choice(len(o['Zev']), min(len(o['Zev']), 1000), replace=False)
        d_att = NearestNeighbors(n_neighbors=1).fit(o['Ztr'][ia]).kneighbors(o['Zev'][ee])[0].ravel()
        d_nor = NearestNeighbors(n_neighbors=1).fit(o['Ztr'][inn]).kneighbors(o['Zev'][ee])[0].ravel()
        geo = dict(ratio=float(np.median(d_nor / np.maximum(d_att, 1e-9))), frac_closer_att=float((d_att < d_nor).mean()), n_benign_ref=len(inn))
        rows = []
        for model in ('RandomForest', 'LightGBM', 'XGBoost'):
            w = model != 'XGBoost'; r = dict(seed=seed, family=fam, model=model, n_eval=len(ev), **geo)
            for k in ('loo', 'base'):
                o = out[k]; p = o['le'].classes_[fit_predict(model, w, seed, o['Ztr'], o['ytr'], o['Zev'])]
                r['det_' + k] = float((p != 'BENIGN').mean()); r['exact_' + k] = float((p == fam).mean())
                if k == 'loo':
                    vc = pd.Series(p).value_counts(); r['top_pred'] = vc.index[0]; r['top_share'] = float(vc.iloc[0] / len(p))
            rows.append(r)
        res3 = pd.concat([res3, pd.DataFrame(rows)], ignore_index=True); res3.to_csv(RES3, index=False)
    print(f'seed {seed} done in {time.time()-t0:.0f}s | rows {len(res3)}', flush=True)
big = res3[res3.n_eval >= 100]
g = big.groupby(['model', 'family']).agg(seeds=('seed', 'nunique'), det_loo=('det_loo', 'mean'), det_loo_sd=('det_loo', 'std'), det_base=('det_base', 'mean'), exact_base=('exact_base', 'mean'), exact_loo=('exact_loo', 'mean'), top_pred=('top_pred', lambda s: s.mode().iloc[0]), top_share=('top_share', 'mean'), ratio=('ratio', 'mean'), fc=('frac_closer_att', 'mean'), n_eval=('n_eval', 'mean')).reset_index()
print('\n=== single-family hold-out, 8 seeds, families with >=100 flows ===')
for mdl in ('RandomForest', 'LightGBM', 'XGBoost'):
    print('\n', mdl); print(g[g.model == mdl].sort_values('det_loo').drop(columns='model').round(3).to_string(index=False))
print('\n=== Spearman across families: det_loo vs geometry ===')
for mdl in ('RandomForest', 'LightGBM', 'XGBoost'):
    s = g[g.model == mdl]
    for col in ('fc', 'ratio'):
        r_, p_ = stats.spearmanr(s[col], s['det_loo']); print(f'{mdl:12s} {col:6s} rho={r_:+.2f} p={p_:.3f} n={len(s)}')
print('\nfamilies dropped for <100 flows:', sorted(set(res3.family) - set(big.family)))


families: ['LDAP', 'MSSQL', 'NetBIOS', 'Portmap', 'Syn', 'UDP', 'UDPLag', 'dns', 'ntp', 'snmp', 'ssdp', 'tftp', 'webddos'] | pooled rows 1933068 | benign 3923
seed 42 done in 0s | rows 117
seed 43 done in 0s | rows 117
seed 44 done in 0s | rows 117
seed 45 done in 1326s | rows 156
seed 46 done in 1306s | rows 195
seed 47 done in 1320s | rows 234
seed 48 done in 1321s | rows 273
seed 49 done in 1294s | rows 312

=== single-family hold-out, 8 seeds, families with >=100 flows ===

 RandomForest
 family  seeds  det_loo  det_loo_sd  det_base  exact_base  exact_loo top_pred  top_share        ratio    fc  n_eval
    Syn      8    0.329       0.141     0.999       0.899        0.0   BENIGN      0.671 1.233390e+02 0.928  5000.0
 UDPLag      8    0.996       0.005     0.998       0.767        0.0      Syn      0.840 1.602402e+07 0.821  5000.0
    ntp      8    0.997       0.000     0.998       0.951        0.0      UDP      0.442 3.022000e+00 0.955  5000.0
Portmap      8    1.000       0.000    

In [ ]:
# ===== CELL 29: CIC-DDoS2019 timestamp check from the raw CSVs (read-only, Timestamp column only) =====
import os, glob, time
import pandas as pd
assert 'ROOT' in globals(), 'run the setup cell that defines ROOT first'
cands = [f for f in glob.glob(os.path.join(ROOT, '**', '*.csv'), recursive=True) if ('01-12' in f or '03-11' in f)]
print('CSV files found:', len(cands))
for f in cands[:40]: print('  ', f, round(os.path.getsize(f) / 1e9, 2), 'GB')
summary = {}
for f in cands:
    day = '01-12' if '01-12' in f else '03-11'
    try:
        head = pd.read_csv(f, nrows=0); col = [c for c in head.columns if 'imestamp' in c]
        if not col: continue
        t0 = time.time(); lo = hi = None; dates = {}
        for ch in pd.read_csv(f, usecols=[col[0]], chunksize=2_000_000, low_memory=False):
            t = pd.to_datetime(ch[col[0]], errors='coerce').dropna()
            if t.empty: continue
            lo = t.min() if lo is None else min(lo, t.min()); hi = t.max() if hi is None else max(hi, t.max())
            for d, n in t.dt.date.value_counts().items(): dates[d] = dates.get(d, 0) + int(n)
        s = summary.setdefault(day, dict(lo=lo, hi=hi, dates={}))
        s['lo'] = min(s['lo'], lo); s['hi'] = max(s['hi'], hi)
        for d, n in dates.items(): s['dates'][d] = s['dates'].get(d, 0) + n
        print(os.path.basename(f), '| min', lo, '| max', hi, '|', round(time.time() - t0), 's', flush=True)
    except Exception as e:
        print('skip', f, type(e).__name__, e)
print('\n=== per day ===')
for day, s in summary.items():
    print(day, '| first', s['lo'], '| last', s['hi'], '| dates', dict(sorted(s['dates'].items())))

CSV files found: 18
   /content/drive/MyDrive/UNSW-NB15/CSV Files/CICIDS2019/CSV-01-12/01-12/UDPLag.csv 0.16 GB
   /content/drive/MyDrive/UNSW-NB15/CSV Files/CICIDS2019/CSV-01-12/01-12/TFTP.csv 9.3 GB
   /content/drive/MyDrive/UNSW-NB15/CSV Files/CICIDS2019/CSV-01-12/01-12/Syn.csv 0.64 GB
   /content/drive/MyDrive/UNSW-NB15/CSV Files/CICIDS2019/CSV-01-12/01-12/DrDoS_UDP.csv 1.51 GB
   /content/drive/MyDrive/UNSW-NB15/CSV Files/CICIDS2019/CSV-01-12/01-12/DrDoS_SSDP.csv 1.25 GB
   /content/drive/MyDrive/UNSW-NB15/CSV Files/CICIDS2019/CSV-01-12/01-12/DrDoS_SNMP.csv 2.17 GB
   /content/drive/MyDrive/UNSW-NB15/CSV Files/CICIDS2019/CSV-01-12/01-12/DrDoS_NTP.csv 0.65 GB
   /content/drive/MyDrive/UNSW-NB15/CSV Files/CICIDS2019/CSV-01-12/01-12/DrDoS_NetBIOS.csv 1.7 GB
   /content/drive/MyDrive/UNSW-NB15/CSV Files/CICIDS2019/CSV-01-12/01-12/DrDoS_MSSQL.csv 1.89 GB
   /content/drive/MyDrive/UNSW-NB15/CSV Files/CICIDS2019/CSV-01-12/01-12/DrDoS_LDAP.csv 0.92 GB
   /content/drive/MyDrive/UNSW-NB15/C

In [ ]:
# ===== CELL 38: extra attack windows, ORIGINAL pipeline (functions from CELL 37 must be in memory), captured + dedup =====
# Windows tried: GoldenEye, Slowloris, Slowhttptest (Wednesday), Web Attack Brute Force, XSS (Thursday morning).
# A window is used only if BOTH sides of the 80/20 chronological cut hold at least MIN_SIDE attack flows. New result file only.
import os, gc, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from scipy import stats

RUNS2 = OUT + '/cic2017_orig_pipeline_extra_runs.csv'
MIN_SIDE = 100
KEYS = {'GoldenEye': 'goldeneye', 'Slowloris': 'slowloris', 'Slowhttptest': 'slowhttp',
        'WebBruteForce': 'brute force', 'WebXSS': 'xss'}
FILES = ['Wednesday.csv', 'Thursday.csv', 'Thursday+1.csv']

# --- resolve exact label strings and files ---
EXTRA = {}
for fn in FILES:
    p = D + '/' + fn
    if not os.path.exists(p):
        print('missing file', fn); continue
    lab = pd.read_csv(p, usecols=lambda c: c.strip() == 'Label', low_memory=False)
    lab.columns = ['Label']
    vc = lab['Label'].value_counts()
    print(fn, dict(vc))
    for name, key in KEYS.items():
        for l in vc.index:
            if key in str(l).lower() and name not in EXTRA:
                EXTRA[name] = (fn, l)
print('resolved:', EXTRA)

# --- screen windows by the MIN_SIDE rule ---
USE = []
for name, (fn, l) in EXTRA.items():
    X0, y0 = load_block(fn, l)
    cut = int(len(y0) * 0.8)
    a_tr, a_te = int(y0[:cut].sum()), int(y0[cut:].sum())
    ok = (a_tr >= MIN_SIDE) and (a_te >= MIN_SIDE)
    print('%-14s block rows %7d attack %6d | chron train/test attack %6d/%6d -> %s' % (name, len(y0), int(y0.sum()), a_tr, a_te, 'USE' if ok else 'skip'))
    if ok: USE.append(name)
    del X0, y0; gc.collect()
print('USE:', USE)

done2 = set()
if os.path.exists(RUNS2):
    r0 = pd.read_csv(RUNS2)
    done2 = set(zip(r0.variant, r0.window, r0.seed, r0.split, r0.model))

for cls in USE:
    fname, lab = EXTRA[cls]
    X0, y0 = load_block(fname, lab)
    h0 = pd.util.hash_pandas_object(X0, index=False).values
    print(); print('WINDOW', cls, '| block rows', len(X0), '| attack rows', int(y0.sum()))
    for var in VARIANTS:
        if var == 'dedup':
            keep = ~pd.DataFrame({'h': h0, 'y': y0}).duplicated().values
            X, y, h = X0[keep].reset_index(drop=True), y0[keep], h0[keep]
        else:
            X, y, h = X0, y0, h0
        n = len(y); cut = int(n * 0.8)
        a_tr, a_te = int(y[:cut].sum()), int(y[cut:].sum())
        print('  [' + var + '] rows', n, '| chron train/test attack', a_tr, a_te)
        if a_tr < MIN_SIDE or a_te < MIN_SIDE:
            print('  skipped after dedup (fewer than MIN_SIDE attack flows on one side)'); continue
        chron_cache = None
        for seed in SEEDS:
            todo = [(sp, m) for sp in ['chron', 'random'] for m in MNAMES if (var, cls, seed, sp, m) not in done2]
            if not todo: continue
            r_tr, r_te = train_test_split(np.arange(n), test_size=0.2, random_state=seed, stratify=y)
            splits = {'chron': (np.arange(cut), np.arange(cut, n)), 'random': (r_tr, r_te)}
            new = []
            for sp, (itr, ite) in splits.items():
                if not any(s == sp for s, _ in todo): continue
                if sp == 'chron' and chron_cache is not None:
                    A = chron_cache
                else:
                    A = prep(X.iloc[itr], y[itr], X.iloc[ite], y[ite])
                    if sp == 'chron': chron_cache = A
                Xtr_s, ytr_s, Xte_s, yte_s = A
                trh = set(h[itr]); dup = float(np.mean([hh in trh for hh in h[ite]]))
                models = make_models(seed)
                for mn in MNAMES:
                    if (var, cls, seed, sp, mn) in done2: continue
                    models[mn].fit(Xtr_s, ytr_s)
                    f1m, rec, fpr, pr = ev(models[mn], Xte_s, yte_s)
                    new.append(dict(variant=var, window=cls, seed=seed, split=sp, model=mn, f1_macro=round(f1m, 4),
                                    attack_recall=round(rec, 4), FPR=round(fpr, 4), PR_AUC=round(pr, 4),
                                    n_train=len(ytr_s), n_test=len(yte_s), att_train=int(ytr_s.sum()), att_test=int(yte_s.sum()),
                                    dup_share=round(dup, 4)))
                    done2.add((var, cls, seed, sp, mn))
                gc.collect()
            if new:
                pd.DataFrame(new).to_csv(RUNS2, mode='a', header=not os.path.exists(RUNS2), index=False)
            print('   ', var, cls, 'seed', seed, 'saved', len(new))
    del X0, y0, h0; gc.collect()

# --- combined summary: five main windows (CELL 37 file) + extras (this file) ---
R = pd.concat([pd.read_csv(RUNS), pd.read_csv(RUNS2)], ignore_index=True) if os.path.exists(RUNS2) else pd.read_csv(RUNS)
WALL = RUN_WINDOWS + [w for w in USE if w in set(R.window)]
def ci(v):
    v = np.asarray(v, float); k = len(v)
    if k < 2: return (float('nan'),) * 3
    m = v.mean(); se = v.std(ddof=1) / np.sqrt(k); t = stats.t.ppf(0.975, k - 1)
    return m, m - t * se, m + t * se
print(); print('=' * 70); print('SUMMARY (all windows): random minus chronological F1-macro (pp)')
for var in VARIANTS:
    print(); print('VARIANT:', var)
    wm = []; allc = []
    for cls in WALL:
        per = []
        for mn in MNAMES:
            g = R[(R.variant == var) & (R.window == cls) & (R.model == mn)]
            a = g[g.split == 'random'].set_index('seed')['f1_macro']; b = g[g.split == 'chron'].set_index('seed')['f1_macro']
            s = a.index.intersection(b.index)
            if len(s) < 2: continue
            d = (a.loc[s] - b.loc[s]) * 100; per.append(d); allc.append(d)
            m, lo, hi = ci(d)
            print('  %-13s %-13s dF1=%+6.2f [%+6.2f, %+6.2f] | att tr/te=%d/%d | dup(chron)=%.3f' % (
                cls, mn, m, lo, hi, g[g.split == 'chron'].att_train.iloc[0], g[g.split == 'chron'].att_test.iloc[0], g[g.split == 'chron'].dup_share.mean()))
        if per:
            wm.append(pd.concat(per, axis=1).mean(axis=1).mean()); print('  window mean', cls, '%+.2f' % wm[-1])
    m, lo, hi = ci(pd.concat(allc, axis=1).mean(axis=1))
    print('  POOLED over all cells: %+.2f [%+.2f, %+.2f]' % (m, lo, hi))
    m, lo, hi = ci(wm)
    print('  WINDOW-LEVEL (n=%d, t df=%d): %+.2f [%+.2f, %+.2f]' % (len(wm), len(wm) - 1, m, lo, hi))
    print('  windows with positive mean: %d of %d' % (sum(1 for v in wm if v > 0), len(wm)))
print(); print('Saved ->', RUNS2)


Wednesday.csv {'BENIGN': np.int64(440031), 'DoS Hulk': np.int64(231073), 'DoS GoldenEye': np.int64(10293), 'DoS slowloris': np.int64(5796), 'DoS Slowhttptest': np.int64(5499), 'Heartbleed': np.int64(11)}
Thursday.csv {'BENIGN': np.int64(288566), 'Infiltration': np.int64(36)}
Thursday+1.csv {'BENIGN': np.int64(168186), 'Web Attack � Brute Force': np.int64(1507), 'Web Attack � XSS': np.int64(652), 'Web Attack � Sql Injection': np.int64(21)}
resolved: {'GoldenEye': ('Wednesday.csv', 'DoS GoldenEye'), 'Slowloris': ('Wednesday.csv', 'DoS slowloris'), 'Slowhttptest': ('Wednesday.csv', 'DoS Slowhttptest'), 'WebBruteForce': ('Thursday+1.csv', 'Web Attack � Brute Force'), 'WebXSS': ('Thursday+1.csv', 'Web Attack � XSS')}
GoldenEye      block rows  361653 attack  10293 | chron train/test attack   6559/  3734 -> USE
Slowloris      block rows   62716 attack   5796 | chron train/test attack   4063/  1733 -> USE
Slowhttptest   block rows    5585 attack   5499 | chron train/test attack   4391/  1108 

In [ ]:
# ===== CELL 39: Bot window diagnostics + forward-chaining cut points (original pipeline; CELL 37 functions must be in memory) =====
import numpy as np, pandas as pd, gc
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import precision_score

MIN_SIDE = 100
X0, y0 = load_block('Friday.csv', 'Bot')
h0 = pd.util.hash_pandas_object(X0, index=False).values
BOT_OUT = OUT + '/cic2017_bot_diagnostics_runs.csv'
rows_out = []
for var in ['captured', 'dedup']:
    if var == 'dedup':
        keep = ~pd.DataFrame({'h': h0, 'y': y0}).duplicated().values
        X, y = X0[keep].reset_index(drop=True), y0[keep]
    else:
        X, y = X0, y0
    n = len(y)
    print(); print('=' * 70); print('BOT window,', var, '| rows', n, '| attack', int(y.sum()))
    dec = np.array_split(np.arange(n), 10)
    print('attack flows per decile of the block:', [int(y[d].sum()) for d in dec])
    cut = int(n * 0.8)
    print('attack rate train %.4f%% | test %.4f%%' % (100 * y[:cut].mean(), 100 * y[cut:].mean()))
    r_tr, r_te = train_test_split(np.arange(n), test_size=0.2, random_state=42, stratify=y)
    for sp, (itr, ite) in {'chron': (np.arange(cut), np.arange(cut, n)), 'random': (r_tr, r_te)}.items():
        Xtr_s, ytr_s, Xte_s, yte_s = prep(X.iloc[itr], y[itr], X.iloc[ite], y[ite])
        nn = NearestNeighbors(n_neighbors=1).fit(Xtr_s[ytr_s == 1])
        dist = nn.kneighbors(Xte_s[yte_s == 1])[0][:, 0]
        nnb = NearestNeighbors(n_neighbors=1).fit(Xtr_s[ytr_s == 0])
        distb = nnb.kneighbors(Xte_s[yte_s == 1])[0][:, 0]
        print('  %-6s test-attack -> nearest train ATTACK: median %.3f p90 %.3f | -> nearest train BENIGN: median %.3f | attack n_tr/n_te %d/%d' % (
            sp, np.median(dist), np.percentile(dist, 90), np.median(distb), int(ytr_s.sum()), int(yte_s.sum())))
        models = make_models(42)
        for mn in MNAMES:
            models[mn].fit(Xtr_s, ytr_s)
            p = models[mn].predict(Xte_s)
            prec = precision_score(yte_s, p, zero_division=0)
            f1m, rec, fpr, pr = ev(models[mn], Xte_s, yte_s)
            print('     %-12s F1m %.4f | attack precision %.3f recall %.3f | FPR %.4f | benign test n %d, FP %d' % (
                mn, f1m, prec, rec, fpr, int((yte_s == 0).sum()), int(round(fpr * (yte_s == 0).sum()))))
        del Xtr_s, Xte_s; gc.collect()
    print('  forward-chaining (train first f, test next 20 percent slice), mean over seeds 42-44; random 80/20 on whole block, seeds 42-44')
    for f in [0.5, 0.6, 0.7, 0.8, 0.9]:
        c0 = int(n * f); c1 = min(n, c0 + int(n * 0.2))
        itr, ite = np.arange(c0), np.arange(c0, c1)
        a_tr, a_te = int(y[itr].sum()), int(y[ite].sum())
        if a_tr < MIN_SIDE or a_te < MIN_SIDE:
            print('   f=%.1f train attack %d, test attack %d -> skip (fewer than %d)' % (f, a_tr, a_te, MIN_SIDE)); continue
        Xtr_s, ytr_s, Xte_s, yte_s = prep(X.iloc[itr], y[itr], X.iloc[ite], y[ite])
        res = {mn: [] for mn in MNAMES}
        for seed in [42, 43, 44]:
            models = make_models(seed)
            for mn in MNAMES:
                models[mn].fit(Xtr_s, ytr_s)
                res[mn].append(ev(models[mn], Xte_s, yte_s)[0])
        for mn in MNAMES:
            rows_out.append(dict(variant=var, f=f, model=mn, f1_macro=float(np.mean(res[mn])), att_train=a_tr, att_test=a_te))
        print('   f=%.1f attack tr/te %4d/%4d | F1m RF %.4f LGBM %.4f XGB %.4f' % (f, a_tr, a_te, np.mean(res['RandomForest']), np.mean(res['LightGBM']), np.mean(res['XGBoost'])))
        del Xtr_s, Xte_s; gc.collect()
    rnd = {mn: [] for mn in MNAMES}
    for seed in [42, 43, 44]:
        r_tr, r_te = train_test_split(np.arange(n), test_size=0.2, random_state=seed, stratify=y)
        Xtr_s, ytr_s, Xte_s, yte_s = prep(X.iloc[r_tr], y[r_tr], X.iloc[r_te], y[r_te])
        models = make_models(seed)
        for mn in MNAMES:
            models[mn].fit(Xtr_s, ytr_s); rnd[mn].append(ev(models[mn], Xte_s, yte_s)[0])
    print('   random 80/20      | F1m RF %.4f LGBM %.4f XGB %.4f' % (np.mean(rnd['RandomForest']), np.mean(rnd['LightGBM']), np.mean(rnd['XGBoost'])))
pd.DataFrame(rows_out).to_csv(BOT_OUT, index=False)
print(); print('Saved ->', BOT_OUT)



BOT window, captured | rows 166951 | attack 1966
attack flows per decile of the block: [2, 10, 102, 148, 477, 735, 78, 64, 91, 259]
attack rate train 1.2099% | test 1.0482%
  chron  test-attack -> nearest train ATTACK: median 0.000 p90 0.000 | -> nearest train BENIGN: median 0.016 | attack n_tr/n_te 1606/350
     RandomForest F1m 0.7713 | attack precision 0.381 recall 0.997 | FPR 0.0172 | benign test n 33026, FP 567
     LightGBM     F1m 0.9964 | attack precision 0.989 recall 0.997 | FPR 0.0001 | benign test n 33026, FP 4
     XGBoost      F1m 0.9964 | attack precision 0.986 recall 1.000 | FPR 0.0002 | benign test n 33026, FP 5
  random test-attack -> nearest train ATTACK: median 0.000 p90 0.002 | -> nearest train BENIGN: median 0.016 | attack n_tr/n_te 1565/391
     RandomForest F1m 0.7728 | attack precision 0.385 recall 0.997 | FPR 0.0189 | benign test n 32974, FP 624
     LightGBM     F1m 0.9910 | attack precision 0.977 recall 0.987 | FPR 0.0003 | benign test n 32974, FP 9
     XGB

In [ ]:
# ===== CELL 41 (self-contained) =====
import os
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'): drive.mount('/content/drive')
import pandas as pd, numpy as np, os, time, gc, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from scipy import stats
CIC = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017/preprocessed/all_days_merged.csv'
OUT = '/content/drive/MyDrive/UNSW-NB15'
if 'cic' not in globals():
    cic = pd.read_csv(CIC); cic.columns = [c.strip() for c in cic.columns]
    cic['day'] = cic['day'].str.lower(); cic['Label'] = cic['Label'].str.strip()
    for c in cic.columns:
        if cic[c].dtype == 'float64': cic[c] = cic[c].astype('float32')
FEATS = [c for c in cic.columns if c not in ('Label', 'day', 'Destination Port')]
TRAIN_DAYS = ['monday', 'tuesday', 'wednesday', 'thursday']
print('rows', len(cic), 'features before filters', len(FEATS), '| days', cic['day'].value_counts().to_dict())
def prop_sample(df, n, seed):
    frac = n / len(df); parts = []
    for _, g in df.groupby('Label'):
        k = int(frac * len(g))
        if k > 0: parts.append(g.sample(n=k, random_state=seed))
    return pd.concat(parts)
def make_partitions(proto, seed):
    if proto == 'temporal':
        return prop_sample(cic[cic['day'].isin(TRAIN_DAYS)], 199_999, seed), prop_sample(cic[cic['day'] == 'friday'], 99_999, seed)
    pooled = prop_sample(cic, 199_999, seed)
    return train_test_split(pooled, test_size=0.2, stratify=(pooled['Label'] != 'BENIGN'), random_state=seed)
def prepare(tr, te, smote=True, seed=42):
    Xtr, Xte = tr[FEATS].astype('float64'), te[FEATS].astype('float64')
    vt = VarianceThreshold(0.01).fit(Xtr); cols = Xtr.columns[vt.get_support()]
    Xtr, Xte = Xtr[cols], Xte[cols]
    cm = Xtr.corr().abs(); up = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool))
    drop = [c for c in up.columns if (up[c] > 0.95).any()]; Xtr, Xte = Xtr.drop(columns=drop), Xte.drop(columns=drop)
    sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
    ytr, yte = (tr['Label'] != 'BENIGN').astype(int).values, (te['Label'] != 'BENIGN').astype(int).values
    n_syn = 0
    if smote:
        n_min, n_maj = int(ytr.sum()), int((ytr == 0).sum()); target = max(n_min, min(30000, n_maj)); k = max(1, min(5, n_min - 1))
        if target > n_min:
            Ztr, ytr = SMOTE(sampling_strategy={1: target}, k_neighbors=k, random_state=seed).fit_resample(Ztr, ytr); n_syn = target - n_min
    return Ztr, ytr, Zte, yte, Xtr.shape[1], n_syn
def metrics(yte, p): return dict(f1_macro=f1_score(yte, p, average='macro'), recall=recall_score(yte, p), fpr=((p == 1) & (yte == 0)).sum() / max((yte == 0).sum(), 1))
def base_models(seed):
    return {'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed),
            'LightGBM': LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1),
            'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, eval_metric='logloss', verbosity=0)}

# ===== Table 6 rerun with exact duplicates removed BEFORE sampling/splitting =====
RES = OUT + '/cic2017_table6_dedup_runs.csv'
SEEDS = list(range(42, 52))
h = pd.util.hash_pandas_object(cic[FEATS], index=False).values
dup = pd.DataFrame({'h': h, 'y': cic['Label'].values}).duplicated().values
print('rows before %d | exact duplicates removed %d (%.1f%%)' % (len(cic), dup.sum(), 100 * dup.mean()))
print(pd.DataFrame({'day': cic['day'].values, 'dup': dup}).groupby('day')['dup'].agg(['size', 'sum', 'mean']).round(3))
cic_full = cic
cic = cic_full[~dup].reset_index(drop=True)
fri = cic[cic['day'] == 'friday']; tr_pool = cic[cic['day'].isin(TRAIN_DAYS)]
rho = fri['Label'].isin(sorted(set(fri['Label']) - set(tr_pool['Label']))).mean()
print('after removal: train-days rows %d, Friday rows %d, Friday attack share %.3f | rho %.4f -> ceiling %.4f' % (len(tr_pool), len(fri), (fri['Label'] != 'BENIGN').mean(), rho, 1 - rho))
print('Friday class counts:', fri['Label'].value_counts().to_dict())
done = set()
if os.path.exists(RES):
    r0 = pd.read_csv(RES); done = set(zip(r0.seed, r0.proto, r0.model))
print('resuming, done:', len(done))
t0 = time.time()
try:
    for seed in SEEDS:
        for proto in ['random', 'temporal']:
            todo = [m for m in ['RandomForest', 'LightGBM', 'XGBoost'] if (seed, proto, m) not in done]
            if not todo: continue
            tr, te = make_partitions(proto, seed)
            Ztr, ytr, Zte, yte, nf, ns = prepare(tr, te, True, seed)
            new = []
            for name, m in base_models(seed).items():
                if name not in todo: continue
                m.fit(Ztr, ytr); p = m.predict(Zte); r = metrics(yte, p)
                new.append(dict(seed=seed, proto=proto, model=name, f1_macro=round(r['f1_macro'], 4), recall=round(r['recall'], 4), fpr=round(r['fpr'], 4), n_train=len(tr), n_test=len(te), att_test=int(yte.sum()), n_feat=nf, smote_added=ns))
                done.add((seed, proto, name))
            pd.DataFrame(new).to_csv(RES, mode='a', header=not os.path.exists(RES), index=False)
            print('seed', seed, proto, 'feat', nf, 'smote', ns, '| elapsed %.0fs' % (time.time() - t0))
            del Ztr, Zte; gc.collect()
finally:
    cic = cic_full
R = pd.read_csv(RES)
def ci(v):
    v = np.asarray(v, float); k = len(v); m = v.mean(); se = v.std(ddof=1) / np.sqrt(k); t = stats.t.ppf(0.975, k - 1)
    return m, m - t * se, m + t * se
print(); print('TABLE 6 AFTER EXACT-DUPLICATE REMOVAL: random minus temporal F1-macro (pp), 95% t-CI (df 9)')
for mn in ['RandomForest', 'LightGBM', 'XGBoost']:
    g = R[R.model == mn]; a = g[g.proto == 'random'].set_index('seed').f1_macro; b = g[g.proto == 'temporal'].set_index('seed').f1_macro
    s = a.index.intersection(b.index)
    if len(s) < 2: continue
    d = (a.loc[s] - b.loc[s]) * 100; m, lo, hi = ci(d)
    print('  %-12s gap %6.2f [%6.2f, %6.2f] | random F1 %.4f (sd %.4f) | temporal F1 %.4f (sd %.4f) | seeds %d' % (mn, m, lo, hi, a.loc[s].mean(), a.loc[s].std(ddof=1), b.loc[s].mean(), b.loc[s].std(ddof=1), len(s)))
print('Saved ->', RES)


rows 2827876 features before filters 77 | days {'friday': 702718, 'wednesday': 691406, 'monday': 529481, 'thursday': 458626, 'tuesday': 445645}
rows before 2827876 | exact duplicates removed 628663 (22.2%)
             size     sum   mean
day                             
friday     702718  229253  0.326
monday     529481   70650  0.133
thursday   458626  130830  0.285
tuesday    445645   67035  0.150
wednesday  691406  130895  0.189
after removal: train-days rows 1725748, Friday rows 473465, Friday attack share 0.278 | rho 0.2775 -> ceiling 0.7225
Friday class counts: {'BENIGN': 342058, 'DDoS': 128014, 'PortScan': 1956, 'Bot': 1437}
resuming, done: 21
seed 45 temporal feat 44 smote 6236 | elapsed 95s
seed 46 random feat 43 smote 5526 | elapsed 141s
seed 46 temporal feat 43 smote 6236 | elapsed 202s
seed 47 random feat 43 smote 5526 | elapsed 248s
seed 47 temporal feat 42 smote 6236 | elapsed 302s
seed 48 random feat 42 smote 5526 | elapsed 350s
seed 48 temporal feat 42 smote 6236 | ela

In [ ]:
# ===== CELL 42: Table 20 (multiclass ceiling) rerun with exact duplicates removed first. Needs CELL 41 run in this session. New file only. =====
import os, time, numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder
for nm in ['cic', 'FEATS', 'make_partitions', 'VarianceThreshold', 'StandardScaler', 'OUT', 'TRAIN_DAYS']: assert nm in globals(), 'run CELL 41 first (missing ' + nm + ')'
RESX = os.path.join(OUT, 'cic_multiclass_temporal_dedup_results.csv'); MX_SEEDS = list(range(42, 52))
def build_mc(model, seed):
    if model == 'RandomForest': return RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed)
    if model == 'LightGBM': return LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1)
    return XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, eval_metric='mlogloss', verbosity=0)
h = pd.util.hash_pandas_object(cic[FEATS], index=False).values
dup = pd.DataFrame({'h': h, 'y': cic['Label'].values}).duplicated().values
cic_full = cic; cic = cic_full[~dup].reset_index(drop=True)
print('duplicates removed', int(dup.sum()), '| Friday classes:', cic[cic.day == 'friday'].Label.value_counts().to_dict())
res = pd.read_csv(RESX) if os.path.exists(RESX) else pd.DataFrame()
done = set(zip(res.seed, res.model)) if len(res) else set()
try:
    for seed in MX_SEEDS:
        t0 = time.time(); tr, te = make_partitions('temporal', seed)
        Xtr, Xte = tr[FEATS].astype('float64'), te[FEATS].astype('float64')
        vt = VarianceThreshold(0.01).fit(Xtr); cols = Xtr.columns[vt.get_support()]; Xtr, Xte = Xtr[cols], Xte[cols]
        cm_ = Xtr.corr().abs(); up = cm_.where(np.triu(np.ones(cm_.shape), k=1).astype(bool)); drop = [c for c in up.columns if (up[c] > 0.95).any()]
        Xtr, Xte = Xtr.drop(columns=drop), Xte.drop(columns=drop); sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
        le = LabelEncoder().fit(tr['Label']); ytr = le.transform(tr['Label']); known = te['Label'].isin(le.classes_).values
        rho = 1 - known.mean(); yte = np.where(known, le.transform(te['Label'].where(known, le.classes_[0])), -1)
        for name in ['RandomForest', 'LightGBM', 'XGBoost']:
            if (seed, name) in done: continue
            m = build_mc(name, seed).fit(Ztr, ytr); p = m.predict(Zte); acc = float((p == yte).mean())
            r = dict(seed=seed, model=name, rho=rho, accuracy=acc, ceiling=1 - rho, slack=1 - rho - acc, violated=bool(acc > 1 - rho + 1e-12), n_classes_train=len(le.classes_))
            res = pd.concat([res, pd.DataFrame([r])], ignore_index=True); res.to_csv(RESX, index=False)
        print('seed', seed, 'done in %.0fs' % (time.time() - t0), '| rows', len(res))
finally:
    cic = cic_full
print('\n=== Multiclass temporal evaluation AFTER duplicate removal (train Mon-Thu classes, test Friday), 10 seeds ===')
print(res.groupby('model').agg(rho=('rho', 'mean'), accuracy=('accuracy', 'mean'), ceiling=('ceiling', 'mean'), slack=('slack', 'mean'), violations=('violated', 'sum'), runs=('seed', 'count')).round(4).to_string())
print('Saved ->', RESX)


duplicates removed 628663 | Friday classes: {'BENIGN': 342058, 'DDoS': 128014, 'PortScan': 1956, 'Bot': 1437}
seed 42 done in 159s | rows 3
seed 43 done in 115s | rows 6
seed 44 done in 119s | rows 9
seed 45 done in 122s | rows 12
seed 46 done in 117s | rows 15
seed 47 done in 115s | rows 18
seed 48 done in 118s | rows 21
seed 49 done in 129s | rows 24
seed 50 done in 125s | rows 27
seed 51 done in 122s | rows 30

=== Multiclass temporal evaluation AFTER duplicate removal (train Mon-Thu classes, test Friday), 10 seeds ===
                 rho  accuracy  ceiling   slack  violations  runs
model                                                            
LightGBM      0.2775    0.7222   0.7225  0.0002           0    10
RandomForest  0.2775    0.7155   0.7225  0.0070           0    10
XGBoost       0.2775    0.7210   0.7225  0.0014           0    10
Saved -> /content/drive/MyDrive/UNSW-NB15/cic_multiclass_temporal_dedup_results.csv


In [ ]:
# ===== CELL 43: benign-only detectors (Isolation Forest, autoencoder), random vs temporal, captured and deduplicated =====
# Needs CELL 41 run in this session. Trained on BENIGN flows of the training partition only (no attack labels, no SMOTE).
# Alert threshold fixed on training benign scores only (99th percentile, 1% training FPR); no test labels used for it. Resumable. New file only.
import os, gc, time, numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import f1_score, roc_auc_score, recall_score
from scipy import stats
for nm in ['cic', 'FEATS', 'make_partitions', 'prepare', 'OUT']: assert nm in globals(), 'run CELL 41 first (missing ' + nm + ')'
RESB = OUT + '/cic2017_benign_only_runs.csv'; SEEDSB = list(range(42, 52))
res = pd.read_csv(RESB) if os.path.exists(RESB) else pd.DataFrame()
doneb = set(zip(res.variant, res.seed, res.proto, res.model)) if len(res) else set()
h = pd.util.hash_pandas_object(cic[FEATS], index=False).values
dupm = pd.DataFrame({'h': h, 'y': cic['Label'].values}).duplicated().values
cic_full = cic
def score_ae(Ztr_b, Zte, seed):
    ae = MLPRegressor(hidden_layer_sizes=(32, 16, 32), activation='relu', max_iter=30, batch_size=512, random_state=seed, early_stopping=False)
    ae.fit(Ztr_b, Ztr_b)
    f = lambda Z: ((ae.predict(Z) - Z) ** 2).mean(axis=1)
    return f(Ztr_b), f(Zte)
def score_if(Ztr_b, Zte, seed):
    m = IsolationForest(n_estimators=100, random_state=seed, n_jobs=-1).fit(Ztr_b)
    return -m.score_samples(Ztr_b), -m.score_samples(Zte)
t0 = time.time()
try:
    for var in ['captured', 'dedup']:
        cic = cic_full if var == 'captured' else cic_full[~dupm].reset_index(drop=True)
        for seed in SEEDSB:
            for proto in ['random', 'temporal']:
                todo = [m for m in ['IsolationForest', 'Autoencoder'] if (var, seed, proto, m) not in doneb]
                if not todo: continue
                tr, te = make_partitions(proto, seed)
                Ztr, ytr, Zte, yte, nf, _ = prepare(tr, te, False, seed)
                Zb = Ztr[ytr == 0]; new = []
                for name in todo:
                    s_tr, s_te = (score_if if name == 'IsolationForest' else score_ae)(Zb, Zte, seed)
                    thr = np.percentile(s_tr, 99); p = (s_te > thr).astype(int)
                    new.append(dict(variant=var, seed=seed, proto=proto, model=name, f1_macro=round(f1_score(yte, p, average='macro'), 4),
                                    recall=round(recall_score(yte, p), 4), fpr=round(((p == 1) & (yte == 0)).sum() / max((yte == 0).sum(), 1), 4),
                                    roc_auc=round(roc_auc_score(yte, s_te), 4), n_benign_train=len(Zb), n_test=len(yte), att_test=int(yte.sum()), n_feat=nf))
                    doneb.add((var, seed, proto, name))
                pd.DataFrame(new).to_csv(RESB, mode='a', header=not os.path.exists(RESB), index=False)
                print(var, 'seed', seed, proto, '| elapsed %.0fs' % (time.time() - t0)); del Ztr, Zte, Zb; gc.collect()
finally:
    cic = cic_full
R = pd.read_csv(RESB)
def ci(v):
    v = np.asarray(v, float); k = len(v); m = v.mean(); se = v.std(ddof=1) / np.sqrt(k); t = stats.t.ppf(0.975, k - 1)
    return m, m - t * se, m + t * se
print(); print('BENIGN-ONLY DETECTORS: random minus temporal (pp), 95% t-CI; ROC-AUC and recall by protocol')
for var in ['captured', 'dedup']:
    for mn in ['IsolationForest', 'Autoencoder']:
        g = R[(R.variant == var) & (R.model == mn)]
        a = g[g.proto == 'random'].set_index('seed'); b = g[g.proto == 'temporal'].set_index('seed'); s = a.index.intersection(b.index)
        if len(s) < 2: continue
        d = (a.f1_macro.loc[s] - b.f1_macro.loc[s]) * 100; m, lo, hi = ci(d)
        print('  %-9s %-16s F1 rnd %.4f tmp %.4f | gap %+6.2f [%+6.2f, %+6.2f] | AUC rnd %.3f tmp %.3f | recall rnd %.3f tmp %.3f | FPR rnd %.3f tmp %.3f | n %d' % (
            var, mn, a.f1_macro.loc[s].mean(), b.f1_macro.loc[s].mean(), m, lo, hi, a.roc_auc.loc[s].mean(), b.roc_auc.loc[s].mean(),
            a.recall.loc[s].mean(), b.recall.loc[s].mean(), a.fpr.loc[s].mean(), b.fpr.loc[s].mean(), len(s)))
print('Saved ->', RESB)


captured seed 42 random | elapsed 61s
captured seed 42 temporal | elapsed 130s
captured seed 43 random | elapsed 151s
captured seed 43 temporal | elapsed 180s
captured seed 44 random | elapsed 201s
captured seed 44 temporal | elapsed 231s
captured seed 45 random | elapsed 253s
captured seed 45 temporal | elapsed 282s
captured seed 46 random | elapsed 302s
captured seed 46 temporal | elapsed 331s
captured seed 47 random | elapsed 351s
captured seed 47 temporal | elapsed 377s
captured seed 48 random | elapsed 401s
captured seed 48 temporal | elapsed 430s
captured seed 49 random | elapsed 452s
captured seed 49 temporal | elapsed 480s
captured seed 50 random | elapsed 503s
captured seed 50 temporal | elapsed 532s
captured seed 51 random | elapsed 552s
captured seed 51 temporal | elapsed 579s
dedup seed 42 random | elapsed 600s
dedup seed 42 temporal | elapsed 627s
dedup seed 43 random | elapsed 650s
dedup seed 43 temporal | elapsed 678s
dedup seed 44 random | elapsed 702s
dedup seed 44 tem

In [ ]:
# ===== CELL 44b (one seed, as captured): forward-chaining cut sweep on all ten windows (original pipeline, self-contained), captured and deduplicated =====
import os, gc, warnings, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, confusion_matrix, average_precision_score
import lightgbm as lgb, xgboost as xgb
from scipy import stats
warnings.filterwarnings('ignore')

D    = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017'
OUT  = '/content/drive/MyDrive/UNSW-NB15'
SEEDS = list(range(42, 52))
EXCLUDE = ['Destination Port', 'Source Port']
WINDOWS = {
    'SSH-Patator': ('Tuesday.csv',   'SSH-Patator'),
    'DoS Hulk':    ('Wednesday.csv', 'DoS Hulk'),
    'Bot':         ('Friday.csv',    'Bot'),
    'PortScan':    ('Friday+1.csv',  'PortScan'),
    'DDoS':        ('Friday+2.csv',  'DDoS'),
}
MNAMES = ['RandomForest', 'LightGBM', 'XGBoost']

def make_models(rs):
    return {
        'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=rs, n_jobs=-1),
        'LightGBM': lgb.LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                                       class_weight='balanced', random_state=rs, n_jobs=-1, verbose=-1),
        'XGBoost': xgb.XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                                     eval_metric='logloss', random_state=rs, n_jobs=-1, verbosity=0),
    }

def load_block(fname, cls):
    df = pd.read_csv(D + '/' + fname, low_memory=False)
    df.columns = df.columns.str.strip()
    df = df[df['Label'] != 'Label'].reset_index(drop=True)
    pos = np.where(df['Label'].values == cls)[0]
    w = df.iloc[pos[0]:pos[-1] + 1]
    w = w[w['Label'].isin(['BENIGN', cls])].reset_index(drop=True)
    y = (w['Label'].astype(str).str.strip() != 'BENIGN').astype(int).values
    X = w.drop(columns=['Label'], errors='ignore').drop(columns=EXCLUDE, errors='ignore')
    X = X.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    return X.reset_index(drop=True), y

def prep(Xtr, ytr, Xte, yte):
    mtr, mte = Xtr.notna().all(axis=1).values, Xte.notna().all(axis=1).values
    Xtr, ytr, Xte, yte = Xtr[mtr], ytr[mtr], Xte[mte], yte[mte]
    sc = StandardScaler()
    return sc.fit_transform(Xtr.values), ytr, sc.transform(Xte.values), yte, Xtr, Xte

def ev(m, Xte, yte):
    p = m.predict(Xte)
    f1m = f1_score(yte, p, average='macro', zero_division=0)
    am = (yte == 1)
    rec = (p[am] == 1).mean() if am.any() else float('nan')
    cm = confusion_matrix(yte, p, labels=[0, 1]); TN, FP = cm[0, 0], cm[0, 1]
    fpr = FP / (FP + TN) if (FP + TN) else 0.0
    return f1m, rec, fpr, float('nan')
MIN_SIDE = 100
KEYS = {'GoldenEye': 'goldeneye', 'Slowloris': 'slowloris', 'Slowhttptest': 'slowhttp',
        'WebBruteForce': 'brute force', 'WebXSS': 'xss'}
FILES = ['Wednesday.csv', 'Thursday.csv', 'Thursday+1.csv']
EXTRA = {}
for fn in FILES:
    p = D + '/' + fn
    if not os.path.exists(p):
        print('missing file', fn); continue
    lab = pd.read_csv(p, usecols=lambda c: c.strip() == 'Label', low_memory=False)
    lab.columns = ['Label']
    vc = lab['Label'].value_counts()
    for name, key in KEYS.items():
        for l in vc.index:
            if key in str(l).lower() and name not in EXTRA:
                EXTRA[name] = (fn, l)
print('resolved:', EXTRA)

SWEEP = OUT + '/cic2017_cut_sweep_1seed_runs.csv'
ALLW = dict(WINDOWS); ALLW.update(EXTRA)
FS = [0.5, 0.6, 0.7, 0.8, 0.9]; SEEDS_S = [42]
rows_done = pd.read_csv(SWEEP) if os.path.exists(SWEEP) else pd.DataFrame()
dn = set(zip(rows_done.variant, rows_done.window, rows_done.f.astype(str), rows_done.model)) if len(rows_done) else set()
for cls, (fname, lab) in ALLW.items():
    X0, y0 = load_block(fname, lab); h0 = pd.util.hash_pandas_object(X0, index=False).values
    for var in ['captured']:
        if var == 'dedup':
            keep = ~pd.DataFrame({'h': h0, 'y': y0}).duplicated().values; X, y = X0[keep].reset_index(drop=True), y0[keep]
        else: X, y = X0, y0
        n = len(y); out = []
        for f in FS + ['random']:
            if all((var, cls, str(f), m) in dn for m in MNAMES): continue
            if f == 'random':
                parts = [train_test_split(np.arange(n), test_size=0.2, random_state=s, stratify=y) for s in SEEDS_S]
                a_tr, a_te = int(y[parts[0][0]].sum()), int(y[parts[0][1]].sum())
            else:
                c0 = int(n * f); c1 = min(n, c0 + int(n * 0.2)); itr, ite = np.arange(c0), np.arange(c0, c1)
                a_tr, a_te = int(y[itr].sum()), int(y[ite].sum())
                if a_tr < MIN_SIDE or a_te < MIN_SIDE:
                    print(var, cls, 'f', f, 'skip (attack tr/te %d/%d)' % (a_tr, a_te)); continue
                parts = [(itr, ite)] * len(SEEDS_S)
            res = {m: [] for m in MNAMES}; cache = None
            for s, (itr, ite) in zip(SEEDS_S, parts):
                if f != 'random' and cache is not None: A = cache
                else:
                    A = prep(X.iloc[itr], y[itr], X.iloc[ite], y[ite]); cache = A if f != 'random' else None
                models = make_models(s)
                for m in MNAMES:
                    models[m].fit(A[0], A[1]); res[m].append(ev(models[m], A[2], A[3])[0])
            for m in MNAMES:
                out.append(dict(variant=var, window=cls, f=str(f), model=m, f1_macro=float(np.mean(res[m])), att_train=a_tr, att_test=a_te)); dn.add((var, cls, str(f), m))
            print(var, cls, 'f', f, 'F1 RF %.4f LGBM %.4f XGB %.4f' % tuple(np.mean(res[m]) for m in MNAMES), '| att tr/te', a_tr, a_te); gc.collect()
        if out: pd.DataFrame(out).to_csv(SWEEP, mode='a', header=not os.path.exists(SWEEP), index=False)
    del X0, y0, h0; gc.collect()
R = pd.read_csv(SWEEP); R['f'] = R['f'].astype(str)
print(); print('CUT SWEEP SUMMARY: random-minus-forward-chaining F1-macro (pp), model mean, by window and cut')
for var in ['captured']:
    print('VARIANT', var)
    for cls in ALLW:
        g = R[(R.variant == var) & (R.window == cls)]
        if not len(g): continue
        rnd = g[g.f == 'random'].f1_macro.mean()
        line = []
        for f in FS:
            gg = g[g.f == str(f)]
            line.append('f%.1f:%+.2f' % (f, 100 * (rnd - gg.f1_macro.mean())) if len(gg) else 'f%.1f:skip' % f)
        print('  %-13s' % cls, ' '.join(line), '| random F1 %.4f' % rnd)
print('Saved ->', SWEEP)


resolved: {'GoldenEye': ('Wednesday.csv', 'DoS GoldenEye'), 'Slowloris': ('Wednesday.csv', 'DoS slowloris'), 'Slowhttptest': ('Wednesday.csv', 'DoS Slowhttptest'), 'WebBruteForce': ('Thursday+1.csv', 'Web Attack � Brute Force'), 'WebXSS': ('Thursday+1.csv', 'Web Attack � XSS')}
captured SSH-Patator f 0.5 F1 RF 0.9574 LGBM 0.9977 XGB 1.0000 | att tr/te 3428 1000
captured SSH-Patator f 0.6 F1 RF 0.9622 LGBM 0.9995 XGB 1.0000 | att tr/te 3937 981


KeyboardInterrupt: 

In [ ]:
# ===== E1: verify deduplication counts (no model training) =====
# CONFIG
import os
MERGED = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017/preprocessed/all_days_merged.csv'
RAWDIR = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017'
OUT = '/content/drive/MyDrive/UNSW-NB15'
OUTCSV = OUT + '/e1_dedup_counts.csv'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, gc, sklearn, scipy
print('pandas', pd.__version__, '| numpy', np.__version__, '| scipy', scipy.__version__, '| sklearn', sklearn.__version__)

# ---- load merged file as float64 (no float32 cast) ----
cdf = pd.read_csv(MERGED); cdf.columns = [c.strip() for c in cdf.columns]
cdf['day'] = cdf['day'].str.lower(); cdf['Label'] = cdf['Label'].str.strip()
print('merged rows', len(cdf), '| days', cdf['day'].value_counts().to_dict())
FE = [c for c in cdf.columns if c not in ('Label', 'day', 'Destination Port')]       # ports excluded (paper definition)
FE_P = FE + ['Destination Port']                                                      # port included (control)
lab = cdf['Label'].values; day = cdf['day'].values

def dupflag(cols, f32=False):
    X = cdf[cols]
    if f32: X = X.astype('float32')
    h = pd.util.hash_pandas_object(X, index=False).values
    return pd.DataFrame({'h': h, 'y': lab}).duplicated().values          # True = duplicate of an EARLIER row with the same label

V = {'A ports excl, float32 (as run)': dupflag(FE, True),
     'B ports excl, float64': dupflag(FE, False),
     'C ports INCL, float64': dupflag(FE_P, False)}
gc.collect()

rows = []
for vn, dup in V.items():
    keep = ~dup
    for scope, m in [('all', np.ones(len(cdf), bool)), ('friday', day == 'friday')]:
        before = pd.Series(lab[m]).value_counts(); after = pd.Series(lab[m & keep]).value_counts()
        for cl in before.index:
            rows.append(dict(variant=vn, scope=scope, label=cl, before=int(before[cl]), after=int(after.get(cl, 0))))
R = pd.DataFrame(rows); R['removed_pct'] = (100 * (1 - R.after / R.before)).round(1)
R.to_csv(OUTCSV, index=False)

print('\n## Total duplicates and per-day duplicate share')
print('| variant | removed | share | Mon | Tue | Wed | Thu | Fri |'); print('|---|---|---|---|---|---|---|---|')
for vn, dup in V.items():
    s = pd.Series(dup).groupby(day).mean()
    print('| %s | %d | %.1f%% | %s |' % (vn, dup.sum(), 100 * dup.mean(), ' | '.join('%.1f%%' % (100 * s[d]) for d in ['monday', 'tuesday', 'wednesday', 'thursday', 'friday'])))

for scope in ['all', 'friday']:
    print('\n## Per-class counts, scope =', scope, '(before -> after)')
    P = R[R.scope == scope].pivot(index='label', columns='variant', values='after')
    B = R[(R.scope == scope) & (R.variant == list(V)[0])].set_index('label')['before']
    T = pd.concat([B.rename('before'), P], axis=1).sort_values('before', ascending=False)
    print(T.to_markdown())
    print('TOTAL', T.sum().astype(int).to_dict())

print('\n## Friday rho (share of Friday flows from classes absent from Mon-Thu), per variant')
for vn, dup in V.items():
    k = ~dup; trc = set(lab[k & np.isin(day, ['monday', 'tuesday', 'wednesday', 'thursday'])])
    fr = pd.Series(lab[k & (day == 'friday')]); rho = 1 - fr.isin(trc).mean()
    print('%-34s Friday rows %d | rho %.4f | ceiling %.4f' % (vn, len(fr), rho, 1 - rho))

print('\n## Bot reconciliation')
bot_m = (lab == 'Bot'); print('Bot rows in merged file before dedup:', int(bot_m.sum()), '| days with Bot:', pd.Series(day[bot_m]).value_counts().to_dict())
for vn, dup in V.items(): print('  whole-file dedup, %-32s Bot kept: %d' % (vn, int((bot_m & ~dup).sum())))
def raw_block(fname, cls):
    df = pd.read_csv(RAWDIR + '/' + fname, low_memory=False); df.columns = df.columns.str.strip()
    df = df[df['Label'] != 'Label'].reset_index(drop=True)
    pos = np.where(df['Label'].values == cls)[0]; w = df.iloc[pos[0]:pos[-1] + 1]
    w = w[w['Label'].isin(['BENIGN', cls])].reset_index(drop=True)
    y = (w['Label'].astype(str).str.strip() != 'BENIGN').astype(int).values
    X = w.drop(columns=['Label'], errors='ignore').drop(columns=['Destination Port', 'Source Port'], errors='ignore')
    X = X.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    return X.reset_index(drop=True), y
X0, y0 = raw_block('Friday.csv', 'Bot'); bad = X0.isna().any(axis=1).values
print('raw Friday.csv Bot block: rows', len(y0), '| Bot flows', int(y0.sum()), '| Bot flows with NaN/inf', int((bad & (y0 == 1)).sum()))
for nm, X in [('float64', X0), ('float32', X0.astype('float32'))]:
    h = pd.util.hash_pandas_object(X, index=False).values
    d = pd.DataFrame({'h': h, 'y': y0}).duplicated().values
    print('  within-window dedup (%s, NaN rows still present): Bot kept %d | after then dropping NaN rows: %d' % (nm, int(((y0 == 1) & ~d).sum()), int(((y0 == 1) & ~d & ~bad).sum())))
    Xc, yc = X[~bad].reset_index(drop=True), y0[~bad]
    hc = pd.util.hash_pandas_object(Xc, index=False).values
    dc = pd.DataFrame({'h': hc, 'y': yc}).duplicated().values
    print('  NaN rows dropped FIRST, then dedup (%s): Bot kept %d' % (nm, int(((yc == 1) & ~dc).sum())))
bm = cdf.loc[bot_m, FE]; hb = pd.util.hash_pandas_object(bm.astype('float32'), index=False).values
print('  distinct Bot feature vectors in merged file (float32 / float64): %d / %d' % (len(set(hb)), len(set(pd.util.hash_pandas_object(bm, index=False).values))))
print('\nSaved ->', OUTCSV)


pandas 2.2.3 | numpy 2.1.3 | scipy 1.16.3 | sklearn 1.6.1


KeyboardInterrupt: 

In [ ]:
# ===== E1b: exact replication of the dedup run in CELL 41 (mixed dtypes: float64 columns cast to float32, integer columns left as is) + integer tables. Needs E1 run first. =====
import numpy as np, pandas as pd
for nm in ['cdf', 'FE', 'lab', 'day', 'V', 'R']: assert nm in globals(), 'run E1 first'
Xm = cdf[FE].copy()
for c in Xm.columns:
    if Xm[c].dtype == 'float64': Xm[c] = Xm[c].astype('float32')
print('dtypes after cast:', Xm.dtypes.astype(str).value_counts().to_dict())
hm = pd.util.hash_pandas_object(Xm, index=False).values
dupM = pd.DataFrame({'h': hm, 'y': lab}).duplicated().values
print('CELL-41 replication: removed', int(dupM.sum()), '(paper: 628,663) | share %.2f%%' % (100 * dupM.mean()))
V2 = {'M replication of CELL 41 (mixed dtype float32)': dupM}; V2.update(V)
for scope, m in [('all', np.ones(len(cdf), bool)), ('friday', day == 'friday')]:
    rows = {'before': pd.Series(lab[m]).value_counts()}
    for vn, dup in V2.items(): rows[vn[:2]] = pd.Series(lab[m & ~dup]).value_counts()
    T = pd.DataFrame(rows).fillna(0).astype(int).sort_values('before', ascending=False)
    T.loc['TOTAL'] = T.sum()
    print('\nPer-class flows, scope ' + scope + ' (columns: M = CELL-41 replication, A = float32 all, B = float64 ports excl, C = float64 ports incl)')
    print(T.to_string())
k = ~dupM; trc = set(lab[k & np.isin(day, ['monday', 'tuesday', 'wednesday', 'thursday'])]); fr = pd.Series(lab[k & (day == 'friday')])
print('\nM: Friday rows', len(fr), '| rho %.4f' % (1 - fr.isin(trc).mean()))
print('M: per-day dup share', {d: round(float(pd.Series(dupM).groupby(day).mean()[d]) * 100, 1) for d in ['monday', 'tuesday', 'wednesday', 'thursday', 'friday']})


dtypes after cast: {'int64': 53, 'float32': 24}
CELL-41 replication: removed 628663 (paper: 628,663) | share 22.23%

Per-class flows, scope all (columns: M = CELL-41 replication, A = float32 all, B = float64 ports excl, C = float64 ports incl)
                             before       M        A        B        C 
BENIGN                      2271320  1862721  1862527  1896672  2095057
DoS Hulk                     230124   172846   172846   172846   172846
PortScan                     158804     1956     1956     1956    90694
DDoS                         128025   128014   128014   128014   128014
DoS GoldenEye                 10293    10286    10282    10286    10286
FTP-Patator                    7935     5931     5931     5931     5931
SSH-Patator                    5897     3219     3219     3219     3219
DoS slowloris                  5796     5385     5374     5385     5385
DoS Slowhttptest               5499     5228     5228     5228     5228
Bot                            1956 

In [ ]:
# ===== E1c (self-contained): Table 6 rerun, duplicates defined WITH Destination Port; ports still excluded from model features =====
# CONFIG
import os, gc, time, warnings
MERGED = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017/preprocessed/all_days_merged.csv'
OUT = '/content/drive/MyDrive/UNSW-NB15'
RES = OUT + '/cic2017_table6_dedup_ports_runs.csv'
SEEDS = list(range(42, 52))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, sklearn, lightgbm, xgboost, imblearn, scipy
from scipy import stats
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
print('sklearn', sklearn.__version__, '| lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| imblearn', imblearn.__version__, '| scipy', scipy.__version__, '| pandas', pd.__version__)
TRAIN_DAYS = ['monday', 'tuesday', 'wednesday', 'thursday']
cic = pd.read_csv(MERGED); cic.columns = [c.strip() for c in cic.columns]
cic['day'] = cic['day'].str.lower(); cic['Label'] = cic['Label'].str.strip()
FEATS = [c for c in cic.columns if c not in ('Label', 'day', 'Destination Port')]
FE_P = FEATS + ['Destination Port']
hh = pd.util.hash_pandas_object(cic[FE_P], index=False).values
dupP = pd.DataFrame({'h': hh, 'y': cic['Label'].values}).duplicated().values
print('removed %d of %d (%.2f%%)' % (dupP.sum(), len(cic), 100 * dupP.mean()))
cic = cic[~dupP].reset_index(drop=True); del hh; gc.collect()
for c in cic.columns:
    if cic[c].dtype == 'float64': cic[c] = cic[c].astype('float32')
fri = cic[cic['day'] == 'friday']; trp = cic[cic['day'].isin(TRAIN_DAYS)]
rho = 1 - fri['Label'].isin(set(trp['Label'])).mean()
print('Friday rows %d | classes %s | rho %.4f | ceiling %.4f' % (len(fri), fri['Label'].value_counts().to_dict(), rho, 1 - rho)); del fri, trp
def prop_sample(df, n, seed):
    frac = n / len(df); parts = []
    for _, g in df.groupby('Label'):
        k = int(frac * len(g))
        if k > 0: parts.append(g.sample(n=k, random_state=seed))
    return pd.concat(parts)
def make_partitions(proto, seed):
    if proto == 'temporal':
        return prop_sample(cic[cic['day'].isin(TRAIN_DAYS)], 199_999, seed), prop_sample(cic[cic['day'] == 'friday'], 99_999, seed)
    pooled = prop_sample(cic, 199_999, seed)
    return train_test_split(pooled, test_size=0.2, stratify=(pooled['Label'] != 'BENIGN'), random_state=seed)
def prepare(tr, te, smote=True, seed=42):
    Xtr, Xte = tr[FEATS].astype('float64'), te[FEATS].astype('float64')
    vt = VarianceThreshold(0.01).fit(Xtr); cols = Xtr.columns[vt.get_support()]
    Xtr, Xte = Xtr[cols], Xte[cols]
    cm = Xtr.corr().abs(); up = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool))
    drop = [c for c in up.columns if (up[c] > 0.95).any()]; Xtr, Xte = Xtr.drop(columns=drop), Xte.drop(columns=drop)
    sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
    ytr, yte = (tr['Label'] != 'BENIGN').astype(int).values, (te['Label'] != 'BENIGN').astype(int).values
    n_syn = 0
    if smote:
        n_min, n_maj = int(ytr.sum()), int((ytr == 0).sum()); target = max(n_min, min(30000, n_maj)); k = max(1, min(5, n_min - 1))
        if target > n_min:
            Ztr, ytr = SMOTE(sampling_strategy={1: target}, k_neighbors=k, random_state=seed).fit_resample(Ztr, ytr); n_syn = target - n_min
    return Ztr, ytr, Zte, yte, Xtr.shape[1], n_syn
def metrics(yte, p): return dict(f1_macro=f1_score(yte, p, average='macro'), recall=recall_score(yte, p), fpr=((p == 1) & (yte == 0)).sum() / max((yte == 0).sum(), 1))

def base_models_det(seed):
    return {'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed),
            'LightGBM': LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1, deterministic=True, force_row_wise=True),
            'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, eval_metric='logloss', verbosity=0)}
done = set()
if os.path.exists(RES):
    r0 = pd.read_csv(RES); done = set(zip(r0.seed, r0.proto, r0.model))
print('resuming, done:', len(done)); t0 = time.time()
try:
    for seed in SEEDS:
        for proto in ['random', 'temporal']:
            todo = [m for m in ['RandomForest', 'LightGBM', 'XGBoost'] if (seed, proto, m) not in done]
            if not todo: continue
            tr, te = make_partitions(proto, seed); Ztr, ytr, Zte, yte, nf, ns = prepare(tr, te, True, seed); new = []
            for name, m in base_models_det(seed).items():
                if name not in todo: continue
                m.fit(Ztr, ytr); p = m.predict(Zte); r = metrics(yte, p)
                new.append(dict(seed=seed, proto=proto, model=name, f1_macro=round(r['f1_macro'], 4), recall=round(r['recall'], 4), fpr=round(r['fpr'], 4),
                                n_train=len(tr), n_test=len(te), att_test=int(yte.sum()), n_feat=nf, smote_added=ns, lgbm_deterministic=True)); done.add((seed, proto, name))
            pd.DataFrame(new).to_csv(RES, mode='a', header=not os.path.exists(RES), index=False)
            print('seed', seed, proto, 'feat', nf, 'smote', ns, '| elapsed %.0fs' % (time.time() - t0)); del Ztr, Zte; gc.collect()
finally:
    pass
R6 = pd.read_csv(RES)
def ci(v):
    v = np.asarray(v, float); k = len(v); m = v.mean(); se = v.std(ddof=1) / np.sqrt(k); t = stats.t.ppf(0.975, k - 1); return m, m - t * se, m + t * se
print('\n## Table 6 rerun, duplicates defined with Destination Port (runs completed: %d of 60)' % len(R6))
print('| model | random F1 | temporal F1 | gap pp | 95% CI | exact Wilcoxon p | positive pairs |'); print('|---|---|---|---|---|---|---|')
for mn in ['RandomForest', 'LightGBM', 'XGBoost']:
    g = R6[R6.model == mn]; a = g[g.proto == 'random'].set_index('seed').f1_macro; b = g[g.proto == 'temporal'].set_index('seed').f1_macro; s = a.index.intersection(b.index)
    d = (a.loc[s] - b.loc[s]) * 100; m, lo, hi = ci(d); pw = stats.wilcoxon(d, method='exact').pvalue
    print('| %s | %.4f | %.4f | %.2f | [%.2f, %.2f] | %.5f | %d of %d |' % (mn, a.loc[s].mean(), b.loc[s].mean(), m, lo, hi, pw, int((d > 0).sum()), len(s)))
print('\nPaper values (ports-excluded dedup): RF 48.56, LGBM 56.29, XGB 50.29; as captured 54.38, 60.75, 56.76. Table 6 rerun here: LightGBM deterministic=True, force_row_wise=True.')
print('Saved ->', RES)


sklearn 1.6.1 | lightgbm 4.6.0 | xgboost 3.4.1 | imblearn 0.14.2 | scipy 1.16.3 | pandas 2.2.3
removed 307078 of 2827876 (10.86%)
Friday rows 595860 | classes {'BENIGN': 375204, 'DDoS': 128014, 'PortScan': 90694, 'Bot': 1948} | rho 0.3703 | ceiling 0.6297
resuming, done: 3
seed 42 temporal feat 43 smote 8697 | elapsed 74s
seed 43 random feat 41 smote 2984 | elapsed 117s
seed 43 temporal feat 42 smote 8697 | elapsed 171s
seed 44 random feat 44 smote 2984 | elapsed 213s
seed 44 temporal feat 43 smote 8697 | elapsed 270s
seed 45 random feat 43 smote 2984 | elapsed 317s
seed 45 temporal feat 42 smote 8697 | elapsed 371s
seed 46 random feat 43 smote 2984 | elapsed 420s
seed 46 temporal feat 43 smote 8697 | elapsed 478s
seed 47 random feat 41 smote 2984 | elapsed 525s
seed 47 temporal feat 42 smote 8697 | elapsed 582s
seed 48 random feat 42 smote 2984 | elapsed 626s
seed 48 temporal feat 41 smote 8697 | elapsed 683s
seed 49 random feat 43 smote 2984 | elapsed 726s
seed 49 temporal feat 42 sm

In [ ]:
# ===== E1d (self-contained): Table 20 multiclass rerun, duplicates defined WITH Destination Port =====
# CONFIG
import os, gc, time, warnings
MERGED = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017/preprocessed/all_days_merged.csv'
OUT = '/content/drive/MyDrive/UNSW-NB15'
RES = OUT + '/cic_multiclass_temporal_dedup_ports_results.csv'
SEEDS = list(range(42, 52))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, sklearn, lightgbm, xgboost, imblearn, scipy
from scipy import stats
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
print('sklearn', sklearn.__version__, '| lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| imblearn', imblearn.__version__, '| scipy', scipy.__version__, '| pandas', pd.__version__)
TRAIN_DAYS = ['monday', 'tuesday', 'wednesday', 'thursday']
# ---- load, flag duplicates (identical float64 vector INCLUDING Destination Port + identical label, first kept), then cast as in the paper pipeline ----
cic = pd.read_csv(MERGED); cic.columns = [c.strip() for c in cic.columns]
cic['day'] = cic['day'].str.lower(); cic['Label'] = cic['Label'].str.strip()
FEATS = [c for c in cic.columns if c not in ('Label', 'day', 'Destination Port')]      # model features (ports excluded)
FE_P = FEATS + ['Destination Port']                                                      # duplicate definition (port included)
hh = pd.util.hash_pandas_object(cic[FE_P], index=False).values
dupP = pd.DataFrame({'h': hh, 'y': cic['Label'].values}).duplicated().values
print('removed %d of %d (%.2f%%)' % (dupP.sum(), len(cic), 100 * dupP.mean()))
cic = cic[~dupP].reset_index(drop=True); del hh; gc.collect()
for c in cic.columns:
    if cic[c].dtype == 'float64': cic[c] = cic[c].astype('float32')
fri = cic[cic['day'] == 'friday']; trp = cic[cic['day'].isin(TRAIN_DAYS)]
rho = 1 - fri['Label'].isin(set(trp['Label'])).mean()
print('Friday rows %d | classes %s | rho %.4f | ceiling %.4f' % (len(fri), fri['Label'].value_counts().to_dict(), rho, 1 - rho)); del fri, trp

def prop_sample(df, n, seed):
    frac = n / len(df); parts = []
    for _, g in df.groupby('Label'):
        k = int(frac * len(g))
        if k > 0: parts.append(g.sample(n=k, random_state=seed))
    return pd.concat(parts)
from sklearn.preprocessing import LabelEncoder
def build_mc(model, seed):
    if model == 'RandomForest': return RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=seed)
    if model == 'LightGBM': return LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1, deterministic=True, force_row_wise=True)
    return XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, eval_metric='mlogloss', verbosity=0)
res = pd.read_csv(RES) if os.path.exists(RES) else pd.DataFrame()
done = set(zip(res.seed, res.model)) if len(res) else set(); t00 = time.time()
for seed in SEEDS:
    tr = prop_sample(cic[cic['day'].isin(TRAIN_DAYS)], 199_999, seed); te = prop_sample(cic[cic['day'] == 'friday'], 99_999, seed)
    Xtr, Xte = tr[FEATS].astype('float64'), te[FEATS].astype('float64')
    vt = VarianceThreshold(0.01).fit(Xtr); cols = Xtr.columns[vt.get_support()]; Xtr, Xte = Xtr[cols], Xte[cols]
    cm_ = Xtr.corr().abs(); up = cm_.where(np.triu(np.ones(cm_.shape), k=1).astype(bool)); drop = [c for c in up.columns if (up[c] > 0.95).any()]
    Xtr, Xte = Xtr.drop(columns=drop), Xte.drop(columns=drop); sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
    le = LabelEncoder().fit(tr['Label']); ytr = le.transform(tr['Label']); known = te['Label'].isin(le.classes_).values
    rho = 1 - known.mean(); yte = np.where(known, le.transform(te['Label'].where(known, le.classes_[0])), -1)
    for name in ['RandomForest', 'LightGBM', 'XGBoost']:
        if (seed, name) in done: continue
        m = build_mc(name, seed).fit(Ztr, ytr); p = m.predict(Zte); acc = float((p == yte).mean())
        r = dict(seed=seed, model=name, rho=rho, accuracy=acc, ceiling=1 - rho, slack=1 - rho - acc, violated=bool(acc > 1 - rho + 1e-12), n_classes_train=len(le.classes_), n_test=len(te), lgbm_deterministic=True)
        res = pd.concat([res, pd.DataFrame([r])], ignore_index=True); res.to_csv(RES, index=False)
    print('seed', seed, 'rows', len(res), '| elapsed %.0fs' % (time.time() - t00)); gc.collect()
print('\n## Multiclass temporal after duplicate removal WITH Destination Port (runs completed: %d of 30)' % len(res))
print(res.groupby('model').agg(rho=('rho', 'mean'), accuracy=('accuracy', 'mean'), ceiling=('ceiling', 'mean'), slack=('slack', 'mean'), violations=('violated', 'sum'), runs=('seed', 'count')).round(4).to_string())
print('Saved ->', RES)


sklearn 1.6.1 | lightgbm 4.6.0 | xgboost 3.4.1 | imblearn 0.14.2 | scipy 1.16.3 | pandas 2.2.3
removed 307078 of 2827876 (10.86%)
Friday rows 595860 | classes {'BENIGN': 375204, 'DDoS': 128014, 'PortScan': 90694, 'Bot': 1948} | rho 0.3703 | ceiling 0.6297
seed 42 rows 3 | elapsed 140s
seed 43 rows 6 | elapsed 262s
seed 44 rows 9 | elapsed 389s
seed 45 rows 12 | elapsed 513s
seed 46 rows 15 | elapsed 635s
seed 47 rows 18 | elapsed 755s
seed 48 rows 21 | elapsed 877s
seed 49 rows 24 | elapsed 1008s
seed 50 rows 27 | elapsed 1123s
seed 51 rows 30 | elapsed 1246s

## Multiclass temporal after duplicate removal WITH Destination Port (runs completed: 30 of 30)
                 rho  accuracy  ceiling   slack  violations  runs
model                                                            
LightGBM      0.3703    0.6295   0.6297  0.0002           0    10
RandomForest  0.3703    0.6249   0.6297  0.0048           0    10
XGBoost       0.3703    0.6282   0.6297  0.0015           0    10
Saved -

In [ ]:
# ===== E1e (self-contained): ten within-window controls, duplicates defined WITH Destination Port (ports still excluded from features) =====
# Reduced pipeline of Section 3.2 (CELL 37/38): ports excluded, to_numeric, inf->NaN, NaN rows dropped, StandardScaler fit on train, NO variance/corr filter, NO SMOTE.
# Block = first..last flow of the family, BENIGN + family only; chronological cut = first 80 percent; random = stratified 80/20 of the same rows. LightGBM deterministic=True, force_row_wise=True.
import os, gc, time, warnings, numpy as np, pandas as pd, sklearn, lightgbm, xgboost, scipy
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, confusion_matrix, average_precision_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from scipy import stats
warnings.filterwarnings('ignore')
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
D = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017'
OUT = '/content/drive/MyDrive/UNSW-NB15'
RUNS = OUT + '/cic2017_windows_dedup_ports_runs.csv'
SEEDS = list(range(42, 52)); MIN_SIDE = 100; MNAMES = ['RandomForest', 'LightGBM', 'XGBoost']
print('sklearn', sklearn.__version__, '| lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| scipy', scipy.__version__, '| pandas', pd.__version__)
WINDOWS = {'SSH-Patator': ('Tuesday.csv', 'SSH-Patator'), 'DoS Hulk': ('Wednesday.csv', 'DoS Hulk'), 'Bot': ('Friday.csv', 'Bot'), 'PortScan': ('Friday+1.csv', 'PortScan'), 'DDoS': ('Friday+2.csv', 'DDoS')}
KEYS = {'GoldenEye': 'goldeneye', 'Slowloris': 'slowloris', 'Slowhttptest': 'slowhttp', 'WebBruteForce': 'brute force', 'WebXSS': 'xss'}
for fn in ['Wednesday.csv', 'Thursday.csv', 'Thursday+1.csv']:
    lab = pd.read_csv(D + '/' + fn, usecols=lambda c: c.strip() == 'Label', low_memory=False); lab.columns = ['Label']; vc = lab['Label'].value_counts()
    for name, key in KEYS.items():
        for l in vc.index:
            if key in str(l).lower() and name not in WINDOWS: WINDOWS[name] = (fn, l)
print('windows:', {k: v[1] for k, v in WINDOWS.items()})
def make_models(rs):
    return {'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=rs, n_jobs=-1),
            'LightGBM': LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=rs, n_jobs=-1, verbose=-1, deterministic=True, force_row_wise=True),
            'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=rs, n_jobs=-1, verbosity=0)}
def load_block(fname, cls):
    df = pd.read_csv(D + '/' + fname, low_memory=False); df.columns = df.columns.str.strip()
    df = df[df['Label'] != 'Label'].reset_index(drop=True)
    pos = np.where(df['Label'].values == cls)[0]; w = df.iloc[pos[0]:pos[-1] + 1]; w = w[w['Label'].isin(['BENIGN', cls])].reset_index(drop=True)
    y = (w['Label'].astype(str).str.strip() != 'BENIGN').astype(int).values
    X = w.drop(columns=['Label'], errors='ignore'); X = X.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    port = X['Destination Port'].values
    X = X.drop(columns=['Destination Port', 'Source Port'], errors='ignore')
    return X.reset_index(drop=True), y, port
def prep(Xtr, ytr, Xte, yte):
    mtr, mte = Xtr.notna().all(axis=1).values, Xte.notna().all(axis=1).values
    Xtr, ytr, Xte, yte = Xtr[mtr], ytr[mtr], Xte[mte], yte[mte]; sc = StandardScaler()
    return sc.fit_transform(Xtr.values), ytr, sc.transform(Xte.values), yte
def ev(m, Xte, yte):
    p = m.predict(Xte); f1m = f1_score(yte, p, average='macro', zero_division=0); am = (yte == 1); rec = (p[am] == 1).mean() if am.any() else float('nan')
    cm = confusion_matrix(yte, p, labels=[0, 1]); TN, FP = cm[0, 0], cm[0, 1]; fpr = FP / (FP + TN) if (FP + TN) else 0.0
    try: pr = average_precision_score(am.astype(int), m.predict_proba(Xte)[:, 1])
    except Exception: pr = float('nan')
    return f1m, rec, fpr, pr
done = set()
if os.path.exists(RUNS):
    r0 = pd.read_csv(RUNS); done = set(zip(r0.window, r0.seed, r0.split, r0.model))
print('resuming, rows already done:', len(done)); t00 = time.time(); skips = []
for cls, (fname, lab) in WINDOWS.items():
    X0, y0, port0 = load_block(fname, lab)
    h0 = pd.util.hash_pandas_object(X0.assign(_port=port0), index=False).values      # float64 features + port
    keep = ~pd.DataFrame({'h': h0, 'y': y0}).duplicated().values
    X, y, h = X0[keep].reset_index(drop=True), y0[keep], h0[keep]; n = len(y); cut = int(n * 0.8)
    a_tr, a_te = int(y[:cut].sum()), int(y[cut:].sum())
    print('\nWINDOW', cls, '| block rows', len(y0), '-> dedup', n, '(removed %.1f%%)' % (100 * (1 - n / len(y0))), '| chron attack train/test', a_tr, a_te)
    del X0, y0, port0, h0
    if a_tr < MIN_SIDE or a_te < MIN_SIDE: skips.append(cls); print('  SKIPPED (fewer than %d attack flows on one side)' % MIN_SIDE); continue
    chron_cache = None
    for seed in SEEDS:
        todo = [(sp, m) for sp in ['chron', 'random'] for m in MNAMES if (cls, seed, sp, m) not in done]
        if not todo: continue
        r_tr, r_te = train_test_split(np.arange(n), test_size=0.2, random_state=seed, stratify=y)
        splits = {'chron': (np.arange(cut), np.arange(cut, n)), 'random': (r_tr, r_te)}; new = []
        for sp, (itr, ite) in splits.items():
            if not any(s == sp for s, _ in todo): continue
            if sp == 'chron' and chron_cache is not None: A = chron_cache
            else:
                A = prep(X.iloc[itr], y[itr], X.iloc[ite], y[ite])
                if sp == 'chron': chron_cache = A
            Xtr_s, ytr_s, Xte_s, yte_s = A; trh = set(h[itr]); dup = float(np.mean([hh in trh for hh in h[ite]])); models = make_models(seed)
            for mn in MNAMES:
                if (cls, seed, sp, mn) in done: continue
                models[mn].fit(Xtr_s, ytr_s); f1m, rec, fpr, pr = ev(models[mn], Xte_s, yte_s)
                new.append(dict(variant='dedup_ports', window=cls, seed=seed, split=sp, model=mn, f1_macro=round(f1m, 4), attack_recall=round(rec, 4), FPR=round(fpr, 4), PR_AUC=round(pr, 4),
                                n_train=len(ytr_s), n_test=len(yte_s), att_train=int(ytr_s.sum()), att_test=int(yte_s.sum()), dup_share=round(dup, 4), lgbm_deterministic=True)); done.add((cls, seed, sp, mn))
            gc.collect()
        if new: pd.DataFrame(new).to_csv(RUNS, mode='a', header=not os.path.exists(RUNS), index=False)
        print('   ', cls, 'seed', seed, 'saved', len(new), '| elapsed %.0fs' % (time.time() - t00))
    del X, y, h, chron_cache; gc.collect()
R = pd.read_csv(RUNS)
def ci(v):
    v = np.asarray(v, float); k = len(v)
    if k < 2: return (float('nan'),) * 3
    m = v.mean(); se = v.std(ddof=1) / np.sqrt(k); t = stats.t.ppf(0.975, k - 1); return m, m - t * se, m + t * se
print('\n## Within-window gap (random minus chronological F1-macro, pp), duplicates defined with Destination Port. Runs completed: %d (expected 600). Skipped windows: %s' % (len(R), skips or 'none'))
print('| window | RF | LightGBM | XGBoost | window mean | attack train/test | chron dup share |'); print('|---|---|---|---|---|---|---|')
wm = {}
for cls in WINDOWS:
    cells = []; txt = []
    for mn in MNAMES:
        g = R[(R.window == cls) & (R.model == mn)]; a = g[g.split == 'random'].set_index('seed').f1_macro; b = g[g.split == 'chron'].set_index('seed').f1_macro; s = a.index.intersection(b.index)
        if len(s) < 2: txt.append('n/a'); continue
        d = (a.loc[s] - b.loc[s]) * 100; cells.append(d); m, lo, hi = ci(d); txt.append('%+.2f [%+.2f, %+.2f]' % (m, lo, hi))
    if not cells: continue
    wm[cls] = pd.concat(cells, axis=1).mean(axis=1).mean(); gc_ = R[(R.window == cls) & (R.split == 'chron')]
    print('| %s | %s | %s | %s | %+.2f | %d/%d | %.3f |' % (cls, txt[0], txt[1], txt[2], wm[cls], gc_.att_train.iloc[0], gc_.att_test.iloc[0], gc_.dup_share.mean()))
m, lo, hi = ci(list(wm.values()))
print('\nWINDOW-LEVEL (n=%d windows, t df=%d): %+.2f [%+.2f, %+.2f] | windows with positive mean: %d of %d' % (len(wm), len(wm) - 1, m, lo, hi, sum(v > 0 for v in wm.values()), len(wm)))
print('For comparison, ports-excluded dedup: +1.43 [-0.05, +2.91]; as captured: +0.76 [-0.06, +1.58].')
print('Saved ->', RUNS)


sklearn 1.6.1 | lightgbm 4.6.0 | xgboost 3.4.1 | scipy 1.16.3 | pandas 2.2.3
windows: {'SSH-Patator': 'SSH-Patator', 'DoS Hulk': 'DoS Hulk', 'Bot': 'Bot', 'PortScan': 'PortScan', 'DDoS': 'DDoS', 'GoldenEye': 'DoS GoldenEye', 'Slowloris': 'DoS slowloris', 'Slowhttptest': 'DoS Slowhttptest', 'WebBruteForce': 'Web Attack � Brute Force', 'WebXSS': 'Web Attack � XSS'}
resuming, rows already done: 600

WINDOW SSH-Patator | block rows 283829 -> dedup 268430 (removed 5.4%) | chron attack train/test 2676 543

WINDOW DoS Hulk | block rows 256173 -> dedup 197669 (removed 22.8%) | chron attack train/test 146080 26769

WINDOW Bot | block rows 166951 -> dedup 161258 (removed 3.4%) | chron attack train/test 1596 357

WINDOW PortScan | block rows 277626 -> dedup 205808 (removed 25.9%) | chron attack train/test 77581 13238

WINDOW DDoS | block rows 178660 -> dedup 177308 (removed 0.8%) | chron attack train/test 104762 23254

WINDOW GoldenEye | block rows 361653 -> dedup 342656 (removed 5.3%) | chron at

In [ ]:
# ===== E3 (self-contained): cut-point sensitivity over the ten within-window controls =====
# Reduced pipeline of Section 3.2 (CELL 37/38): ports excluded, to_numeric, inf->NaN, NaN rows dropped, StandardScaler fit on train, NO variance/corr filter, NO SMOTE.
# Block = first..last flow of the family, BENIGN + family only; train = first f of the block, test = rows f..min(f+0.2,1.0); random = stratified split of the SAME union of rows with the same train size. LightGBM deterministic=True, force_row_wise=True.
import os, gc, time, warnings, numpy as np, pandas as pd, sklearn, lightgbm, xgboost, scipy
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, confusion_matrix, average_precision_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from scipy import stats
warnings.filterwarnings('ignore')
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
D = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017'
OUT = '/content/drive/MyDrive/UNSW-NB15'
RUNS = OUT + '/cic2017_e3_cut_sweep_runs.csv'
REF_E1E = OUT + '/cic2017_windows_dedup_ports_runs.csv'; REF_O1 = OUT + '/cic2017_orig_pipeline_runs.csv'; REF_O2 = OUT + '/cic2017_orig_pipeline_extra_runs.csv'
SEEDS = [42, 43, 44]; FS = [0.5, 0.6, 0.7, 0.8, 0.9]; MIN_SIDE = 100; MNAMES = ['RandomForest', 'LightGBM', 'XGBoost']
print('sklearn', sklearn.__version__, '| lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| scipy', scipy.__version__, '| pandas', pd.__version__)
WINDOWS = {'SSH-Patator': ('Tuesday.csv', 'SSH-Patator'), 'DoS Hulk': ('Wednesday.csv', 'DoS Hulk'), 'Bot': ('Friday.csv', 'Bot'), 'PortScan': ('Friday+1.csv', 'PortScan'), 'DDoS': ('Friday+2.csv', 'DDoS')}
KEYS = {'GoldenEye': 'goldeneye', 'Slowloris': 'slowloris', 'Slowhttptest': 'slowhttp', 'WebBruteForce': 'brute force', 'WebXSS': 'xss'}
for fn in ['Wednesday.csv', 'Thursday.csv', 'Thursday+1.csv']:
    lab = pd.read_csv(D + '/' + fn, usecols=lambda c: c.strip() == 'Label', low_memory=False); lab.columns = ['Label']; vc = lab['Label'].value_counts()
    for name, key in KEYS.items():
        for l in vc.index:
            if key in str(l).lower() and name not in WINDOWS: WINDOWS[name] = (fn, l)
print('windows:', {k: v[1] for k, v in WINDOWS.items()})
def make_models(rs):
    return {'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=rs, n_jobs=-1),
            'LightGBM': LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=rs, n_jobs=-1, verbose=-1, deterministic=True, force_row_wise=True),
            'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=rs, n_jobs=-1, verbosity=0)}
def load_block(fname, cls):
    df = pd.read_csv(D + '/' + fname, low_memory=False); df.columns = df.columns.str.strip()
    df = df[df['Label'] != 'Label'].reset_index(drop=True)
    pos = np.where(df['Label'].values == cls)[0]; w = df.iloc[pos[0]:pos[-1] + 1]; w = w[w['Label'].isin(['BENIGN', cls])].reset_index(drop=True)
    y = (w['Label'].astype(str).str.strip() != 'BENIGN').astype(int).values
    X = w.drop(columns=['Label'], errors='ignore'); X = X.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    port = X['Destination Port'].values
    X = X.drop(columns=['Destination Port', 'Source Port'], errors='ignore')
    return X.reset_index(drop=True), y, port
def prep(Xtr, ytr, Xte, yte):
    mtr, mte = Xtr.notna().all(axis=1).values, Xte.notna().all(axis=1).values
    Xtr, ytr, Xte, yte = Xtr[mtr], ytr[mtr], Xte[mte], yte[mte]; sc = StandardScaler()
    return sc.fit_transform(Xtr.values), ytr, sc.transform(Xte.values), yte
def ev(m, Xte, yte):
    p = m.predict(Xte); f1m = f1_score(yte, p, average='macro', zero_division=0); am = (yte == 1); rec = (p[am] == 1).mean() if am.any() else float('nan')
    cm = confusion_matrix(yte, p, labels=[0, 1]); TN, FP = cm[0, 0], cm[0, 1]; fpr = FP / (FP + TN) if (FP + TN) else 0.0
    try: pr = average_precision_score(am.astype(int), m.predict_proba(Xte)[:, 1])
    except Exception: pr = float('nan')
    return f1m, rec, fpr, pr

done = set()
if os.path.exists(RUNS):
    r0 = pd.read_csv(RUNS); done = set(zip(r0.variant, r0.window, r0.f, r0.seed, r0.split, r0.model))
print('resuming, rows already done:', len(done)); t00 = time.time(); skips = []
for cls, (fname, lab) in WINDOWS.items():
    X0, y0, port0 = load_block(fname, lab)
    h0 = pd.util.hash_pandas_object(X0.assign(_port=port0), index=False).values
    keepd = ~pd.DataFrame({'h': h0, 'y': y0}).duplicated().values
    for var in ['captured', 'dedup_ports']:
        X, y = (X0, y0) if var == 'captured' else (X0[keepd].reset_index(drop=True), y0[keepd]); n = len(y)
        for f in FS:
            c0 = int(n * f); c1 = min(n, c0 + int(n * 0.2)); itr, ite = np.arange(c0), np.arange(c0, c1); a_tr, a_te = int(y[itr].sum()), int(y[ite].sum())
            if a_tr < MIN_SIDE or a_te < MIN_SIDE:
                skips.append((var, cls, f, a_tr, a_te)); print('  SKIP', var, cls, 'f=%.1f' % f, 'attack train/test', a_tr, a_te); continue
            chron_cache = None
            for seed in SEEDS:
                todo = [(sp, m) for sp in ['chron', 'random'] for m in MNAMES if (var, cls, f, seed, sp, m) not in done]
                if not todo: continue
                union = np.arange(c1); r_tr, r_te = train_test_split(union, train_size=c0, random_state=seed, stratify=y[union])
                splits = {'chron': (itr, ite), 'random': (np.sort(r_tr), np.sort(r_te))}; new = []
                for sp, (a, b) in splits.items():
                    if not any(s == sp for s, _ in todo): continue
                    if sp == 'chron' and chron_cache is not None: A = chron_cache
                    else:
                        A = prep(X.iloc[a], y[a], X.iloc[b], y[b])
                        if sp == 'chron': chron_cache = A
                    Xtr_s, ytr_s, Xte_s, yte_s = A; models = make_models(seed)
                    for mn in MNAMES:
                        if (var, cls, f, seed, sp, mn) in done: continue
                        models[mn].fit(Xtr_s, ytr_s); f1m, rec, fpr, pr = ev(models[mn], Xte_s, yte_s)
                        new.append(dict(variant=var, window=cls, f=f, seed=seed, split=sp, model=mn, f1_macro=round(f1m, 4), attack_recall=round(rec, 4), FPR=round(fpr, 4), PR_AUC=round(pr, 4),
                                        n_train=len(ytr_s), n_test=len(yte_s), att_train=int(ytr_s.sum()), att_test=int(yte_s.sum()), lgbm_deterministic=True)); done.add((var, cls, f, seed, sp, mn))
                    gc.collect()
                if new: pd.DataFrame(new).to_csv(RUNS, mode='a', header=not os.path.exists(RUNS), index=False)
            print('   ', var, cls, 'f=%.1f' % f, 'done | elapsed %.0fs' % (time.time() - t00))
    del X0, y0, port0, h0, X, y; gc.collect()
R = pd.read_csv(RUNS)
def ci(v):
    v = np.asarray(v, float); k = len(v)
    if k < 2: return (float('nan'),) * 3
    m = v.mean(); se = v.std(ddof=1) / np.sqrt(k); t = stats.t.ppf(0.975, k - 1); return m, m - t * se, m + t * se
def gaps(Rv):      # paired over seed and model: random minus chron, pp, per (window, f)
    a = Rv[Rv.split == 'random'].set_index(['window', 'f', 'seed', 'model']).f1_macro; b = Rv[Rv.split == 'chron'].set_index(['window', 'f', 'seed', 'model']).f1_macro
    d = ((a - b) * 100).dropna().rename('gap').reset_index(); return d
print('\n## E3 runs completed: %d | skipped (variant, window, f, attack train, attack test): %s' % (len(R), skips or 'none'))
WK = {}
for var in ['captured', 'dedup_ports']:
    d = gaps(R[R.variant == var]); att = R[(R.variant == var) & (R.split == 'chron')].groupby(['window', 'f'])[['att_train', 'att_test']].first()
    pw = d.groupby(['window', 'f']).gap.mean().unstack('f'); WK[var] = pw
    print('\n### %s: mean random-minus-chron F1-macro gap (pp), mean over seeds 42-44 and three models; attack train/test in brackets' % var)
    print('| window | ' + ' | '.join('f=%.1f' % f for f in FS) + ' |'); print('|---|' + '---|' * len(FS))
    for w in pw.index:
        cells = []
        for f in FS:
            if f in pw.columns and not np.isnan(pw.loc[w, f]): cells.append('%+.2f [%d/%d]' % (pw.loc[w, f], att.loc[(w, f)].att_train, att.loc[(w, f)].att_test))
            else: cells.append('skip')
        print('| %s | %s |' % (w, ' | '.join(cells)))
    print('\nPer f, mean across windows with window-level 95% t interval (n windows varies with skips):')
    print('| f | n windows | mean gap | 95% CI | windows positive |'); print('|---|---|---|---|---|')
    for f in FS:
        v = pw[f].dropna().values if f in pw.columns else []
        if len(v) >= 2: m, lo, hi = ci(v); print('| %.1f | %d | %+.2f | [%+.2f, %+.2f] | %d of %d |' % (f, len(v), m, lo, hi, int((v > 0).sum()), len(v)))
        else: print('| %.1f | %d | n/a | n/a | |' % (f, len(v)))
# ---- check f=0.8 against the Table-8 window means (seeds 42-51) ----
def ref_means(paths, var_name):
    Q = pd.concat([pd.read_csv(p) for p in paths if os.path.exists(p)], ignore_index=True); Q = Q[Q.variant == var_name]
    a = Q[Q.split == 'random'].set_index(['window', 'seed', 'model']).f1_macro; b = Q[Q.split == 'chron'].set_index(['window', 'seed', 'model']).f1_macro
    return ((a - b) * 100).dropna().groupby('window').mean()
print('\n### f=0.8 reproduction check against the earlier within-window runs (window means, pp); cells differing by more than 0.5 pp are flagged')
for var, paths, vn in [('captured', [REF_O1, REF_O2], 'captured'), ('dedup_ports', [REF_E1E], 'dedup_ports')]:
    ref = ref_means(paths, vn); print('\n' + var); print('| window | f=0.8 here | earlier (seeds 42-51) | difference | flag |'); print('|---|---|---|---|---|')
    for w in WK[var].index:
        if 0.8 in WK[var].columns and w in ref.index and not np.isnan(WK[var].loc[w, 0.8]):
            dlt = WK[var].loc[w, 0.8] - ref.loc[w]; print('| %s | %+.2f | %+.2f | %+.2f | %s |' % (w, WK[var].loc[w, 0.8], ref.loc[w], dlt, 'FLAG' if abs(dlt) > 0.5 else ''))
print('\nSaved ->', RUNS)


sklearn 1.6.1 | lightgbm 4.6.0 | xgboost 3.4.1 | scipy 1.16.3 | pandas 2.2.3
windows: {'SSH-Patator': 'SSH-Patator', 'DoS Hulk': 'DoS Hulk', 'Bot': 'Bot', 'PortScan': 'PortScan', 'DDoS': 'DDoS', 'GoldenEye': 'DoS GoldenEye', 'Slowloris': 'DoS slowloris', 'Slowhttptest': 'DoS Slowhttptest', 'WebBruteForce': 'Web Attack � Brute Force', 'WebXSS': 'Web Attack � XSS'}
resuming, rows already done: 0
    captured SSH-Patator f=0.5 done | elapsed 212s
    captured SSH-Patator f=0.6 done | elapsed 381s
    captured SSH-Patator f=0.7 done | elapsed 710s
    captured SSH-Patator f=0.8 done | elapsed 916s
    captured SSH-Patator f=0.9 done | elapsed 1140s
    dedup_ports SSH-Patator f=0.5 done | elapsed 1262s
    dedup_ports SSH-Patator f=0.6 done | elapsed 1414s
    dedup_ports SSH-Patator f=0.7 done | elapsed 1586s
    dedup_ports SSH-Patator f=0.8 done | elapsed 1781s
    dedup_ports SSH-Patator f=0.9 done | elapsed 2069s
    captured DoS Hulk f=0.5 done | elapsed 2220s
    captured DoS Hulk f

In [ ]:
# ===== E4 (self-contained): CSE-CIC-IDS 2018 feature-set confound. Original (filters fitted separately per protocol) vs filters fitted ONCE and applied to both protocols =====
# Pipeline = Objective 1 cell 100/102 (Table 12): Tuesday excluded; week1 (Feb 14-16) train / week2 (Feb 21-23) test; random = all weeks, cap 200k, stratified 80/20 on the original label;
# ports excluded; numeric coercion; inf->NaN; NaN rows dropped; VarianceThreshold 0.01; |r|>=0.95 filter; StandardScaler; SMOTE rule (Eq. 11); Table 2 models (LightGBM deterministic=True, force_row_wise=True).
# CONFIGS: orig = filters fitted on each protocol's own training partition (reproduces Table 12); week1_fixed = filters fitted once on the Week-1 training partition, same feature list for both protocols;
#          pooled_fixed = filters fitted once on the random-protocol training partition, same feature list for both protocols.
import os, gc, time, warnings, numpy as np, pandas as pd, sklearn, lightgbm, xgboost, imblearn, scipy
from collections import Counter
from scipy import stats
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
RAW = '/content/drive/MyDrive/New Approach/cicids2018/raw'; OUT = '/content/drive/MyDrive/UNSW-NB15'; RES = OUT + '/cic2018_e4_feature_confound_runs.csv'
SEEDS = list(range(42, 52)); LABEL = 'Label'; BEN = 'Benign'; EXCL = ['Dst Port', 'Src Port', 'Destination Port', 'Source Port']
WEEK = {'14-02-2018': 'week1', '15-02-2018': 'week1', '16-02-2018': 'week1', '21-02-2018': 'week2', '22-02-2018': 'week2', '23-02-2018': 'week2', '28-02-2018': 'week3', '01-03-2018': 'week3', '02-03-2018': 'week3'}
print('sklearn', sklearn.__version__, '| lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| imblearn', imblearn.__version__, '| scipy', scipy.__version__, '| pandas', pd.__version__)
FILES = []
for fn in sorted(os.listdir(RAW)):
    if not fn.endswith('.csv') or '20-02-2018' in fn: continue
    wk = next((w for d, w in WEEK.items() if d in fn), None)
    if wk: FILES.append((fn, wk))
print('files:', len(FILES))
# ---- pass 1: labels only (light frame with a global row id, same row order as the original concat) ----
parts = []
for fn, wk in FILES:
    lab = pd.read_csv(os.path.join(RAW, fn), usecols=lambda c: c.strip() == LABEL, low_memory=False); lab.columns = [LABEL]
    lab = lab[lab[LABEL] != LABEL].reset_index(drop=True); lab['week'] = wk; parts.append(lab)
light = pd.concat(parts, ignore_index=True); light['gid'] = np.arange(len(light)); del parts; gc.collect()
print('rows:', len(light), '| unseen-class share of week2 follows from labels')
def cap_df(df, n, seed):
    if len(df) <= n: return df.copy()
    return (df.groupby(LABEL, group_keys=False).apply(lambda x: x.sample(min(len(x), max(6, int(n * len(x) / len(df)))), random_state=seed)).reset_index(drop=True))
SEL = {}
for seed in SEEDS:
    trt = cap_df(light[light.week == 'week1'], 200_000, seed); tet = cap_df(light[light.week == 'week2'], 100_000, seed); rnd = cap_df(light, 200_000, seed)
    r_tr, r_te = train_test_split(rnd, test_size=0.2, random_state=seed, stratify=rnd[LABEL])
    SEL[seed] = dict(trt=trt.gid.values, tet=tet.gid.values, rtr=r_tr.gid.values, rte=r_te.gid.values)
need = np.unique(np.concatenate([a for s in SEL.values() for a in s.values()])); print('distinct rows needed over all seeds:', len(need))
# ---- pass 2: load features for the needed rows only (float64, as in the original pipeline) ----
parts = []; off = 0
for fn, wk in FILES:
    df = pd.read_csv(os.path.join(RAW, fn), low_memory=False); df.columns = df.columns.str.strip(); df = df[df[LABEL] != LABEL].reset_index(drop=True)
    gids = np.arange(off, off + len(df)); off += len(df); m = np.isin(gids, need); sub = df[m].copy(); del df; sub = sub.drop(columns=[c for c in sub.columns if c == 'Timestamp'], errors='ignore')
    for c in sub.columns:
        if c != LABEL: sub[c] = pd.to_numeric(sub[c], errors='coerce')
    sub['gid'] = gids[m]; parts.append(sub); del sub; gc.collect()
DATA = pd.concat(parts, ignore_index=True).set_index('gid'); del parts; gc.collect()
print('loaded', DATA.shape, '| memory %.1f GB' % (DATA.memory_usage(deep=True).sum() / 1e9))
def preprocess(gtr, gte, seed, fixed=None):
    dtr, dte = DATA.loc[gtr], DATA.loc[gte]
    Xtr, Xte = dtr.drop(columns=[LABEL], errors='ignore').copy(), dte.drop(columns=[LABEL], errors='ignore').copy()
    ytr_raw = dtr[LABEL].apply(lambda x: 'BENIGN' if str(x).strip() == BEN else 'ATTACK').reset_index(drop=True); yte_raw = dte[LABEL].apply(lambda x: 'BENIGN' if str(x).strip() == BEN else 'ATTACK').reset_index(drop=True)
    for X in (Xtr, Xte): X.drop(columns=[c for c in X.columns if c in EXCL], inplace=True, errors='ignore')
    Xtr = Xtr.apply(pd.to_numeric, errors='coerce'); Xte = Xte.apply(pd.to_numeric, errors='coerce')
    Xtr.dropna(axis=1, how='all', inplace=True); Xte.dropna(axis=1, how='all', inplace=True)
    Xtr.replace([np.inf, -np.inf], np.nan, inplace=True); Xte.replace([np.inf, -np.inf], np.nan, inplace=True)
    mtr, mte = Xtr.notna().all(axis=1).values, Xte.notna().all(axis=1).values
    Xtr = Xtr[mtr].reset_index(drop=True); ytr_raw = ytr_raw[mtr].reset_index(drop=True); Xte = Xte[mte].reset_index(drop=True); yte_raw = yte_raw[mte].reset_index(drop=True)
    common = [c for c in Xtr.columns if c in Xte.columns]; Xtr, Xte = Xtr[common].copy(), Xte[common].copy()
    if fixed is None:
        vt = VarianceThreshold(threshold=0.01); arr = vt.fit_transform(Xtr); cols = np.array(common)[vt.get_support()]
        Xtr = pd.DataFrame(arr, columns=cols); Xte = pd.DataFrame(vt.transform(Xte), columns=cols)
        corr = Xtr.corr().abs(); up = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)); drop = [c for c in up.columns if any(up[c] >= 0.95)]
        Xtr.drop(columns=drop, inplace=True); Xte.drop(columns=drop, inplace=True, errors='ignore')
    else:
        miss = [c for c in fixed if c not in Xtr.columns]; assert not miss, 'fixed feature(s) missing: %s' % miss
        Xtr, Xte = Xtr[fixed].copy(), Xte[fixed].copy()
    names = Xtr.columns.tolist(); sc = StandardScaler(); Ztr = pd.DataFrame(sc.fit_transform(Xtr), columns=names); Zte = pd.DataFrame(sc.transform(Xte), columns=names)
    le = LabelEncoder().fit(['BENIGN', 'ATTACK']); ytr = pd.Series(le.transform(ytr_raw)); yte = pd.Series(le.transform(yte_raw))
    cnt = Counter(ytr); mc = min(cnt, key=cnt.get); mn = cnt[mc]; mj = cnt[max(cnt, key=cnt.get)]; k = max(1, min(5, mn - 1)); target = max(mn, min(30_000, mj))
    Zs, ys = SMOTE(random_state=seed, k_neighbors=k, sampling_strategy={mc: target}).fit_resample(Ztr, ytr)
    return Zs, ys, Zte, yte, le, names, len(common)
def models(seed):
    return {'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=seed, n_jobs=-1),
            'LightGBM': LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1, deterministic=True, force_row_wise=True),
            'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=seed, n_jobs=-1, verbosity=0)}
def evaluate(m, Xte, yte, le):
    p = m.predict(Xte); cl = list(le.classes_); bi, ai = cl.index('BENIGN'), cl.index('ATTACK'); am = (yte == ai).values
    cm = confusion_matrix(yte, p, labels=[bi, ai]); TN, FP = cm[0, 0], cm[0, 1]
    try: roc = roc_auc_score(am.astype(int), m.predict_proba(Xte)[:, ai])
    except Exception: roc = float('nan')
    return dict(accuracy=accuracy_score(yte, p), f1_macro=f1_score(yte, p, average='macro', zero_division=0), attack_recall=(p[am] == ai).mean() if am.any() else 0.0, FPR=FP / (FP + TN) if (FP + TN) else 0.0, ROC_AUC=roc)
res = pd.read_csv(RES) if os.path.exists(RES) else pd.DataFrame(); done = set(zip(res.seed, res.config, res.proto, res.model)) if len(res) else set(); t0 = time.time()
for seed in SEEDS:
    s = SEL[seed]
    if all((seed, c, p, m) in done for c in ['orig', 'week1_fixed', 'pooled_fixed'] for p in ['random', 'temporal'] for m in ['RandomForest', 'LightGBM', 'XGBoost']): continue
    # original per-protocol preprocessing (also yields the two feature lists)
    P = {}
    P[('orig', 'temporal')] = preprocess(s['trt'], s['tet'], seed); P[('orig', 'random')] = preprocess(s['rtr'], s['rte'], seed)
    f_week1 = P[('orig', 'temporal')][5]; f_pool = P[('orig', 'random')][5]
    P[('week1_fixed', 'temporal')] = P[('orig', 'temporal')]; P[('week1_fixed', 'random')] = preprocess(s['rtr'], s['rte'], seed, fixed=f_week1)
    P[('pooled_fixed', 'random')] = P[('orig', 'random')]; P[('pooled_fixed', 'temporal')] = preprocess(s['trt'], s['tet'], seed, fixed=f_pool)
    new = []
    for (cfg, proto), (Ztr, ytr, Zte, yte, le, names, ncommon) in P.items():
        ms = models(seed)
        for mn, m in ms.items():
            if (seed, cfg, proto, mn) in done: continue
            m.fit(Ztr, ytr); r = evaluate(m, Zte, yte, le)
            new.append(dict(seed=seed, config=cfg, proto=proto, model=mn, n_feat=len(names), n_common=ncommon, n_train=len(ytr), n_test=len(yte), att_test=int(((yte == list(le.classes_).index('ATTACK'))).sum()), lgbm_deterministic=True, **{k: round(float(v), 4) for k, v in r.items()})); done.add((seed, cfg, proto, mn))
    pd.DataFrame(new).to_csv(RES, mode='a', header=not os.path.exists(RES), index=False)
    print('seed', seed, 'features: week1 %d | pooled %d | shared %d' % (len(f_week1), len(f_pool), len(set(f_week1) & set(f_pool))), '| saved', len(new), '| elapsed %.0fs' % (time.time() - t0)); del P; gc.collect()
R = pd.read_csv(RES)
def ci(v):
    v = np.asarray(v, float); k = len(v); m = v.mean()
    if k < 2: return m, float('nan'), float('nan')
    se = v.std(ddof=1) / np.sqrt(k); t = stats.t.ppf(0.975, k - 1); return m, m - t * se, m + t * se
print('\n## E4 CSE-CIC-IDS 2018 feature-set confound. Runs completed: %d of 180 | seeds %s' % (len(R), sorted(R.seed.unique())))
print('\n### Seed 42 (as Table 12): F1-macro, attack recall, FPR; random | temporal; gap = random minus temporal F1-macro (pp)')
print('| config | model | features rnd/tmp | F1 rnd | F1 tmp | gap pp | recall rnd/tmp | FPR rnd/tmp |'); print('|---|---|---|---|---|---|---|---|')
for cfg in ['orig', 'week1_fixed', 'pooled_fixed']:
    for mn in ['RandomForest', 'LightGBM', 'XGBoost']:
        a = R[(R.seed == 42) & (R.config == cfg) & (R.model == mn) & (R.proto == 'random')]; b = R[(R.seed == 42) & (R.config == cfg) & (R.model == mn) & (R.proto == 'temporal')]
        if len(a) and len(b): a, b = a.iloc[0], b.iloc[0]; print('| %s | %s | %d/%d | %.4f | %.4f | %+.2f | %.4f/%.4f | %.4f/%.4f |' % (cfg, mn, a.n_feat, b.n_feat, a.f1_macro, b.f1_macro, (a.f1_macro - b.f1_macro) * 100, a.attack_recall, b.attack_recall, a.FPR, b.FPR))
print('Paper Table 12 (seed 42, original): RF .9663/.4383, LGBM .9721/.4383, XGB .9730/.4383; recall rnd .9387/.9372/.9278, tmp 0; FPR rnd .0133/.0065/.0020.')
print('\n### All seeds: mean gap with 95% t-CI over seeds, attack recall and FPR (means)')
print('| config | model | seeds | F1 rnd | F1 tmp | gap pp [95% CI] | recall rnd/tmp | FPR rnd/tmp | features rnd/tmp |'); print('|---|---|---|---|---|---|---|---|---|')
for cfg in ['orig', 'week1_fixed', 'pooled_fixed']:
    for mn in ['RandomForest', 'LightGBM', 'XGBoost']:
        g = R[(R.config == cfg) & (R.model == mn)]; a = g[g.proto == 'random'].set_index('seed'); b = g[g.proto == 'temporal'].set_index('seed'); sd = a.index.intersection(b.index)
        if len(sd) < 1: continue
        d = (a.f1_macro.loc[sd] - b.f1_macro.loc[sd]) * 100; m, lo, hi = ci(d)
        print('| %s | %s | %d | %.4f | %.4f | %+.2f [%+.2f, %+.2f] | %.4f/%.4f | %.4f/%.4f | %.1f/%.1f |' % (cfg, mn, len(sd), a.f1_macro.loc[sd].mean(), b.f1_macro.loc[sd].mean(), m, lo, hi, a.attack_recall.loc[sd].mean(), b.attack_recall.loc[sd].mean(), a.FPR.loc[sd].mean(), b.FPR.loc[sd].mean(), a.n_feat.loc[sd].mean(), b.n_feat.loc[sd].mean()))
print('\nSaved ->', RES)


sklearn 1.6.1 | lightgbm 4.6.0 | xgboost 3.4.1 | imblearn 0.14.2 | scipy 1.16.3 | pandas 2.2.3
files: 9
rows: 8284195 | unseen-class share of week2 follows from labels
distinct rows needed over all seeds: 3437692
loaded (3437692, 79) | memory 2.4 GB
seed 42 features: week1 43 | pooled 41 | shared 40 | saved 18 | elapsed 204s
seed 43 features: week1 38 | pooled 41 | shared 35 | saved 18 | elapsed 404s
seed 44 features: week1 43 | pooled 41 | shared 40 | saved 18 | elapsed 614s
seed 45 features: week1 42 | pooled 41 | shared 40 | saved 18 | elapsed 811s
seed 46 features: week1 39 | pooled 35 | shared 35 | saved 18 | elapsed 972s
seed 47 features: week1 42 | pooled 42 | shared 40 | saved 18 | elapsed 1164s
seed 48 features: week1 42 | pooled 41 | shared 40 | saved 18 | elapsed 1361s
seed 49 features: week1 41 | pooled 42 | shared 39 | saved 18 | elapsed 1548s
seed 50 features: week1 39 | pooled 39 | shared 35 | saved 18 | elapsed 1718s
seed 51 features: week1 37 | pooled 42 | shared 34 | 

In [ ]:
# ===== E6 (self-contained): statistics pass over archived runs (no training) =====
import pandas as pd, numpy as np, scipy
from scipy import stats
U='/content/drive/MyDrive/UNSW-NB15/'; P='/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017/preprocessed/'
print('scipy', scipy.__version__, 'pandas', pd.__version__)
def wil(d):
    d=np.asarray(d,float); n=len(d)
    p=stats.wilcoxon(d, method='exact', alternative='two-sided').pvalue if np.any(d!=0) else 1.0
    nz=d[d!=0]; r=stats.rankdata(np.abs(nz)); wp=r[nz>0].sum(); wm=r[nz<0].sum()
    rb=(wp-wm)/(wp+wm) if (wp+wm)>0 else 0.0
    return p, float((d>0).mean()), rb
def tci(x):
    x=np.asarray(x,float); n=len(x); m=x.mean(); se=x.std(ddof=1)/np.sqrt(n); t=stats.t.ppf(0.975,n-1); return m,m-t*se,m+t*se
def holm(ps):
    ps=np.asarray(ps,float); o=np.argsort(ps); m=len(ps); adj=np.empty(m); run=0
    for k,i in enumerate(o):
        run=max(run,(m-k)*ps[i]); adj[i]=min(1.0,run)
    return adj
def table(rows, title):
    df=pd.DataFrame(rows,columns=['cell','n','mean gap pp','CI lo','CI hi','p exact','share>0','rank-biserial'])
    df['p Holm']=holm(df['p exact'].values)
    print('\n### '+title+' (Holm family size %d)'%len(df)); print('| cell | n | gap pp [95% CI] | exact Wilcoxon p | Holm p | share positive | rank-biserial |'); print('|---|---|---|---|---|---|---|')
    for _,r in df.iterrows(): print('| %s | %d | %+.2f [%+.2f, %+.2f] | %.5f | %.5f | %.2f | %+.2f |'%(r['cell'],r['n'],r['mean gap pp'],r['CI lo'],r['CI hi'],r['p exact'],r['p Holm'],r['share>0'],r['rank-biserial']))
    return df
def row(cell,d):
    m,lo,hi=tci(d); p,sh,rb=wil(d); return [cell,len(d),m,lo,hi,p,sh,rb]
out=[]
# ---- Table 6
sig=pd.read_csv(P+'significance_test_raw_runs.csv')
print('\nsignificance_test_raw_runs.csv F1 means (check against paper Table 6 captured 54.38/60.75/56.76):')
for mdl,g in sig.groupby('model'): print('  ',mdl,'gap %.2f'%(100*(g.f1m_random-g.f1m_temporal).mean()))
rows=[]
for mdl,g in sig.groupby('model'): g=g.sort_values('seed'); rows.append(row('captured(sig file) '+mdl,100*(g.f1m_random.values-g.f1m_temporal.values)))
for nm,f in [('dedup ports-excl','cic2017_table6_dedup_runs.csv'),('dedup ports-incl','cic2017_table6_dedup_ports_runs.csv')]:
    d=pd.read_csv(U+f)
    for mdl in ['RandomForest','LightGBM','XGBoost']:
        a=d[(d.model==mdl)&(d.proto=='random')].sort_values('seed').f1_macro.values; b=d[(d.model==mdl)&(d.proto=='temporal')].sort_values('seed').f1_macro.values
        rows.append(row(nm+' '+mdl,100*(a-b)))
out.append(('Table 6',table(rows,'Table 6: random minus chronological F1-macro, 10 seeds')))
# ---- Table 7
lc=pd.read_csv(P+'leakage_control_raw_runs.csv'); rows=[]
for (w,mdl),g in lc.groupby(['window','model']): g=g.sort_values('seed'); rows.append(row('%s %s'%(w,mdl),100*(g.f1_rand.values-g.f1_chron.values)))
out.append(('Table 7',table(rows,'Table 7: Tuesday/Wednesday within-window control')))
# ---- Table 8
frames={'captured':[], 'dedup ports-excl':[], 'dedup ports-incl':[]}
o1=pd.read_csv(U+'cic2017_orig_pipeline_runs.csv'); o2=pd.read_csv(U+'cic2017_orig_pipeline_extra_runs.csv'); o3=pd.read_csv(U+'cic2017_windows_dedup_ports_runs.csv')
allo=pd.concat([o1,o2]); cond={'captured':allo[allo.variant=='captured'],'dedup ports-excl':allo[allo.variant=='dedup'],'dedup ports-incl':o3}
print('\nTable 8 conditions: variants in ports file:', sorted(o3.variant.unique()))
for cn,d in cond.items():
    rows=[]; wl={}
    for (w,mdl),g in d.groupby(['window','model']):
        a=g[g.split=='random'].sort_values('seed').f1_macro.values; b=g[g.split=='chron'].sort_values('seed').f1_macro.values
        dd=100*(a-b); rows.append(row('%s %s'%(w,mdl),dd)); wl.setdefault(w,[]).append(dd)
    df=table(rows,'Table 8 cells, '+cn); out.append(('Table 8 '+cn,df))
    wm=np.array([np.mean(v,axis=0).mean() for v in wl.values()]); m,lo,hi=tci(wm)
    print('WINDOW-LEVEL (n=10 windows, df 9) %s: %+.2f [%+.2f, %+.2f]; windows positive %d/10; cells with Holm p<0.05: %d/30; cells positive share>0.5: %d/30'%(cn,m,lo,hi,(wm>0).sum(),(df['p Holm']<0.05).sum(),(df['share>0']>0.5).sum()))
# ---- Table 14 (UNSW-NB15)
u=pd.read_csv(U+'unsw_results.csv'); print('\nUNSW protocols:',sorted(u.protocol.unique())); rows=[]
for mdl in ['RandomForest','LightGBM','XGBoost']:
    for pr in [p for p in sorted(u.protocol.unique()) if p!='random']:
        a=u[(u.model==mdl)&(u.protocol=='random')].sort_values('seed').f1_macro.values; b=u[(u.model==mdl)&(u.protocol==pr)].sort_values('seed').f1_macro.values
        if len(a)==len(b): rows.append(row('%s random-%s'%(mdl,pr),100*(a-b)))
out.append(('Table 14',table(rows,'Table 14: UNSW-NB15')))
pd.concat([df.assign(table=t) for t,df in out]).to_csv(U+'cic_e6_statistics.csv',index=False); print('\nSaved -> cic_e6_statistics.csv')


scipy 1.16.3 pandas 2.2.3

significance_test_raw_runs.csv F1 means (check against paper Table 6 captured 54.38/60.75/56.76):
   LightGBM gap 60.75
   RandomForest gap 55.69
   XGBoost gap 59.47

### Table 6: random minus chronological F1-macro, 10 seeds (Holm family size 9)
| cell | n | gap pp [95% CI] | exact Wilcoxon p | Holm p | share positive | rank-biserial |
|---|---|---|---|---|---|---|
| captured(sig file) LightGBM | 10 | +60.75 [+59.17, +62.33] | 0.00195 | 0.01758 | 1.00 | +1.00 |
| captured(sig file) RandomForest | 10 | +55.69 [+51.08, +60.30] | 0.00195 | 0.01758 | 1.00 | +1.00 |
| captured(sig file) XGBoost | 10 | +59.47 [+57.13, +61.82] | 0.00195 | 0.01758 | 1.00 | +1.00 |
| dedup ports-excl RandomForest | 10 | +48.56 [+44.62, +52.50] | 0.00195 | 0.01758 | 1.00 | +1.00 |
| dedup ports-excl LightGBM | 10 | +56.29 [+54.60, +57.98] | 0.00195 | 0.01758 | 1.00 | +1.00 |
| dedup ports-excl XGBoost | 10 | +50.29 [+45.29, +55.28] | 0.00195 | 0.01758 | 1.00 | +1.00 |
| dedup ports-i

In [ ]:
# ===== E7 (self-contained): LightGBM determinism check on the Table 6 pipeline (duplicates with Destination Port) =====
# CONFIG
import os, gc, time, warnings
MERGED = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017/preprocessed/all_days_merged.csv'
OUT = '/content/drive/MyDrive/UNSW-NB15'
ARCH = OUT + '/cic2017_table6_dedup_ports_runs.csv'
RES = OUT + '/cic2017_e7_lgbm_determinism_runs.csv'
SEEDS = list(range(42, 52))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, sklearn, lightgbm, xgboost, imblearn, scipy
from scipy import stats
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
print('sklearn', sklearn.__version__, '| lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| imblearn', imblearn.__version__, '| scipy', scipy.__version__, '| pandas', pd.__version__)
TRAIN_DAYS = ['monday', 'tuesday', 'wednesday', 'thursday']
# ---- load, flag duplicates (identical float64 vector INCLUDING Destination Port + identical label, first kept), then cast as in the paper pipeline ----
cic = pd.read_csv(MERGED); cic.columns = [c.strip() for c in cic.columns]
cic['day'] = cic['day'].str.lower(); cic['Label'] = cic['Label'].str.strip()
FEATS = [c for c in cic.columns if c not in ('Label', 'day', 'Destination Port')]      # model features (ports excluded)
FE_P = FEATS + ['Destination Port']                                                      # duplicate definition (port included)
hh = pd.util.hash_pandas_object(cic[FE_P], index=False).values
dupP = pd.DataFrame({'h': hh, 'y': cic['Label'].values}).duplicated().values
print('removed %d of %d (%.2f%%)' % (dupP.sum(), len(cic), 100 * dupP.mean()))
cic = cic[~dupP].reset_index(drop=True); del hh; gc.collect()
for c in cic.columns:
    if cic[c].dtype == 'float64': cic[c] = cic[c].astype('float32')
fri = cic[cic['day'] == 'friday']; trp = cic[cic['day'].isin(TRAIN_DAYS)]
rho = 1 - fri['Label'].isin(set(trp['Label'])).mean()
print('Friday rows %d | classes %s | rho %.4f | ceiling %.4f' % (len(fri), fri['Label'].value_counts().to_dict(), rho, 1 - rho)); del fri, trp
def prop_sample(df, n, seed):
    frac = n / len(df); parts = []
    for _, g in df.groupby('Label'):
        k = int(frac * len(g))
        if k > 0: parts.append(g.sample(n=k, random_state=seed))
    return pd.concat(parts)
def make_partitions(proto, seed):
    if proto == 'temporal':
        return prop_sample(cic[cic['day'].isin(TRAIN_DAYS)], 199_999, seed), prop_sample(cic[cic['day'] == 'friday'], 99_999, seed)
    pooled = prop_sample(cic, 199_999, seed)
    return train_test_split(pooled, test_size=0.2, stratify=(pooled['Label'] != 'BENIGN'), random_state=seed)
def prepare(tr, te, smote=True, seed=42):
    Xtr, Xte = tr[FEATS].astype('float64'), te[FEATS].astype('float64')
    vt = VarianceThreshold(0.01).fit(Xtr); cols = Xtr.columns[vt.get_support()]
    Xtr, Xte = Xtr[cols], Xte[cols]
    cm = Xtr.corr().abs(); up = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool))
    drop = [c for c in up.columns if (up[c] > 0.95).any()]; Xtr, Xte = Xtr.drop(columns=drop), Xte.drop(columns=drop)
    sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
    ytr, yte = (tr['Label'] != 'BENIGN').astype(int).values, (te['Label'] != 'BENIGN').astype(int).values
    n_syn = 0
    if smote:
        n_min, n_maj = int(ytr.sum()), int((ytr == 0).sum()); target = max(n_min, min(30000, n_maj)); k = max(1, min(5, n_min - 1))
        if target > n_min:
            Ztr, ytr = SMOTE(sampling_strategy={1: target}, k_neighbors=k, random_state=seed).fit_resample(Ztr, ytr); n_syn = target - n_min
    return Ztr, ytr, Zte, yte, Xtr.shape[1], n_syn
def metrics(yte, p): return dict(f1_macro=f1_score(yte, p, average='macro'), recall=recall_score(yte, p), fpr=((p == 1) & (yte == 0)).sum() / max((yte == 0).sum(), 1))


def lgbm(seed, det):
    kw = dict(deterministic=True, force_row_wise=True) if det else {}
    return LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1, **kw)
arch = pd.read_csv(ARCH); arch = arch[arch.model == 'LightGBM'].set_index(['seed', 'proto'])
done = set()
if os.path.exists(RES):
    r0 = pd.read_csv(RES); done = set(zip(r0.seed, r0.proto))
print('resuming, done:', len(done)); t0 = time.time()
for seed in SEEDS:
    for proto in ['random', 'temporal']:
        if (seed, proto) in done: continue
        tr, te = make_partitions(proto, seed); Ztr, ytr, Zte, yte, nf, ns = prepare(tr, te, True, seed); preds = {}
        for tag, det in [('det1', True), ('det2', True), ('def1', False), ('def2', False)]:
            m = lgbm(seed, det).fit(Ztr, ytr); preds[tag] = (m.predict(Zte), m.predict_proba(Zte)[:, 1])
        row = dict(seed=seed, proto=proto, n_feat=nf)
        for tag, (p, pr) in preds.items():
            row['f1_' + tag] = round(f1_score(yte, p, average='macro'), 6); row['recall_' + tag] = round(recall_score(yte, p), 6)
        row['det_identical_pred'] = bool((preds['det1'][0] == preds['det2'][0]).all()); row['det_identical_proba'] = bool(np.array_equal(preds['det1'][1], preds['det2'][1]))
        row['def_identical_pred'] = bool((preds['def1'][0] == preds['def2'][0]).all()); row['def_identical_proba'] = bool(np.array_equal(preds['def1'][1], preds['def2'][1]))
        row['det_vs_default_pred_diff'] = int((preds['det1'][0] != preds['def1'][0]).sum())
        row['archived_f1'] = float(arch.loc[(seed, proto), 'f1_macro']); row['archived_matches_det1_4dp'] = bool(round(row['f1_det1'], 4) == row['archived_f1'])
        pd.DataFrame([row]).to_csv(RES, mode='a', header=not os.path.exists(RES), index=False)
        print('seed', seed, proto, 'det==det', row['det_identical_pred'], 'default==default', row['def_identical_pred'], 'archived match', row['archived_matches_det1_4dp'], '| elapsed %.0fs' % (time.time() - t0)); del Ztr, Zte; gc.collect()
R = pd.read_csv(RES)
print('\n## E7 LightGBM determinism check, Table 6 pipeline (ports-included dedup); runs completed: %d of 20' % len(R))
print('Deterministic fit twice, identical predictions: %d of %d | identical probabilities: %d of %d' % (R.det_identical_pred.sum(), len(R), R.det_identical_proba.sum(), len(R)))
print('Default (non-deterministic) fit twice, identical predictions: %d of %d | identical probabilities: %d of %d' % (R.def_identical_pred.sum(), len(R), R.def_identical_proba.sum(), len(R)))
print('Archived E1c F1 (4 dp) equals deterministic rerun: %d of %d' % (R.archived_matches_det1_4dp.sum(), len(R)))
print('Test rows whose prediction differs, deterministic vs default fit: mean %.1f, max %d' % (R.det_vs_default_pred_diff.mean(), R.det_vs_default_pred_diff.max()))
print('| proto | mean F1 det | mean F1 default (run1) | mean F1 default (run2) | max abs F1 diff det vs default |'); print('|---|---|---|---|---|')
for pr in ['random', 'temporal']:
    g = R[R.proto == pr]; print('| %s | %.5f | %.5f | %.5f | %.5f |' % (pr, g.f1_det1.mean(), g.f1_def1.mean(), g.f1_def2.mean(), (g.f1_det1 - g.f1_def1).abs().max()))
a = R[R.proto == 'random'].set_index('seed'); b = R[R.proto == 'temporal'].set_index('seed')
for tag in ['det1', 'def1']:
    d = 100 * (a['f1_' + tag] - b['f1_' + tag]); k = len(d); m = d.mean(); se = d.std(ddof=1) / np.sqrt(k); t = stats.t.ppf(0.975, k - 1)
    print('LightGBM gap (%s): %.2f [%.2f, %.2f]' % (tag, m, m - t * se, m + t * se))
print('Saved ->', RES)


sklearn 1.6.1 | lightgbm 4.6.0 | xgboost 3.4.1 | imblearn 0.14.2 | scipy 1.16.3 | pandas 2.2.3
removed 307078 of 2827876 (10.86%)
Friday rows 595860 | classes {'BENIGN': 375204, 'DDoS': 128014, 'PortScan': 90694, 'Bot': 1948} | rho 0.3703 | ceiling 0.6297
resuming, done: 0
seed 42 random det==det True default==default True archived match True | elapsed 44s
seed 42 temporal det==det True default==default True archived match True | elapsed 74s
seed 43 random det==det True default==default True archived match True | elapsed 83s
seed 43 temporal det==det True default==default True archived match True | elapsed 94s
seed 44 random det==det True default==default True archived match True | elapsed 103s
seed 44 temporal det==det True default==default True archived match True | elapsed 114s
seed 45 random det==det True default==default True archived match True | elapsed 123s
seed 45 temporal det==det True default==default True archived match True | elapsed 134s
seed 46 random det==det True defau

In [ ]:
# ===== E7b (self-contained): LightGBM determinism check on the within-window (Table 8) pipeline, dedup with Destination Port; windows Slowhttptest, Bot, WebXSS =====
import os, gc, time, warnings, numpy as np, pandas as pd, sklearn, lightgbm, xgboost, scipy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
D = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017'
OUT = '/content/drive/MyDrive/UNSW-NB15'
ARCH = OUT + '/cic2017_windows_dedup_ports_runs.csv'
RUNS = OUT + '/cic2017_e7b_lgbm_window_determinism_runs.csv'
SEEDS = list(range(42, 52))
print('sklearn', sklearn.__version__, '| lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| scipy', scipy.__version__, '| pandas', pd.__version__)
WINDOWS = {'Bot': ('Friday.csv', 'Bot')}
KEYS = {'Slowhttptest': 'slowhttp', 'WebXSS': 'xss'}
for fn in ['Wednesday.csv', 'Thursday.csv', 'Thursday+1.csv']:
    lab = pd.read_csv(D + '/' + fn, usecols=lambda c: c.strip() == 'Label', low_memory=False); lab.columns = ['Label']; vc = lab['Label'].value_counts()
    for name, key in KEYS.items():
        for l in vc.index:
            if key in str(l).lower() and name not in WINDOWS: WINDOWS[name] = (fn, l)
print('windows:', {k: v[1] for k, v in WINDOWS.items()})
def load_block(fname, cls):
    df = pd.read_csv(D + '/' + fname, low_memory=False); df.columns = df.columns.str.strip()
    df = df[df['Label'] != 'Label'].reset_index(drop=True)
    pos = np.where(df['Label'].values == cls)[0]; w = df.iloc[pos[0]:pos[-1] + 1]; w = w[w['Label'].isin(['BENIGN', cls])].reset_index(drop=True)
    y = (w['Label'].astype(str).str.strip() != 'BENIGN').astype(int).values
    X = w.drop(columns=['Label'], errors='ignore'); X = X.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    port = X['Destination Port'].values
    X = X.drop(columns=['Destination Port', 'Source Port'], errors='ignore')
    return X.reset_index(drop=True), y, port
def prep(Xtr, ytr, Xte, yte):
    mtr, mte = Xtr.notna().all(axis=1).values, Xte.notna().all(axis=1).values
    Xtr, ytr, Xte, yte = Xtr[mtr], ytr[mtr], Xte[mte], yte[mte]; sc = StandardScaler()
    return sc.fit_transform(Xtr.values), ytr, sc.transform(Xte.values), yte
t00 = time.time()
def lg(rs, det):
    kw = dict(deterministic=True, force_row_wise=True) if det else {}
    return LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=rs, n_jobs=-1, verbose=-1, **kw)
arch = pd.read_csv(ARCH); arch = arch[arch.model == 'LightGBM'].set_index(['window', 'seed', 'split']).f1_macro
rows = []
for cls in ['Slowhttptest', 'Bot', 'WebXSS']:
    fname, lab = WINDOWS[cls]; X0, y0, port0 = load_block(fname, lab)
    h0 = pd.util.hash_pandas_object(X0.assign(_port=port0), index=False).values; keep = ~pd.DataFrame({'h': h0, 'y': y0}).duplicated().values
    X, y = X0[keep].reset_index(drop=True), y0[keep]; n = len(y); cut = int(n * 0.8); del X0, y0, port0, h0
    for seed in SEEDS:
        r_tr, r_te = train_test_split(np.arange(n), test_size=0.2, random_state=seed, stratify=y)
        for sp, (itr, ite) in {'chron': (np.arange(cut), np.arange(cut, n)), 'random': (r_tr, r_te)}.items():
            Xtr_s, ytr_s, Xte_s, yte_s = prep(X.iloc[itr], y[itr], X.iloc[ite], y[ite]); P = {}
            for tag, det in [('det1', True), ('det2', True), ('def1', False), ('def2', False)]:
                m = lg(seed, det).fit(Xtr_s, ytr_s); P[tag] = m.predict(Xte_s)
            f = {t: f1_score(yte_s, p, average='macro', zero_division=0) for t, p in P.items()}
            rows.append(dict(window=cls, seed=seed, split=sp, det_same=bool((P['det1'] == P['det2']).all()), def_same=bool((P['def1'] == P['def2']).all()), n_diff_det_vs_def=int((P['det1'] != P['def1']).sum()),
                             f1_det=f['det1'], f1_def=f['def1'], arch_f1=float(arch.loc[(cls, seed, sp)]), arch_match=bool(round(f['det1'], 4) == float(arch.loc[(cls, seed, sp)]))))
    print(cls, 'done | elapsed %.0fs' % (time.time() - t00)); del X, y; gc.collect()
Rr = pd.DataFrame(rows); Rr.to_csv(RUNS, index=False)
print('\n## E7b LightGBM determinism, within-window pipeline: %d fits per condition' % len(Rr))
for cls, g in Rr.groupby('window'):
    print('%s: deterministic fit twice identical %d/%d | default fit twice identical %d/%d | archived F1 reproduced (4 dp) %d/%d | max rows differing det vs default %d | max |F1 det - F1 default| %.4f' % (cls, g.det_same.sum(), len(g), g.def_same.sum(), len(g), g.arch_match.sum(), len(g), g.n_diff_det_vs_def.max(), (g.f1_det - g.f1_def).abs().max()))
print('Saved ->', RUNS)


sklearn 1.6.1 | lightgbm 4.6.0 | xgboost 3.4.1 | scipy 1.16.3 | pandas 2.2.3
windows: {'Bot': 'Bot', 'Slowhttptest': 'DoS Slowhttptest', 'WebXSS': 'Web Attack � XSS'}
Slowhttptest done | elapsed 92s
Bot done | elapsed 193s
WebXSS done | elapsed 216s

## E7b LightGBM determinism, within-window pipeline: 60 fits per condition
Bot: deterministic fit twice identical 20/20 | default fit twice identical 20/20 | archived F1 reproduced (4 dp) 20/20 | max rows differing det vs default 0 | max |F1 det - F1 default| 0.0000
Slowhttptest: deterministic fit twice identical 20/20 | default fit twice identical 20/20 | archived F1 reproduced (4 dp) 20/20 | max rows differing det vs default 0 | max |F1 det - F1 default| 0.0000
WebXSS: deterministic fit twice identical 20/20 | default fit twice identical 20/20 | archived F1 reproduced (4 dp) 20/20 | max rows differing det vs default 0 | max |F1 det - F1 default| 0.0000
Saved -> /content/drive/MyDrive/UNSW-NB15/cic2017_e7b_lgbm_window_determinism_runs.csv

In [ ]:
# ===== STEP 0 (final run v10): environment pins, data checks, duplicate masks D1 / D2 (no model training) =====
import os, sys, subprocess, importlib, json, time, gc, warnings
warnings.filterwarnings('ignore')
# ---------------- CONFIG ----------------
RAW_DIR = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017'          # per-day MachineLearningCVE CSVs
FILES = ['Monday.csv', 'Tuesday.csv', 'Wednesday.csv', 'Thursday.csv', 'Thursday+1.csv', 'Friday.csv', 'Friday+1.csv', 'Friday+2.csv']   # capture order
OUT = '/content/drive/MyDrive/UNSW-NB15/final_run_v10'                            # ONE output folder for the whole final run
PINS = {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3'}
XGB_TRY, XGB_FALLBACK = '3.3.0', '3.4.1'
EXPECT_TOTAL = 2_827_876
EXPECT_DAY = {'monday': (529_481, 0), 'tuesday': (445_645, 13_832), 'wednesday': (691_406, 251_723), 'thursday': (458_626, 2_216), 'friday': (702_718, 288_785)}   # v9 Table 3 (flows, attack flows)
EXPECT_RHO = 0.411
E1 = {'D1 removed': 307_078, 'D1 Friday flows': 595_860, 'D1 rho': 0.3703, 'D2 removed (E1 float64, ports excl)': 594_712, 'v9 ports-excl removed (mixed dtype)': 628_663}
# ---------------- mount ----------------
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
os.makedirs(OUT, exist_ok=True)
# ---------------- (a) environment ----------------
def sh(cmd): return subprocess.run(cmd, shell=True, capture_output=True, text=True)
r = sh('%s -m pip install -q %s' % (sys.executable, ' '.join('%s==%s' % kv for kv in PINS.items())))
print('pip pins:', 'ok' if r.returncode == 0 else 'FAILED\n' + r.stderr[-800:])
r = sh('%s -m pip install -q xgboost==%s' % (sys.executable, XGB_TRY)); xgb_used = XGB_TRY
if r.returncode != 0:
    print('xgboost==%s could not be installed:\n%s' % (XGB_TRY, r.stderr[-500:]))
    r = sh('%s -m pip install -q xgboost==%s' % (sys.executable, XGB_FALLBACK)); xgb_used = XGB_FALLBACK
    print('FALLBACK: using xgboost %s (v9 states %s)' % (XGB_FALLBACK, XGB_TRY))
import importlib.metadata as md
inst = {k: md.version(k) for k in list(PINS) + ['xgboost']}
import numpy as np, pandas as pd, sklearn, lightgbm, imblearn, scipy, xgboost
loaded = {'scikit-learn': sklearn.__version__, 'lightgbm': lightgbm.__version__, 'imbalanced-learn': imblearn.__version__, 'scipy': scipy.__version__, 'xgboost': xgboost.__version__}
print('installed (disk):', inst); print('loaded (this process):', loaded)
bad = [k for k in loaded if loaded[k] != inst[k]]
if bad:
    raise RuntimeError('Loaded versions differ from installed for %s. Do Runtime > Restart session, then re-run this same cell once (do not run anything else first).' % bad)
for k, v in PINS.items(): assert loaded[k] == v, (k, loaded[k], v)
assert loaded['xgboost'] == xgb_used, (loaded['xgboost'], xgb_used)
with open(OUT + '/environment.txt', 'w') as f:
    f.write('python %s\n' % sys.version.replace('\n', ' ')); f.write('xgboost used: %s (v9 states %s)\n' % (xgb_used, XGB_TRY)); f.write(sh('%s -m pip freeze' % sys.executable).stdout)
print('environment.txt written; python', sys.version.split()[0], '| xgboost used:', xgb_used, '(v9 states %s)' % XGB_TRY)
# ---------------- (b) load, clean, verify ----------------
t0 = time.time(); parts = []; log = []; cols0 = None
for fn in FILES:
    p = os.path.join(RAW_DIR, fn)
    if not os.path.exists(p): print('MISSING FILE:', p, '| folder has:', sorted(os.listdir(RAW_DIR))); raise FileNotFoundError(p)
    df = pd.read_csv(p, low_memory=False); df.columns = df.columns.str.strip(); n_raw = len(df)
    df['src_row'] = np.arange(n_raw)                                            # original row order within the file
    df = df[df['Label'].astype(str).str.strip() != 'Label'].copy(); n_hdr = n_raw - len(df)
    df['Label'] = df['Label'].astype(str).str.strip()
    feats = [c for c in df.columns if c not in ('Label', 'src_row')]
    if cols0 is None: cols0 = feats
    elif feats != cols0: print('COLUMN MISMATCH in', fn, set(feats) ^ set(cols0))
    X = df[feats].apply(pd.to_numeric, errors='coerce').astype('float64').replace([np.inf, -np.inf], np.nan)
    ok = X.notna().all(axis=1).values
    day = fn.split('.')[0].split('+')[0].lower()
    log.append(dict(file=fn, day=day, raw_rows=n_raw, header_rows=n_hdr, nan_or_inf_rows=int((~ok).sum()), kept=int(ok.sum())))
    X['Label'] = df['Label'].values; X['src_row'] = df['src_row'].values; X['src_file'] = fn; X['day'] = day
    parts.append(X[ok]); del df, X; gc.collect()
cl = pd.concat(parts, ignore_index=True); del parts; gc.collect()
cl['rid'] = np.arange(len(cl))                                                  # global row id in capture order
FEATS = cols0; assert 'Destination Port' in FEATS
print('\n### Step 0b: loading and cleaning (%.0fs)' % (time.time() - t0)); print('| file | day | raw rows | header rows | NaN/inf rows dropped | kept |'); print('|---|---|---|---|---|---|')
for r_ in log: print('| %(file)s | %(day)s | %(raw_rows)d | %(header_rows)d | %(nan_or_inf_rows)d | %(kept)d |' % r_)
print('numeric feature columns: %d (incl. Destination Port); first 3: %s' % (len(FEATS), FEATS[:3]))
tot = len(cl); print('\nTotal flows after cleaning: %d (v9 Table 3: %d) -> %s' % (tot, EXPECT_TOTAL, 'MATCH' if tot == EXPECT_TOTAL else 'MISMATCH'))
print('| day | flows | v9 flows | attack flows | v9 attack | attack share | status |'); print('|---|---|---|---|---|---|---|')
for d_, (nf, na) in EXPECT_DAY.items():
    g = cl[cl.day == d_]; a = int((g.Label != 'BENIGN').sum())
    print('| %s | %d | %d | %d | %d | %.2f%% | %s |' % (d_, len(g), nf, a, na, 100 * a / max(len(g), 1), 'MATCH' if (len(g), a) == (nf, na) else 'MISMATCH'))
TRAIN_DAYS = ['monday', 'tuesday', 'wednesday', 'thursday']
def rho_of(frame):
    trc = set(frame.loc[frame.day.isin(TRAIN_DAYS), 'Label']); fr = frame[frame.day == 'friday']
    return 1 - fr['Label'].isin(trc).mean(), len(fr)
rho0, nfri0 = rho_of(cl); print('rho(Friday), no removal: %.4f (v9: %.3f) -> %s' % (rho0, EXPECT_RHO, 'MATCH' if round(rho0, 3) == EXPECT_RHO else 'MISMATCH'))
# ---------------- (c) duplicate masks (pandas duplicated, float64, capture order, keep first) ----------------
t0 = time.time(); F64 = [c for c in FEATS]; assert all(cl[c].dtype == 'float64' for c in F64)
D1_dup = cl[F64 + ['Label']].duplicated(keep='first').values                                            # record duplicates, Destination Port INCLUDED
D2_dup = cl[[c for c in F64 if c != 'Destination Port'] + ['Label']].duplicated(keep='first').values    # model-view duplicates, Destination Port EXCLUDED
print('\nmasks built in %.0fs' % (time.time() - t0))
masks = pd.DataFrame({'rid': cl.rid.values, 'src_file': cl.src_file.values, 'src_row': cl.src_row.values, 'day': cl.day.values, 'Label': cl.Label.values, 'keep_D0': True, 'keep_D1': ~D1_dup, 'keep_D2': ~D2_dup})
masks.to_parquet(OUT + '/dedup_masks.parquet', index=False)
classes = sorted(cl.Label.unique()); rows_cls = []; rows_day = []; rows_sum = []
for name, dup in [('D0 (none)', np.zeros(len(cl), bool)), ('D1 (record, ports incl.)', D1_dup), ('D2 (model view, ports excl.)', D2_dup)]:
    keep = ~dup; kept = cl[keep]; rho, nfri = rho_of(kept)
    rows_sum.append(dict(condition=name, removed=int(dup.sum()), share_removed=dup.mean(), kept=int(keep.sum()), friday_flows=nfri, friday_attack=int((kept[kept.day == 'friday'].Label != 'BENIGN').sum()), rho=rho, ceiling=1 - rho))
    for d_ in EXPECT_DAY:
        m = (cl.day == d_).values; rows_day.append(dict(condition=name, day=d_, flows=int(m.sum()), removed=int((dup & m).sum()), share_removed=float((dup & m).sum() / m.sum()), kept=int((keep & m).sum())))
    for c in classes:
        mc = (cl.Label == c).values; mf = mc & (cl.day == 'friday').values
        rows_cls.append(dict(condition=name, label=c, all_before=int(mc.sum()), all_after=int((mc & keep).sum()), friday_before=int(mf.sum()), friday_after=int((mf & keep).sum())))
S, DY, CL = pd.DataFrame(rows_sum), pd.DataFrame(rows_day), pd.DataFrame(rows_cls)
S.to_csv(OUT + '/step0_mask_summary.csv', index=False); DY.to_csv(OUT + '/step0_mask_by_day.csv', index=False); CL.to_csv(OUT + '/step0_mask_by_class.csv', index=False)
print('\n### Step 0c: duplicate masks'); print('| condition | rows removed | share | kept | Friday flows | Friday attack | rho | ceiling 1-rho |'); print('|---|---|---|---|---|---|---|---|')
for _, r_ in S.iterrows(): print('| %s | %d | %.2f%% | %d | %d | %d | %.4f | %.4f |' % (r_.condition, r_.removed, 100 * r_.share_removed, r_.kept, r_.friday_flows, r_.friday_attack, r_.rho, r_.ceiling))
print('\nShare removed per day:'); print('| condition | Mon | Tue | Wed | Thu | Fri |'); print('|---|---|---|---|---|---|')
for name in S.condition:
    g = DY[DY.condition == name].set_index('day'); print('| %s | %s |' % (name, ' | '.join('%.1f%%' % (100 * g.loc[d_, 'share_removed']) for d_ in EXPECT_DAY)))
print('\nPer-class counts, all days (before -> D1 -> D2) and Friday (before -> D1 -> D2):'); print('| class | all before | all D1 | all D2 | Friday before | Friday D1 | Friday D2 |'); print('|---|---|---|---|---|---|---|')
for c in classes:
    g = CL[CL.label == c].set_index('condition'); k = list(g.index)
    print('| %s | %d | %d | %d | %d | %d | %d |' % (c, g.loc[k[0], 'all_before'], g.loc[k[1], 'all_after'], g.loc[k[2], 'all_after'], g.loc[k[0], 'friday_before'], g.loc[k[1], 'friday_after'], g.loc[k[2], 'friday_after']))
d1 = S[S.condition.str.startswith('D1')].iloc[0]; d2 = S[S.condition.str.startswith('D2')].iloc[0]
print('\nComparison with E1 / v9:')
print('D1 removed %d vs E1 %d -> %s | D1 Friday flows %d vs %d -> %s | D1 rho %.4f vs %.4f -> %s' % (d1.removed, E1['D1 removed'], 'MATCH' if d1.removed == E1['D1 removed'] else 'DIFFERS', d1.friday_flows, E1['D1 Friday flows'], 'MATCH' if d1.friday_flows == E1['D1 Friday flows'] else 'DIFFERS', d1.rho, E1['D1 rho'], 'MATCH' if round(d1.rho, 4) == E1['D1 rho'] else 'DIFFERS'))
print('D2 removed %d (E1 float64 variant B: %d; v9 mixed-dtype hash: %d) -> exact new number for v10 = %d' % (d2.removed, E1['D2 removed (E1 float64, ports excl)'], E1['v9 ports-excl removed (mixed dtype)'], d2.removed))
# ---------------- (d) downcast to float32 for modelling and save ----------------
cl[FEATS] = cl[FEATS].astype('float32')
cl[['rid', 'src_file', 'src_row', 'day', 'Label'] + FEATS].to_parquet(OUT + '/cleaned_float32.parquet', index=False)
json.dump(FEATS, open(OUT + '/feature_columns.json', 'w'))
print('\nSaved to', OUT, ':', sorted(os.listdir(OUT)))
print('Cleaned float32 frame saved for Steps 1-3 (%d rows, %d feature columns). REMINDER: disconnect the L4 runtime when you are finished with this step (Runtime > Disconnect and delete runtime).' % (len(cl), len(FEATS)))


pip pins: ok
installed (disk): {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3', 'xgboost': '3.3.0'}
loaded (this process): {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3', 'xgboost': '3.3.0'}
environment.txt written; python 3.13.16 | xgboost used: 3.3.0 (v9 states 3.3.0)

### Step 0b: loading and cleaning (58s)
| file | day | raw rows | header rows | NaN/inf rows dropped | kept |
|---|---|---|---|---|---|
| Monday.csv | monday | 529918 | 0 | 437 | 529481 |
| Tuesday.csv | tuesday | 445909 | 0 | 264 | 445645 |
| Wednesday.csv | wednesday | 692703 | 0 | 1297 | 691406 |
| Thursday.csv | thursday | 288602 | 0 | 207 | 288395 |
| Thursday+1.csv | thursday | 170366 | 0 | 135 | 170231 |
| Friday.csv | friday | 191033 | 0 | 122 | 190911 |
| Friday+1.csv | friday | 286467 | 0 | 371 | 286096 |
| Friday+2.csv | friday | 225745 | 0 | 34 | 225711 |
numeric feature columns: 78 (incl. Destination Port); first

In [ ]:
# ===== STEP 2 (final run v10): Table 20 - multiclass temporal ceiling; 3 tree models x 10 seeds x 3 conditions = 90 runs =====
import os, sys, json, time, gc, warnings
warnings.filterwarnings('ignore')
# ---------------- CONFIG ----------------
OUT = '/content/drive/MyDrive/UNSW-NB15/final_run_v10'
CONDITIONS = ['D0', 'D1', 'D2']
SEEDS = list(range(42, 52))
MODELS = ['RandomForest', 'LightGBM', 'XGBoost']
N_TRAIN, N_TEST = 199_999, 99_999
TRAIN_DAYS = ['monday', 'tuesday', 'wednesday', 'thursday']
PINS = {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3', 'xgboost': '3.3.0'}
RUNS_CSV, PRED_DIR = OUT + '/step2_runs.csv', OUT + '/step2_preds'
V9_T20 = {'RandomForest': .5726, 'LightGBM': .5888, 'XGBoost': .5886}; V9_CEIL = .589
E1D = {'RandomForest': .6249, 'LightGBM': .6295, 'XGBoost': .6282}; E1D_CEIL = .6297
OLD_D2 = {'RandomForest': .7155, 'LightGBM': .7222, 'XGBoost': .7210}; OLD_D2_CEIL = .7225
import numpy as np, pandas as pd, sklearn, lightgbm, imblearn, scipy, xgboost
loaded = {'scikit-learn': sklearn.__version__, 'lightgbm': lightgbm.__version__, 'imbalanced-learn': imblearn.__version__, 'scipy': scipy.__version__, 'xgboost': xgboost.__version__}
assert loaded == PINS, ('ENVIRONMENT MISMATCH - restart session and run Step 0 again first', loaded, PINS)
print('environment ok:', loaded, '| pandas', pd.__version__, '| python', sys.version.split()[0])
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
os.makedirs(PRED_DIR, exist_ok=True)
D = pd.read_parquet(OUT + '/cleaned_float32.parquet'); M = pd.read_parquet(OUT + '/dedup_masks.parquet')
assert (D.rid.values == M.rid.values).all(); FEATS = json.load(open(OUT + '/feature_columns.json'))
FE_MODEL = [c for c in FEATS if c != 'Destination Port']
X_ALL = D[FE_MODEL].values; LAB = D.Label.values; DAY = D.day.values; RID = D.rid.values; del D; gc.collect()
print('data:', X_ALL.shape, '| model features:', len(FE_MODEL), '(Destination Port excluded)')
def prop_sample(pos, N, seed):
    lab = LAB[pos]; n_tot = len(pos); out = []
    for c in np.unique(lab):
        pg = pos[lab == c]; k = (len(pg) * N) // n_tot
        if k > 0: out.append(pg[np.random.RandomState(seed).choice(len(pg), size=k, replace=False)])
    return np.concatenate(out)
def make_mc(name, seed):
    if name == 'RandomForest': return RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', criterion='gini', n_jobs=-1, random_state=seed)
    if name == 'LightGBM': return LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1, deterministic=True, force_row_wise=True)
    return XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, eval_metric='mlogloss', verbosity=0)
done = set()
if os.path.exists(RUNS_CSV):
    r0 = pd.read_csv(RUNS_CSV); done = set(zip(r0.cond, r0.seed, r0.model))
t0 = time.time(); class_lists = {}
for cond in CONDITIONS:
    keep = M['keep_' + cond].values; n_fri_pop = int((keep & (DAY == 'friday')).sum())
    trc_pop = set(LAB[keep & np.isin(DAY, TRAIN_DAYS)]); rho_pop = float(1 - np.isin(LAB[keep & (DAY == 'friday')], list(trc_pop)).mean())
    for seed in SEEDS:
        if all((cond, seed, m) in done for m in MODELS): continue
        tr = prop_sample(np.where(keep & np.isin(DAY, TRAIN_DAYS))[0], N_TRAIN, seed); te = prop_sample(np.where(keep & (DAY == 'friday'))[0], N_TEST, seed)
        Xtr, Xte = X_ALL[tr].astype('float64'), X_ALL[te].astype('float64')
        kv = np.where(Xtr.var(axis=0) >= 0.01)[0]; Xtr, Xte = Xtr[:, kv], Xte[:, kv]
        C = np.abs(np.corrcoef(Xtr, rowvar=False)); drop = ((C >= 0.95) & np.triu(np.ones(C.shape, bool), k=1)).any(axis=0); kc = np.where(~drop)[0]; Xtr, Xte = Xtr[:, kc], Xte[:, kc]
        sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
        le = LabelEncoder().fit(LAB[tr]); ytr = le.transform(LAB[tr]); true_lab = LAB[te]; known = np.isin(true_lab, le.classes_)
        yte = np.where(known, le.transform(np.where(known, true_lab, le.classes_[0])), -1); rho = float(1 - known.mean()); class_lists[(cond, seed)] = list(le.classes_)
        rows = []; pred_df = pd.DataFrame({'rid': RID[te], 'label': true_lab})
        for name in MODELS:
            m = make_mc(name, seed).fit(Ztr, ytr); p = m.predict(Zte); pred_df[name] = le.classes_[p]
            acc = float((p == yte).mean()); acc_known = float((p[known] == yte[known]).mean())
            conf = {t: {k: int(v) for k, v in pd.Series(le.classes_[p[true_lab == t]]).value_counts().items()} for t in np.unique(true_lab)}
            rows.append(dict(cond=cond, seed=seed, model=name, rho_sample=rho, rho_population=rho_pop, ceiling=1 - rho, accuracy=acc, known_class_accuracy=acc_known, slack=1 - rho - acc, above_ceiling=bool(acc > 1 - rho + 1e-12),
                             n_classes_train=len(le.classes_), n_features=len(kv[kc]), n_train=len(tr), n_test=len(te), friday_flows_population=n_fri_pop, classes_train=json.dumps(list(le.classes_)), friday_assignments=json.dumps(conf)))
        pd.DataFrame(rows).to_csv(RUNS_CSV, mode='a', header=not os.path.exists(RUNS_CSV), index=False); pred_df.to_parquet('%s/%s_s%d.parquet' % (PRED_DIR, cond, seed), index=False)
        for r in rows: done.add((cond, seed, r['model']))
        print(cond, 'seed', seed, '| classes in training sample', len(le.classes_), '| rho(sample) %.4f' % rho, '| acc', {r['model'][:3]: round(r['accuracy'], 4) for r in rows}, '| %.0fs' % (time.time() - t0)); del Ztr, Zte; gc.collect()
R = pd.read_csv(RUNS_CSV)
for cond in [c for c in CONDITIONS if c in set(R.cond)]:
    G = R[R.cond == cond]; print('\n## Table 20, condition %s: runs completed %d of 30' % (cond, len(G)))
    cl = json.loads(G[G.seed == 42].classes_train.iloc[0]) if (G.seed == 42).any() else []
    print('Training classes, seed 42 (%d): %s' % (len(cl), cl)); print('Number of training classes by seed: %s' % dict(G[G.model == 'RandomForest'].set_index('seed').n_classes_train))
    allc = [set(json.loads(x)) for x in G[G.model == 'RandomForest'].classes_train]; print('Classes not present in every seed: %s' % sorted(set.union(*allc) - set.intersection(*allc)))
    print('Population rho (Step 0 definition) %.4f, ceiling %.4f | sample rho mean %.4f (range %.4f-%.4f)' % (G.rho_population.iloc[0], 1 - G.rho_population.iloc[0], G.rho_sample.mean(), G.rho_sample.min(), G.rho_sample.max()))
    print('| model | mean accuracy (sd) | known-class accuracy | ceiling 1-rho (sample) | slack (mean, min) | runs above ceiling | v9 / reference accuracy |'); print('|---|---|---|---|---|---|---|')
    ref = {'D0': V9_T20, 'D1': E1D, 'D2': OLD_D2}[cond]; refc = {'D0': V9_CEIL, 'D1': E1D_CEIL, 'D2': OLD_D2_CEIL}[cond]
    for mn in MODELS:
        g = G[G.model == mn]; print('| %s | %.4f (%.4f) | %.4f | %.4f | %.4f, %.4f | %d of %d | %.4f (ceiling %.4f) |' % (mn, g.accuracy.mean(), g.accuracy.std(), g.known_class_accuracy.mean(), g.ceiling.mean(), g.slack.mean(), g.slack.min(), g.above_ceiling.sum(), len(g), ref[mn], refc))
    print('Friday confusion summary (flows pooled over seeds; share of each true class assigned to each training class; top 4):')
    for mn in MODELS:
        agg = {}
        for s_ in G[G.model == mn].friday_assignments:
            for t, d in json.loads(s_).items():
                for k, v in d.items(): agg.setdefault(t, {}); agg[t][k] = agg[t].get(k, 0) + v
        parts = []
        for t in ['BENIGN', 'Bot', 'DDoS', 'PortScan']:
            if t in agg:
                tot = sum(agg[t].values()); top = sorted(agg[t].items(), key=lambda kv: -kv[1])[:4]; parts.append('%s -> %s' % (t, ', '.join('%s %.1f%%' % (k, 100 * v / tot) for k, v in top)))
        print('  %s: %s' % (mn, ' | '.join(parts)))
print('\nFiles in', OUT, ':', sorted(os.listdir(OUT)))
print('REMINDER: disconnect the L4 runtime when this step is finished (Runtime > Disconnect and delete runtime).')


environment ok: {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3', 'xgboost': '3.3.0'} | pandas 2.2.3 | python 3.13.16
data: (2827876, 77) | model features: 77 (Destination Port excluded)
D0 seed 42 | classes in training sample 12 | rho(sample) 0.4110 | acc {'Ran': 0.5689, 'Lig': 0.5889, 'XGB': 0.5887} | 141s
D0 seed 43 | classes in training sample 12 | rho(sample) 0.4110 | acc {'Ran': 0.5758, 'Lig': 0.5881, 'XGB': 0.5889} | 169s
D0 seed 44 | classes in training sample 12 | rho(sample) 0.4110 | acc {'Ran': 0.5699, 'Lig': 0.5889, 'XGB': 0.5888} | 197s
D0 seed 45 | classes in training sample 12 | rho(sample) 0.4110 | acc {'Ran': 0.5738, 'Lig': 0.5888, 'XGB': 0.5889} | 225s
D0 seed 46 | classes in training sample 12 | rho(sample) 0.4110 | acc {'Ran': 0.575, 'Lig': 0.5888, 'XGB': 0.5876} | 255s
D0 seed 47 | classes in training sample 12 | rho(sample) 0.4110 | acc {'Ran': 0.5664, 'Lig': 0.5889, 'XGB': 0.5888} | 284s
D0 seed 48 | classes in traini

In [ ]:
# ===== STEP 3 (final run v10): Table 18 capacity sweep - D0 only, seeds 42-46, random and temporal protocols; 16 configs x 5 seeds x 2 protocols = 160 runs =====
import os, sys, json, time, gc, warnings
warnings.filterwarnings('ignore')
# ---------------- CONFIG ----------------
OUT = '/content/drive/MyDrive/UNSW-NB15/final_run_v10'
COND = 'D0'
SEEDS = list(range(42, 47))
N_TRAIN, N_TEST, N_POOL = 199_999, 99_999, 199_999
TRAIN_DAYS = ['monday', 'tuesday', 'wednesday', 'thursday']
PINS = {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3', 'xgboost': '3.3.0'}
RUNS_CSV, SCORE_DIR = OUT + '/step3_runs.csv', OUT + '/step3_scores'
STEP1_CSV = OUT + '/step1_runs.csv'
SWEEP = [('RandomForest', 50, 10, 0), ('RandomForest', 100, 10, 1), ('RandomForest', 200, 10, 0), ('RandomForest', 100, 6, 0), ('RandomForest', 100, 14, 0), ('RandomForest', 100, None, 0),
         ('LightGBM', 50, 8, 0), ('LightGBM', 100, 8, 1), ('LightGBM', 200, 8, 0), ('LightGBM', 100, 4, 0), ('LightGBM', 100, 12, 0),
         ('XGBoost', 50, 6, 0), ('XGBoost', 100, 6, 1), ('XGBoost', 200, 6, 0), ('XGBoost', 100, 3, 0), ('XGBoost', 100, 9, 0)]
V9 = {('RandomForest', 50, 10): (55.50, 54.32, 56.69), ('RandomForest', 100, 10): (56.19, 54.58, 57.80), ('RandomForest', 200, 10): (56.23, 54.58, 57.88), ('RandomForest', 100, 6): (35.25, 32.55, 37.95), ('RandomForest', 100, 14): (57.79, 53.66, 61.93), ('RandomForest', 100, None): (57.46, 53.78, 61.14),
      ('LightGBM', 50, 8): (62.38, 62.27, 62.49), ('LightGBM', 100, 8): (61.53, 59.90, 63.16), ('LightGBM', 200, 8): (60.69, 58.45, 62.93), ('LightGBM', 100, 4): (60.10, 57.38, 62.83), ('LightGBM', 100, 12): (58.18, 55.27, 61.10),
      ('XGBoost', 50, 6): (60.31, 56.48, 64.14), ('XGBoost', 100, 6): (59.85, 56.06, 63.65), ('XGBoost', 200, 6): (59.31, 55.78, 62.84), ('XGBoost', 100, 3): (55.25, 46.64, 63.87), ('XGBoost', 100, 9): (58.52, 54.87, 62.16)}
E5 = {('XGBoost', 50, 6): (52.90, 43.58, 62.23), ('XGBoost', 100, 6): (54.53, 50.26, 58.80), ('XGBoost', 200, 6): (55.12, 51.89, 58.35), ('XGBoost', 100, 3): (43.47, 35.73, 51.21), ('XGBoost', 100, 9): (54.14, 53.13, 55.14)}
# ---------------- environment assertions ----------------
import numpy as np, pandas as pd, sklearn, lightgbm, imblearn, scipy, xgboost
loaded = {'scikit-learn': sklearn.__version__, 'lightgbm': lightgbm.__version__, 'imbalanced-learn': imblearn.__version__, 'scipy': scipy.__version__, 'xgboost': xgboost.__version__}
assert loaded == PINS, ('ENVIRONMENT MISMATCH - restart session and run Step 0 again first', loaded, PINS)
print('environment ok:', loaded, '| pandas', pd.__version__, '| python', sys.version.split()[0])
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, matthews_corrcoef, roc_auc_score, average_precision_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
os.makedirs(SCORE_DIR, exist_ok=True)
# ---------------- data and masks (from Step 0) ----------------
D = pd.read_parquet(OUT + '/cleaned_float32.parquet'); M = pd.read_parquet(OUT + '/dedup_masks.parquet')
assert (D.rid.values == M.rid.values).all(); FEATS = json.load(open(OUT + '/feature_columns.json'))
FE_MODEL = [c for c in FEATS if c != 'Destination Port']
X_ALL = D[FE_MODEL].values; LAB = D.Label.values; DAY = D.day.values; RID = D.rid.values; del D; gc.collect()
print('data:', X_ALL.shape, 'model features:', len(FE_MODEL), '(Destination Port excluded)')
# ---------------- pipeline (identical to Step 1) ----------------
def prop_sample(pos, N, seed):
    lab = LAB[pos]; n_tot = len(pos); out = []
    for c in np.unique(lab):
        pg = pos[lab == c]; k = (len(pg) * N) // n_tot
        if k > 0: out.append(pg[np.random.RandomState(seed).choice(len(pg), size=k, replace=False)])
    return np.concatenate(out)
def partitions(keep, proto, seed):
    if proto == 'temporal':
        tr = prop_sample(np.where(keep & np.isin(DAY, TRAIN_DAYS))[0], N_TRAIN, seed); te = prop_sample(np.where(keep & (DAY == 'friday'))[0], N_TEST, seed); return tr, te
    pooled = prop_sample(np.where(keep)[0], N_POOL, seed)
    return train_test_split(pooled, test_size=0.2, stratify=(LAB[pooled] != 'BENIGN'), random_state=seed)
def prepare(tr, te, seed):
    Xtr, Xte = X_ALL[tr].astype('float64'), X_ALL[te].astype('float64')
    ytr, yte = (LAB[tr] != 'BENIGN').astype(int), (LAB[te] != 'BENIGN').astype(int)
    kv = np.where(Xtr.var(axis=0) >= 0.01)[0]; Xtr, Xte = Xtr[:, kv], Xte[:, kv]
    C = np.abs(np.corrcoef(Xtr, rowvar=False)); up = np.triu(np.ones(C.shape, bool), k=1)
    drop = ((C >= 0.95) & up).any(axis=0); kc = np.where(~drop)[0]; Xtr, Xte = Xtr[:, kc], Xte[:, kc]
    names = [FE_MODEL[i] for i in kv[kc]]
    sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte); n_syn = 0
    n_min, n_maj = int(ytr.sum()), int((ytr == 0).sum()); target = max(n_min, min(30000, n_maj)); k = max(1, min(5, n_min - 1))
    if target > n_min:
        Ztr, ytr = SMOTE(sampling_strategy={1: target}, k_neighbors=k, random_state=seed).fit_resample(Ztr, ytr); n_syn = target - n_min
    return Ztr, ytr, Zte, yte, names, n_syn
def make_sweep(name, trees, depth, seed):
    if name == 'RandomForest': return RandomForestClassifier(n_estimators=trees, max_depth=depth, class_weight='balanced', criterion='gini', n_jobs=-1, random_state=seed)
    if name == 'LightGBM': return LGBMClassifier(n_estimators=trees, max_depth=depth, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1, deterministic=True, force_row_wise=True)
    return XGBClassifier(n_estimators=trees, max_depth=depth, subsample=0.8, colsample_bytree=0.8, objective='binary:logistic', random_state=seed, n_jobs=-1, eval_metric='logloss', verbosity=0)
def cid(c): return '%s_%d_%s' % (c[0], c[1], 'none' if c[2] is None else c[2])
def metrics(y, s):
    p = (s > 0.5).astype(int); tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return dict(accuracy=accuracy_score(y, p), f1_macro=f1_score(y, p, average='macro'), recall=recall_score(y, p, zero_division=0), precision=precision_score(y, p, zero_division=0), attack_f1=f1_score(y, p, zero_division=0),
                fpr=fp / max(fp + tn, 1), mcc=matthews_corrcoef(y, p), roc_auc=roc_auc_score(y, s), pr_auc=average_precision_score(y, s), TP=int(tp), FP=int(fp), TN=int(tn), FN=int(fn))
# ---------------- run loop (resumable per seed, protocol, config) ----------------
keep = M['keep_' + COND].values; S1 = pd.read_csv(STEP1_CSV); S1 = S1[S1.cond == COND]
done = set()
if os.path.exists(RUNS_CSV):
    r0 = pd.read_csv(RUNS_CSV); done = set(zip(r0.seed, r0.proto, r0.config))
t0 = time.time(); checks = []; MAXDIFF = {}
for seed in SEEDS:
    for proto in ['random', 'temporal']:
        todo = [c for c in SWEEP if (seed, proto, cid(c)) not in done]
        if not todo: continue
        tr, te = partitions(keep, proto, seed); Ztr, ytr, Zte, yte, names, n_syn = prepare(tr, te, seed); rows = []; b = float((yte == 0).mean())
        sp = SCORE_DIR + '/%s_%s_s%d.parquet' % (COND, proto, seed); sc_df = pd.read_parquet(sp) if os.path.exists(sp) else pd.DataFrame({'rid': RID[te], 'label': LAB[te], 'y_true': yte})
        for c in todo:
            m = make_sweep(c[0], c[1], c[2], seed).fit(Ztr, ytr); s = m.predict_proba(Zte)[:, 1]; sc_df[cid(c)] = s.astype('float32'); mt = metrics(yte, s)
            if c[3]:
                q = S1[(S1.seed == seed) & (S1.proto == proto) & (S1.model == c[0])].iloc[0]
                for k in ['accuracy', 'f1_macro', 'recall', 'fpr', 'roc_auc', 'pr_auc']:
                    dif = abs(mt[k] - q[k]); MAXDIFF[k] = max(MAXDIFF.get(k, 0.0), dif)
                    assert dif < (1e-3 if k == 'pr_auc' else 1e-9), ('DEFAULT ROW DIFFERS FROM STEP 1', c, seed, proto, k, mt[k], q[k])
                checks.append((c[0], seed, proto))
            rows.append(dict(cond=COND, seed=seed, proto=proto, config=cid(c), model=c[0], trees=c[1], depth=-1 if c[2] is None else c[2], is_default=c[3], **mt, n_feat=len(names), smote_added=n_syn, n_train=len(ytr), n_test=len(te), never_alert_f1=b / (1 + b)))
        pd.DataFrame(rows).to_csv(RUNS_CSV, mode='a', header=not os.path.exists(RUNS_CSV), index=False); sc_df.to_parquet(sp, index=False)
        for r in rows: done.add((seed, proto, r['config']))
        print('seed', seed, proto, '| feat', len(names), '| smote', n_syn, '| configs fitted', len(todo), '| %.0fs' % (time.time() - t0)); del Ztr, Zte, sc_df; gc.collect()
print('Default-row assertions passed against Step 1 (1e-9 on accuracy, F1-macro, recall, FPR, ROC-AUC; 1e-3 on PR-AUC (tied RF scores can reorder at 1e-16 and move PR-AUC slightly)): %d checks this session; max abs difference by metric: %s' % (len(checks), {k: float('%.2e' % x) for k, x in MAXDIFF.items()}))
# ---------------- report ----------------
R = pd.read_csv(RUNS_CSV); assert len(R) == 160, ('runs completed', len(R))
def tci(d):
    d = np.asarray(d, float); k = len(d); m_ = d.mean(); se = d.std(ddof=1) / np.sqrt(k); tq = stats.t.ppf(0.975, k - 1); return m_, m_ - tq * se, m_ + tq * se
print('\n## Table 18 (D0, Destination Port excluded; 5 seeds, t 95% CI df 4). Gap = random minus temporal F1-macro, pp.')
print('Never-alert F1-macro on the temporal test set: %.4f. Normalised loss = (F1_random - F1_temporal) / (F1_random - floor).' % R[R.proto == 'temporal'].never_alert_f1.mean())
print('| model | trees | depth | F1 random | F1 temporal | gap pp [95% CI] | normalised loss | recall gap pp | FPR gap pp | v9 gap [CI] | diff vs v9 | E5 aligned [CI] | Step 1 10-seed gap (default only) |'); print('|---|---|---|---|---|---|---|---|---|---|---|---|---|')
rows_out = []
for c in SWEEP:
    g = R[R.config == cid(c)]; a = g[g.proto == 'random'].set_index('seed'); t = g[g.proto == 'temporal'].set_index('seed'); s = a.index.intersection(t.index); assert len(s) == 5
    gap, lo, hi = tci(100 * (a.loc[s].f1_macro - t.loc[s].f1_macro)); rg = tci(100 * (a.loc[s].recall - t.loc[s].recall))[0]; fg = tci(100 * (a.loc[s].fpr - t.loc[s].fpr))[0]
    fl = t.never_alert_f1.mean(); nl = ((a.loc[s].f1_macro - t.loc[s].f1_macro) / (a.loc[s].f1_macro - fl)).mean(); v = V9[(c[0], c[1], c[2])]; e = E5.get((c[0], c[1], c[2]))
    s1g = ''
    if c[3]:
        q = S1[S1.model == c[0]]; qa = q[q.proto == 'random'].set_index('seed').f1_macro; qt = q[q.proto == 'temporal'].set_index('seed').f1_macro; s1g = '%.2f [%.2f, %.2f]' % tci(100 * (qa - qt).loc[qa.index])
    rows_out.append(dict(model=c[0], trees=c[1], depth='none' if c[2] is None else c[2], f1_random=a.f1_macro.mean(), f1_temporal=t.f1_macro.mean(), gap=gap, ci_lo=lo, ci_hi=hi, norm_loss=nl, recall_gap=rg, fpr_gap=fg, floor_temporal=fl, v9_gap=v[0], v9_lo=v[1], v9_hi=v[2], diff_vs_v9=gap - v[0], e5=e[0] if e else np.nan))
    print('| %s | %d | %s%s | %.4f | %.4f | %.2f [%.2f, %.2f] | %.3f | %.2f | %+.2f | %.2f [%.2f, %.2f] | %+.2f | %s | %s |' % (c[0], c[1], 'none' if c[2] is None else c[2], ' (default)' if c[3] else '', a.f1_macro.mean(), t.f1_macro.mean(), gap, lo, hi, nl, rg, fg, v[0], v[1], v[2], gap - v[0], ('%.2f [%.2f, %.2f]' % e) if e else '-', s1g or '-'))
pd.DataFrame(rows_out).to_csv(OUT + '/step3_table18.csv', index=False)
T = pd.DataFrame(rows_out)
for mn in ['RandomForest', 'LightGBM', 'XGBoost']:
    x = T[T.model == mn]; print('%s: gap range %.2f to %.2f pp; v9 range %.2f to %.2f; largest |diff vs v9| %.2f pp' % (mn, x.gap.min(), x.gap.max(), x.v9_gap.min(), x.v9_gap.max(), x.diff_vs_v9.abs().max()))
print('Configs whose v10 CI excludes the v9 point estimate:', [(r.model, r.trees, r.depth) for r in T.itertuples() if not (r.ci_lo <= r.v9_gap <= r.ci_hi)])
print('\nFiles in', OUT, ':', sorted(os.listdir(OUT)))
print('REMINDER: keep the L4 runtime for the ten-seed port check; disconnect it when that step is finished.')


environment ok: {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3', 'xgboost': '3.3.0'} | pandas 2.2.3 | python 3.13.16
data: (2827876, 77) model features: 77 (Destination Port excluded)
Default-row assertions passed against Step 1 (1e-9 on accuracy, F1-macro, recall, FPR, ROC-AUC; 1e-3 on PR-AUC (tied RF scores can reorder at 1e-16 and move PR-AUC slightly)): 0 checks this session; max abs difference by metric: {}

## Table 18 (D0, Destination Port excluded; 5 seeds, t 95% CI df 4). Gap = random minus temporal F1-macro, pp.
Never-alert F1-macro on the temporal test set: 0.3707. Normalised loss = (F1_random - F1_temporal) / (F1_random - floor).
| model | trees | depth | F1 random | F1 temporal | gap pp [95% CI] | normalised loss | recall gap pp | FPR gap pp | v9 gap [CI] | diff vs v9 | E5 aligned [CI] | Step 1 10-seed gap (default only) |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| RandomForest | 50 | 10 | 0.9883 | 0.4386 | 54.97 

In [ ]:
# ===== PORT CHECK, ten seeds (final run v10): D0, seeds 42-51, RF/LightGBM/XGBoost, random and temporal, Destination Port INCLUDED = 60 fits =====
import os, sys, json, time, gc, warnings
warnings.filterwarnings('ignore')
# ---------------- CONFIG ----------------
OUT = '/content/drive/MyDrive/UNSW-NB15/final_run_v10'
COND = 'D0'
SEEDS = list(range(42, 52))
MODELS = ['RandomForest', 'LightGBM', 'XGBoost']
N_TRAIN, N_TEST, N_POOL = 199_999, 99_999, 199_999
TRAIN_DAYS = ['monday', 'tuesday', 'wednesday', 'thursday']
PINS = {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3', 'xgboost': '3.3.0'}
RUNS_CSV, SCORE_DIR = OUT + '/step4_port_runs.csv', OUT + '/step4_port_scores'
STEP1_CSV, SEED42_PORT_CSV = OUT + '/step1_runs.csv', OUT + '/step1_port_check.csv'
import numpy as np, pandas as pd, sklearn, lightgbm, imblearn, scipy, xgboost
loaded = {'scikit-learn': sklearn.__version__, 'lightgbm': lightgbm.__version__, 'imbalanced-learn': imblearn.__version__, 'scipy': scipy.__version__, 'xgboost': xgboost.__version__}
assert loaded == PINS, ('ENVIRONMENT MISMATCH - restart session and run Step 0 again first', loaded, PINS)
print('environment ok:', loaded, '| pandas', pd.__version__, '| python', sys.version.split()[0])
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, matthews_corrcoef, roc_auc_score, average_precision_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
os.makedirs(SCORE_DIR, exist_ok=True)
D = pd.read_parquet(OUT + '/cleaned_float32.parquet'); M = pd.read_parquet(OUT + '/dedup_masks.parquet')
assert (D.rid.values == M.rid.values).all(); FEATS = json.load(open(OUT + '/feature_columns.json'))
FE_MODEL = ['Destination Port'] + [c for c in FEATS if c != 'Destination Port']                 # Destination Port INCLUDED, first column (same layout as the seed-42 port check)
assert len(FE_MODEL) == 78 and len(set(FE_MODEL)) == 78
X_ALL = D[FE_MODEL].values; LAB = D.Label.values; DAY = D.day.values; RID = D.rid.values; del D; gc.collect()
print('data:', X_ALL.shape, '| model features:', len(FE_MODEL), '(Destination Port INCLUDED)')
def prop_sample(pos, N, seed):
    lab = LAB[pos]; n_tot = len(pos); out = []
    for c in np.unique(lab):
        pg = pos[lab == c]; k = (len(pg) * N) // n_tot
        if k > 0: out.append(pg[np.random.RandomState(seed).choice(len(pg), size=k, replace=False)])
    return np.concatenate(out)
def partitions(keep, proto, seed):
    if proto == 'temporal':
        tr = prop_sample(np.where(keep & np.isin(DAY, TRAIN_DAYS))[0], N_TRAIN, seed); te = prop_sample(np.where(keep & (DAY == 'friday'))[0], N_TEST, seed); return tr, te
    pooled = prop_sample(np.where(keep)[0], N_POOL, seed)
    return train_test_split(pooled, test_size=0.2, stratify=(LAB[pooled] != 'BENIGN'), random_state=seed)
def prepare(tr, te, seed):
    Xtr, Xte = X_ALL[tr].astype('float64'), X_ALL[te].astype('float64')
    ytr, yte = (LAB[tr] != 'BENIGN').astype(int), (LAB[te] != 'BENIGN').astype(int)
    kv = np.where(Xtr.var(axis=0) >= 0.01)[0]; Xtr, Xte = Xtr[:, kv], Xte[:, kv]
    C = np.abs(np.corrcoef(Xtr, rowvar=False)); up = np.triu(np.ones(C.shape, bool), k=1)
    drop = ((C >= 0.95) & up).any(axis=0); kc = np.where(~drop)[0]; Xtr, Xte = Xtr[:, kc], Xte[:, kc]
    names = [FE_MODEL[i] for i in kv[kc]]
    sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte); n_syn = 0
    n_min, n_maj = int(ytr.sum()), int((ytr == 0).sum()); target = max(n_min, min(30000, n_maj)); k = max(1, min(5, n_min - 1))
    if target > n_min:
        Ztr, ytr = SMOTE(sampling_strategy={1: target}, k_neighbors=k, random_state=seed).fit_resample(Ztr, ytr); n_syn = target - n_min
    return Ztr, ytr, Zte, yte, names, n_syn
def make_model(name, seed):
    if name == 'RandomForest': return RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', criterion='gini', n_jobs=-1, random_state=seed)
    if name == 'LightGBM': return LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, colsample_bytree=0.8, subsample=0.8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1, deterministic=True, force_row_wise=True)
    return XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, objective='binary:logistic', random_state=seed, n_jobs=-1, eval_metric='logloss', verbosity=0)
def metrics(y, s):
    p = (s > 0.5).astype(int); tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return dict(accuracy=accuracy_score(y, p), f1_macro=f1_score(y, p, average='macro'), recall=recall_score(y, p, zero_division=0), precision=precision_score(y, p, zero_division=0), attack_f1=f1_score(y, p, zero_division=0),
                fpr=fp / max(fp + tn, 1), mcc=matthews_corrcoef(y, p), roc_auc=roc_auc_score(y, s), pr_auc=average_precision_score(y, s), TP=int(tp), FP=int(fp), TN=int(tn), FN=int(fn))
def xgb_info(m):
    try:
        cfg = json.loads(m.get_booster().save_config()); return cfg['learner']['gradient_booster']['gbtree_train_param']['tree_method'], str(cfg['learner']['learner_model_param']['base_score'])
    except Exception as e: return 'n/a', 'n/a'
keep = M['keep_' + COND].values
done = set()
if os.path.exists(RUNS_CSV):
    r0 = pd.read_csv(RUNS_CSV); done = set(zip(r0.seed, r0.proto, r0.model))
t0 = time.time()
for seed in SEEDS:
    for proto in ['random', 'temporal']:
        todo = [m for m in MODELS if (seed, proto, m) not in done]
        if not todo: continue
        tr, te = partitions(keep, proto, seed); Ztr, ytr, Zte, yte, names, n_syn = prepare(tr, te, seed); rows = []; b = float((yte == 0).mean()); sc_df = pd.DataFrame({'rid': RID[te], 'label': LAB[te], 'y_true': yte})
        for mn in MODELS:
            m = make_model(mn, seed).fit(Ztr, ytr); s = m.predict_proba(Zte)[:, 1]; sc_df[mn] = s.astype('float32')
            if mn not in todo: continue
            rows.append(dict(cond=COND, seed=seed, proto=proto, model=mn, port_included=1, port_retained=int('Destination Port' in names), **metrics(yte, s), n_feat=len(names), smote_added=n_syn, never_alert_f1=b / (1 + b),
                             xgb_tree_method=xgb_info(m)[0] if mn == 'XGBoost' else '', xgb_base_score=xgb_info(m)[1] if mn == 'XGBoost' else ''))
        pd.DataFrame(rows).to_csv(RUNS_CSV, mode='a', header=not os.path.exists(RUNS_CSV), index=False); sc_df.to_parquet('%s/%s_%s_s%d.parquet' % (SCORE_DIR, COND, proto, seed), index=False)
        for r in rows: done.add((seed, proto, r['model']))
        print('seed', seed, proto, '| feat', len(names), '| port retained', 'Destination Port' in names, '| smote', n_syn, '| AUC', {r['model'][:3]: round(r['roc_auc'], 4) for r in rows}, '| %.0fs' % (time.time() - t0)); del Ztr, Zte, sc_df; gc.collect()
# ---------------- report ----------------
P = pd.read_csv(RUNS_CSV); assert len(P) == 60, ('runs completed', len(P))
S1 = pd.read_csv(STEP1_CSV); S1 = S1[(S1.cond == COND) & S1.model.isin(MODELS)]; assert len(S1) == 60
q = pd.read_csv(SEED42_PORT_CSV); print('Seed-42 consistency with the earlier port check file (columns:', list(q.columns)[:8], '...)')
try:
    for _, r in q.iterrows():
        mm = P[(P.seed == 42) & (P.proto == r['proto']) & (P.model == r['model'])].iloc[0]
        for k in ['f1_macro', 'recall', 'roc_auc']:
            if k in q.columns: assert abs(mm[k] - r[k]) < 1e-6, ('seed 42 port check differs from earlier file', r['model'], r['proto'], k, mm[k], r[k])
    print('seed-42 rows agree with step1_port_check.csv (1e-6)')
except KeyError as e: print('seed-42 comparison skipped (column layout differs):', e)
def tci(d):
    d = np.asarray(d, float); k = len(d); m_ = d.mean(); se = d.std(ddof=1) / np.sqrt(k); tq = stats.t.ppf(0.975, k - 1); return m_, m_ - tq * se, m_ + tq * se
def wil(d):
    d = np.asarray(d, float); return stats.wilcoxon(d, method='exact', alternative='two-sided').pvalue if (d != 0).any() else 1.0
fl = P[P.proto == 'temporal'].never_alert_f1.mean(); print('\n## Ten-seed port check (D0, seeds 42-51). Never-alert F1-macro floor on the temporal test set: %.4f (identical in every seed: %s)' % (fl, P[P.proto == 'temporal'].never_alert_f1.nunique() == 1))
print('Port retained by the variance/correlation filters in %d of %d fits; retained features: random %s | temporal %s' % (P.port_retained.sum(), len(P), sorted(P[P.proto == 'random'].n_feat.unique()), sorted(P[P.proto == 'temporal'].n_feat.unique())))
rows_out = []
for met in ['roc_auc', 'f1_macro', 'recall']:
    print('\n### %s: random and temporal means, gap pp [95%% t-CI, df 9], port INCLUDED vs port EXCLUDED (Step 1, same seeds)' % met)
    print('| model | port | random | temporal | gap pp [95% CI] | normalised loss (F1 only) | temporal sd |'); print('|---|---|---|---|---|---|---|')
    for mn in MODELS:
        for lab_, DF in [('included', P), ('excluded', S1)]:
            g = DF[DF.model == mn]; a = g[g.proto == 'random'].set_index('seed')[met]; t = g[g.proto == 'temporal'].set_index('seed')[met]; s = a.index.intersection(t.index); gap, lo, hi = tci(100 * (a.loc[s] - t.loc[s]))
            nl = ((a.loc[s] - t.loc[s]) / (a.loc[s] - fl)).mean() if met == 'f1_macro' else np.nan
            rows_out.append(dict(metric=met, model=mn, port=lab_, random=a.mean(), temporal=t.mean(), gap=gap, lo=lo, hi=hi, norm_loss=nl, temporal_sd=t.std()))
            print('| %s | %s | %.4f | %.4f | %.2f [%.2f, %.2f] | %s | %.4f |' % (mn, lab_, a.mean(), t.mean(), gap, lo, hi, '%.3f' % nl if met == 'f1_macro' else '-', t.std()))
pd.DataFrame(rows_out).to_csv(OUT + '/step4_port_summary.csv', index=False)
print('\n### Paired effect of including the port on TEMPORAL scores (included minus excluded, same seeds; exact Wilcoxon, n=10)')
print('| model | metric | mean difference | 95% CI | seeds with included > excluded | exact p |'); print('|---|---|---|---|---|---|')
for mn in MODELS:
    for met in ['roc_auc', 'f1_macro', 'recall']:
        a = P[(P.model == mn) & (P.proto == 'temporal')].set_index('seed')[met]; b_ = S1[(S1.model == mn) & (S1.proto == 'temporal')].set_index('seed')[met]; d = (a - b_.loc[a.index]).values; m_, lo, hi = tci(d)
        print('| %s | %s | %+.4f | [%+.4f, %+.4f] | %d of 10 | %.5f |' % (mn, met, m_, lo, hi, int((d > 0).sum()), wil(d)))
print('\nTemporal ROC-AUC per seed, port included | excluded:')
for mn in MODELS:
    a = P[(P.model == mn) & (P.proto == 'temporal')].sort_values('seed').roc_auc.round(4).tolist(); b_ = S1[(S1.model == mn) & (S1.proto == 'temporal')].sort_values('seed').roc_auc.round(4).tolist(); print(' ', mn, a, '|', b_)
print('Seeds where the temporal AUC with the port is below 0.5:', {mn: int((P[(P.model == mn) & (P.proto == 'temporal')].roc_auc < 0.5).sum()) for mn in MODELS})
print('\nFiles in', OUT, ':', sorted(os.listdir(OUT)))
print('REMINDER: disconnect the L4 runtime when the port check is finished unless the within-window check follows immediately (Runtime > Disconnect and delete runtime).')


environment ok: {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3', 'xgboost': '3.3.0'} | pandas 2.2.3 | python 3.13.16
data: (2827876, 78) | model features: 78 (Destination Port INCLUDED)
Seed-42 consistency with the earlier port check file (columns: ['cond', 'seed', 'proto', 'model', 'port_included', 'port_retained', 'n_feat', 'smote_added'] ...)
seed-42 rows agree with step1_port_check.csv (1e-6)

## Ten-seed port check (D0, seeds 42-51). Never-alert F1-macro floor on the temporal test set: 0.3707 (identical in every seed: True)
Port retained by the variance/correlation filters in 60 of 60 fits; retained features: random [np.int64(41), np.int64(42), np.int64(43), np.int64(44)] | temporal [np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45)]

### roc_auc: random and temporal means, gap pp [95% t-CI, df 9], port INCLUDED vs port EXCLUDED (Step 1, same seeds)
| model | port | random | temporal | gap pp [95% CI] | normalised l

In [ ]:
# ===== STEP 5a (final run v10): within-window controls (Table 8), duplicates removed WITHIN each window block with pandas float64 duplicated(keep='first') =====
# D1 = features incl. Destination Port + label; D2 = features excl. Destination Port + label; D0 = none (control: should reproduce the as-captured within-window result).
# Reduced pipeline of Section 3.2 as in v9 Table 8: Destination Port NOT a model feature, StandardScaler fit on train, NO variance/correlation filter, NO SMOTE.
# 10 windows x 10 seeds x 2 splits (chronological first 80% vs stratified random 80/20 of the same rows) x 3 models = 600 fits per condition.
import os, sys, gc, time, warnings
warnings.filterwarnings('ignore')
# ---------------- CONFIG ----------------
RAW_DIR = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017'
FILES = ['Monday.csv', 'Tuesday.csv', 'Wednesday.csv', 'Thursday.csv', 'Thursday+1.csv', 'Friday.csv', 'Friday+1.csv', 'Friday+2.csv']
OUT = '/content/drive/MyDrive/UNSW-NB15/final_run_v10'
CONDITIONS = ['D1', 'D2', 'D0']
SEEDS = list(range(42, 52)); MIN_SIDE = 100; CUT = 0.8
MNAMES = ['RandomForest', 'LightGBM', 'XGBoost']
PINS = {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3', 'xgboost': '3.3.0'}
RUNS_CSV, BLOCK_CSV = OUT + '/step5a_window_runs.csv', OUT + '/step5a_block_summary.csv'
EARLIER = {'D0': 'as captured +0.76 [-0.06, +1.58]', 'D1': 'hash-based ports-included dedup (E1e) +0.96 [+0.04, +1.88]', 'D2': 'v9 ports-excluded dedup +1.43 [-0.05, +2.91]'}
import numpy as np, pandas as pd, sklearn, lightgbm, imblearn, scipy, xgboost
loaded = {'scikit-learn': sklearn.__version__, 'lightgbm': lightgbm.__version__, 'imbalanced-learn': imblearn.__version__, 'scipy': scipy.__version__, 'xgboost': xgboost.__version__}
assert loaded == PINS, ('ENVIRONMENT MISMATCH - restart session and run Step 0 again first', loaded, PINS)
print('environment ok:', loaded, '| pandas', pd.__version__, '| python', sys.version.split()[0])
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, confusion_matrix, average_precision_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
t0 = time.time(); DF = {}; FEATS = None
for fn in FILES:
    df = pd.read_csv(os.path.join(RAW_DIR, fn), low_memory=False); df.columns = df.columns.str.strip()
    df = df[df['Label'].astype(str).str.strip() != 'Label'].copy(); df['Label'] = df['Label'].astype(str).str.strip()
    feats = [c for c in df.columns if c != 'Label']
    if FEATS is None: FEATS = feats
    assert feats == FEATS, fn
    X = df[feats].apply(pd.to_numeric, errors='coerce').astype('float64').replace([np.inf, -np.inf], np.nan); ok = X.notna().all(axis=1).values
    X['Label'] = df['Label'].values; DF[fn] = X[ok].reset_index(drop=True); del df, X; gc.collect()
print('cleaned rows: %d (Step 0: 2,827,876) | features %d | %.0fs' % (sum(len(v) for v in DF.values()), len(FEATS), time.time() - t0)); assert sum(len(v) for v in DF.values()) == 2_827_876
FE_MODEL = [c for c in FEATS if c != 'Destination Port']; assert len(FE_MODEL) == 77
WINDOWS = {'SSH-Patator': ('Tuesday.csv', 'SSH-Patator'), 'DoS Hulk': ('Wednesday.csv', 'DoS Hulk'), 'Bot': ('Friday.csv', 'Bot'), 'PortScan': ('Friday+1.csv', 'PortScan'), 'DDoS': ('Friday+2.csv', 'DDoS')}
KEYS = {'GoldenEye': 'goldeneye', 'Slowloris': 'slowloris', 'Slowhttptest': 'slowhttp', 'WebBruteForce': 'brute force', 'WebXSS': 'xss'}
for fn in ['Wednesday.csv', 'Thursday.csv', 'Thursday+1.csv']:
    vc = DF[fn]['Label'].value_counts()
    for name, key in KEYS.items():
        for l in vc.index:
            if key in str(l).lower() and name not in WINDOWS: WINDOWS[name] = (fn, l)
print('windows:', {k: v[1] for k, v in WINDOWS.items()}); assert len(WINDOWS) == 10
BLK = {}
for cls, (fn, lab) in WINDOWS.items():
    d = DF[fn]; pos = np.where(d['Label'].values == lab)[0]; w = d.iloc[pos[0]:pos[-1] + 1]; BLK[cls] = w[w['Label'].isin(['BENIGN', lab])].reset_index(drop=True)
del DF; gc.collect()
def make_models(rs):
    return {'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=rs, n_jobs=-1),
            'LightGBM': LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=rs, n_jobs=-1, verbose=-1, deterministic=True, force_row_wise=True),
            'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=rs, n_jobs=-1, verbosity=0)}
def ev(m, Xte, yte):
    p = m.predict(Xte); f1m = f1_score(yte, p, average='macro', zero_division=0); am = (yte == 1); rec = (p[am] == 1).mean() if am.any() else float('nan')
    cm = confusion_matrix(yte, p, labels=[0, 1]); TN, FP = cm[0, 0], cm[0, 1]; fpr = FP / (FP + TN) if (FP + TN) else 0.0
    try: pr = average_precision_score(am.astype(int), m.predict_proba(Xte)[:, 1])
    except Exception: pr = float('nan')
    return f1m, rec, fpr, pr
def dedup_keep(blk, cond):
    if cond == 'D0': return np.ones(len(blk), bool)
    cols = FEATS if cond == 'D1' else FE_MODEL
    return ~blk[cols + ['Label']].duplicated(keep='first').values
done = set()
if os.path.exists(RUNS_CSV):
    r0 = pd.read_csv(RUNS_CSV); done = set(zip(r0.cond, r0.window, r0.seed, r0.split, r0.model))
print('resuming, rows already done:', len(done)); t00 = time.time(); blocks = []; skips = []
for cond in CONDITIONS:
    for cls in WINDOWS:
        blk = BLK[cls]; keep = dedup_keep(blk, cond); b = blk[keep].reset_index(drop=True); y = (b['Label'].values != 'BENIGN').astype(int); n = len(y); cut = int(n * CUT)
        a_tr, a_te = int(y[:cut].sum()), int(y[cut:].sum()); X = b[FE_MODEL]; bn = float((y[cut:] == 0).mean()); floor = bn / (1 + bn)
        d2 = ~blk[FE_MODEL + ['Label']].duplicated(keep='first').values; d1 = ~blk[FEATS + ['Label']].duplicated(keep='first').values
        blocks.append(dict(cond=cond, window=cls, block_rows=len(blk), kept=n, removed_share=1 - n / len(blk), attack_train=a_tr, attack_test=a_te, never_alert_f1_test=floor, d1_share=1 - d1.mean(), d2_share=1 - d2.mean()))
        print('\nWINDOW %s [%s] | block rows %d -> kept %d (removed %.1f%%) | chron attack train/test %d/%d | never-alert floor on chron test %.4f' % (cls, cond, len(blk), n, 100 * (1 - n / len(blk)), a_tr, a_te, floor))
        if a_tr < MIN_SIDE or a_te < MIN_SIDE: skips.append((cond, cls)); print('  SKIPPED (fewer than %d attack flows on one side)' % MIN_SIDE); continue
        chron_cache = None
        for seed in SEEDS:
            todo = [(sp, m) for sp in ['chron', 'random'] for m in MNAMES if (cond, cls, seed, sp, m) not in done]
            if not todo: continue
            r_tr, r_te = train_test_split(np.arange(n), test_size=0.2, random_state=seed, stratify=y); splits = {'chron': (np.arange(cut), np.arange(cut, n)), 'random': (r_tr, r_te)}; new = []
            for sp, (itr, ite) in splits.items():
                if not any(s == sp for s, _ in todo): continue
                if sp == 'chron' and chron_cache is not None: A = chron_cache
                else:
                    sc = StandardScaler(); A = (sc.fit_transform(X.iloc[itr].values), y[itr], sc.transform(X.iloc[ite].values), y[ite])
                    if sp == 'chron': chron_cache = A
                Xtr_s, ytr_s, Xte_s, yte_s = A; models = make_models(seed)
                for mn in MNAMES:
                    if (cond, cls, seed, sp, mn) in done: continue
                    models[mn].fit(Xtr_s, ytr_s); f1m, rec, fpr, pr = ev(models[mn], Xte_s, yte_s)
                    new.append(dict(cond=cond, window=cls, seed=seed, split=sp, model=mn, f1_macro=f1m, attack_recall=rec, FPR=fpr, PR_AUC=pr, n_train=len(ytr_s), n_test=len(yte_s), att_train=int(ytr_s.sum()), att_test=int(yte_s.sum()), never_alert_f1=(1 - yte_s.mean()) / (2 - yte_s.mean()))); done.add((cond, cls, seed, sp, mn))
                gc.collect()
            if new: pd.DataFrame(new).to_csv(RUNS_CSV, mode='a', header=not os.path.exists(RUNS_CSV), index=False)
        print('    done | elapsed %.0fs' % (time.time() - t00)); del X, b; gc.collect()
pd.DataFrame(blocks).drop_duplicates(['cond', 'window'], keep='last').to_csv(BLOCK_CSV, index=False)
R = pd.read_csv(RUNS_CSV)
def ci(v):
    v = np.asarray(v, float); k = len(v)
    if k < 2: return (float('nan'),) * 3
    m = v.mean(); se = v.std(ddof=1) / np.sqrt(k); t = stats.t.ppf(0.975, k - 1); return m, m - t * se, m + t * se
def holm(ps):
    ps = np.asarray(ps, float); o = np.argsort(ps); m = len(ps); adj = np.empty(m); run = 0
    for k_, i in enumerate(o): run = max(run, (m - k_) * ps[i]); adj[i] = min(1.0, run)
    return adj
for cond in [c for c in CONDITIONS if c in set(R.cond)]:
    G = R[R.cond == cond]; print('\n## Table 8 layout, condition %s: runs completed %d (expected 600 less skipped windows %s). Gap = random minus chronological F1-macro, pp, t 95%% CI df 9.' % (cond, len(G), [w for c_, w in skips if c_ == cond] or 'none'))
    print('| window | RF | LightGBM | XGBoost | window mean | attack train/test | never-alert floor (chron test) | chron F1 (mean of models) |'); print('|---|---|---|---|---|---|---|---|'); wm = {}; cellp = []
    for cls in WINDOWS:
        txt = []; cells = []; gw = G[G.window == cls]
        if not len(gw): continue
        for mn in MNAMES:
            g = gw[gw.model == mn]; a = g[g.split == 'random'].set_index('seed').f1_macro; c_ = g[g.split == 'chron'].set_index('seed').f1_macro; s = a.index.intersection(c_.index)
            if len(s) < 2: txt.append('n/a'); continue
            d = (a.loc[s] - c_.loc[s]) * 100; cells.append(d); m, lo, hi = ci(d); txt.append('%+.2f [%+.2f, %+.2f]' % (m, lo, hi)); cellp.append(stats.wilcoxon(d.values, method='exact').pvalue if (d.values != 0).any() else 1.0)
        wm[cls] = pd.concat(cells, axis=1).mean(axis=1).mean(); ch = gw[gw.split == 'chron']
        print('| %s | %s | %s | %s | %+.2f | %d/%d | %.3f | %.3f |' % (cls, txt[0], txt[1], txt[2], wm[cls], ch.att_train.iloc[0], ch.att_test.iloc[0], ch.never_alert_f1.mean(), ch.f1_macro.mean()))
    m, lo, hi = ci(list(wm.values())); adj = holm(cellp)
    print('WINDOW-LEVEL %s (n=%d windows, t df=%d): %+.2f [%+.2f, %+.2f] | windows with positive mean: %d of %d | cells significant after Holm (%d cells, exact Wilcoxon, alpha .05): %d' % (cond, len(wm), len(wm) - 1, m, lo, hi, sum(v > 0 for v in wm.values()), len(wm), len(cellp), int((adj < 0.05).sum())))
    print('Earlier value for comparison: %s' % EARLIER[cond])
print('\n### Block summary (rows removed by pandas float64 duplicated inside each block)'); B = pd.read_csv(BLOCK_CSV); print('| cond | window | block rows | kept | removed | D1-style dup share | D2-style dup share |'); print('|---|---|---|---|---|---|---|')
for _, r in B.iterrows(): print('| %s | %s | %d | %d | %.1f%% | %.1f%% | %.1f%% |' % (r.cond, r.window, r.block_rows, r.kept, 100 * r.removed_share, 100 * r.d1_share, 100 * r.d2_share))
print('\nFiles in', OUT, ':', sorted(os.listdir(OUT)))
print('REMINDER: keep the runtime for step 5b (cut sweep); disconnect it when that step is finished.')


environment ok: {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3', 'xgboost': '3.3.0'} | pandas 2.2.3 | python 3.13.16
cleaned rows: 2827876 (Step 0: 2,827,876) | features 78 | 38s
windows: {'SSH-Patator': 'SSH-Patator', 'DoS Hulk': 'DoS Hulk', 'Bot': 'Bot', 'PortScan': 'PortScan', 'DDoS': 'DDoS', 'GoldenEye': 'DoS GoldenEye', 'Slowloris': 'DoS slowloris', 'Slowhttptest': 'DoS Slowhttptest', 'WebBruteForce': 'Web Attack � Brute Force', 'WebXSS': 'Web Attack � XSS'}
resuming, rows already done: 0

WINDOW SSH-Patator [D1] | block rows 283677 -> kept 268316 (removed 5.4%) | chron attack train/test 2676/543 | never-alert floor on chron test 0.4975
    done | elapsed 207s

WINDOW DoS Hulk [D1] | block rows 255205 -> kept 197648 (removed 22.6%) | chron attack train/test 146070/26776 | never-alert floor on chron test 0.2439
    done | elapsed 317s

WINDOW Bot [D1] | block rows 166847 -> kept 161171 (removed 3.4%) | chron attack train/test 1591/357 

In [ ]:
# ===== STEP 5b (final run v10): cut-point sweep over the ten within-window controls, duplicates removed WITHIN each block with pandas float64 duplicated(keep='first') =====
# D1 = features incl. Destination Port + label; D2 = features excl. Destination Port + label; D0 = none (control: should reproduce the as-captured within-window result).
# Reduced pipeline of Section 3.2 as in v9 Table 8: Destination Port NOT a model feature, StandardScaler fit on train, NO variance/correlation filter, NO SMOTE.
# 10 windows x 10 seeds x 2 splits (chronological first 80% vs stratified random 80/20 of the same rows) x 3 models = 600 fits per condition.
import os, sys, gc, time, warnings
warnings.filterwarnings('ignore')
# ---------------- CONFIG ----------------
RAW_DIR = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017'
FILES = ['Monday.csv', 'Tuesday.csv', 'Wednesday.csv', 'Thursday.csv', 'Thursday+1.csv', 'Friday.csv', 'Friday+1.csv', 'Friday+2.csv']
OUT = '/content/drive/MyDrive/UNSW-NB15/final_run_v10'
CONDITIONS = ['D1', 'D2', 'D0']
SEEDS = [42, 43, 44]; FS = [0.5, 0.6, 0.7, 0.8, 0.9]; MIN_SIDE = 100
MNAMES = ['RandomForest', 'LightGBM', 'XGBoost']
PINS = {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3', 'xgboost': '3.3.0'}
RUNS_CSV = OUT + '/step5b_cut_sweep_runs.csv'
EARLIER = {'D0': {0.5: '+4.66 [-1.09, 10.40]', 0.6: '+6.94 [-2.89, 16.77]', 0.7: '+2.21 [-1.17, 5.59]', 0.8: '+0.72 [-0.08, 1.53]', 0.9: '+3.43 [-2.65, 9.51]'}, 'D1': {0.5: '+6.68 [-2.91, 16.27]', 0.6: '+8.56 [-2.44, 19.56]', 0.7: '+2.65 [-1.15, 6.46]', 0.8: '+0.85 [-0.10, 1.79]', 0.9: '+4.94 [-3.35, 13.22]'}}
import numpy as np, pandas as pd, sklearn, lightgbm, imblearn, scipy, xgboost
loaded = {'scikit-learn': sklearn.__version__, 'lightgbm': lightgbm.__version__, 'imbalanced-learn': imblearn.__version__, 'scipy': scipy.__version__, 'xgboost': xgboost.__version__}
assert loaded == PINS, ('ENVIRONMENT MISMATCH - restart session and run Step 0 again first', loaded, PINS)
print('environment ok:', loaded, '| pandas', pd.__version__, '| python', sys.version.split()[0])
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, confusion_matrix, average_precision_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
t0 = time.time(); DF = {}; FEATS = None
for fn in FILES:
    df = pd.read_csv(os.path.join(RAW_DIR, fn), low_memory=False); df.columns = df.columns.str.strip()
    df = df[df['Label'].astype(str).str.strip() != 'Label'].copy(); df['Label'] = df['Label'].astype(str).str.strip()
    feats = [c for c in df.columns if c != 'Label']
    if FEATS is None: FEATS = feats
    assert feats == FEATS, fn
    X = df[feats].apply(pd.to_numeric, errors='coerce').astype('float64').replace([np.inf, -np.inf], np.nan); ok = X.notna().all(axis=1).values
    X['Label'] = df['Label'].values; DF[fn] = X[ok].reset_index(drop=True); del df, X; gc.collect()
print('cleaned rows: %d (Step 0: 2,827,876) | features %d | %.0fs' % (sum(len(v) for v in DF.values()), len(FEATS), time.time() - t0)); assert sum(len(v) for v in DF.values()) == 2_827_876
FE_MODEL = [c for c in FEATS if c != 'Destination Port']; assert len(FE_MODEL) == 77
WINDOWS = {'SSH-Patator': ('Tuesday.csv', 'SSH-Patator'), 'DoS Hulk': ('Wednesday.csv', 'DoS Hulk'), 'Bot': ('Friday.csv', 'Bot'), 'PortScan': ('Friday+1.csv', 'PortScan'), 'DDoS': ('Friday+2.csv', 'DDoS')}
KEYS = {'GoldenEye': 'goldeneye', 'Slowloris': 'slowloris', 'Slowhttptest': 'slowhttp', 'WebBruteForce': 'brute force', 'WebXSS': 'xss'}
for fn in ['Wednesday.csv', 'Thursday.csv', 'Thursday+1.csv']:
    vc = DF[fn]['Label'].value_counts()
    for name, key in KEYS.items():
        for l in vc.index:
            if key in str(l).lower() and name not in WINDOWS: WINDOWS[name] = (fn, l)
print('windows:', {k: v[1] for k, v in WINDOWS.items()}); assert len(WINDOWS) == 10
BLK = {}
for cls, (fn, lab) in WINDOWS.items():
    d = DF[fn]; pos = np.where(d['Label'].values == lab)[0]; w = d.iloc[pos[0]:pos[-1] + 1]; BLK[cls] = w[w['Label'].isin(['BENIGN', lab])].reset_index(drop=True)
del DF; gc.collect()
def make_models(rs):
    return {'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=rs, n_jobs=-1),
            'LightGBM': LGBMClassifier(n_estimators=100, max_depth=8, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=rs, n_jobs=-1, verbose=-1, deterministic=True, force_row_wise=True),
            'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=rs, n_jobs=-1, verbosity=0)}
def ev(m, Xte, yte):
    p = m.predict(Xte); f1m = f1_score(yte, p, average='macro', zero_division=0); am = (yte == 1); rec = (p[am] == 1).mean() if am.any() else float('nan')
    cm = confusion_matrix(yte, p, labels=[0, 1]); TN, FP = cm[0, 0], cm[0, 1]; fpr = FP / (FP + TN) if (FP + TN) else 0.0
    try: pr = average_precision_score(am.astype(int), m.predict_proba(Xte)[:, 1])
    except Exception: pr = float('nan')
    return f1m, rec, fpr, pr
def dedup_keep(blk, cond):
    if cond == 'D0': return np.ones(len(blk), bool)
    cols = FEATS if cond == 'D1' else FE_MODEL
    return ~blk[cols + ['Label']].duplicated(keep='first').values
done = set()
if os.path.exists(RUNS_CSV):
    r0 = pd.read_csv(RUNS_CSV); done = set(zip(r0.cond, r0.window, r0.f, r0.seed, r0.split, r0.model))
print('resuming, rows already done:', len(done)); t00 = time.time(); skips = []
for cond in CONDITIONS:
    for cls in WINDOWS:
        blk = BLK[cls]; keep = dedup_keep(blk, cond); b = blk[keep].reset_index(drop=True); y = (b['Label'].values != 'BENIGN').astype(int); n = len(y); X = b[FE_MODEL]
        print('\nWINDOW %s [%s] | block rows %d -> kept %d (removed %.1f%%)' % (cls, cond, len(blk), n, 100 * (1 - n / len(blk))))
        for f in FS:
            c0 = int(n * f); c1 = min(n, c0 + int(n * 0.2)); itr, ite = np.arange(c0), np.arange(c0, c1); a_tr, a_te = int(y[itr].sum()), int(y[ite].sum())
            if a_tr < MIN_SIDE or a_te < MIN_SIDE:
                skips.append((cond, cls, f, a_tr, a_te)); print('  SKIP f=%.1f attack train/test %d/%d' % (f, a_tr, a_te)); continue
            chron_cache = None
            for seed in SEEDS:
                todo = [(sp, m) for sp in ['chron', 'random'] for m in MNAMES if (cond, cls, f, seed, sp, m) not in done]
                if not todo: continue
                union = np.arange(c1); r_tr, r_te = train_test_split(union, train_size=c0, random_state=seed, stratify=y[union]); splits = {'chron': (itr, ite), 'random': (np.sort(r_tr), np.sort(r_te))}; new = []
                for sp, (a_, b_) in splits.items():
                    if not any(s == sp for s, _ in todo): continue
                    if sp == 'chron' and chron_cache is not None: A = chron_cache
                    else:
                        sc = StandardScaler(); A = (sc.fit_transform(X.iloc[a_].values), y[a_], sc.transform(X.iloc[b_].values), y[b_])
                        if sp == 'chron': chron_cache = A
                    Xtr_s, ytr_s, Xte_s, yte_s = A; models = make_models(seed)
                    for mn in MNAMES:
                        if (cond, cls, f, seed, sp, mn) in done: continue
                        models[mn].fit(Xtr_s, ytr_s); f1m, rec, fpr, pr = ev(models[mn], Xte_s, yte_s)
                        new.append(dict(cond=cond, window=cls, f=f, seed=seed, split=sp, model=mn, f1_macro=f1m, attack_recall=rec, FPR=fpr, PR_AUC=pr, n_train=len(ytr_s), n_test=len(yte_s), att_train=int(ytr_s.sum()), att_test=int(yte_s.sum()), never_alert_f1=(1 - yte_s.mean()) / (2 - yte_s.mean()))); done.add((cond, cls, f, seed, sp, mn))
                    gc.collect()
                if new: pd.DataFrame(new).to_csv(RUNS_CSV, mode='a', header=not os.path.exists(RUNS_CSV), index=False)
            print('    %s %s f=%.1f done | elapsed %.0fs' % (cond, cls, f, time.time() - t00))
        del X, b; gc.collect()
R = pd.read_csv(RUNS_CSV)
def ci(v):
    v = np.asarray(v, float); k = len(v)
    if k < 2: return (float('nan'),) * 3
    m = v.mean(); se = v.std(ddof=1) / np.sqrt(k); t = stats.t.ppf(0.975, k - 1); return m, m - t * se, m + t * se
for cond in [c for c in CONDITIONS if c in set(R.cond)]:
    G = R[R.cond == cond]; a = G[G.split == 'random'].set_index(['window', 'f', 'seed', 'model']).f1_macro; c_ = G[G.split == 'chron'].set_index(['window', 'f', 'seed', 'model']).f1_macro
    d = ((a - c_) * 100).dropna().rename('gap').reset_index(); att = G[G.split == 'chron'].groupby(['window', 'f'])[['att_train', 'att_test']].first(); pw = d.groupby(['window', 'f']).gap.mean().unstack('f')
    print('\n## Cut sweep, condition %s: runs completed %d | skipped (cond, window, f, attack train, attack test): %s' % (cond, len(G), [s for s in skips if s[0] == cond] or 'none'))
    print('Mean random-minus-chron F1-macro gap (pp), mean over seeds 42-44 and the three models; attack train/test in brackets'); print('| window | ' + ' | '.join('f=%.1f' % f for f in FS) + ' |'); print('|---|' + '---|' * len(FS))
    for w in pw.index: print('| %s | %s |' % (w, ' | '.join(('%+.2f [%d/%d]' % (pw.loc[w, f], att.loc[(w, f)].att_train, att.loc[(w, f)].att_test)) if (f in pw.columns and not np.isnan(pw.loc[w, f])) else 'skip' for f in FS)))
    print('Per f, mean across windows with window-level 95%% t interval (n windows varies with skips); earlier value in brackets:')
    for f in FS:
        vals = pw[f].dropna().values if f in pw.columns else []; m, lo, hi = ci(vals); print('  f=%.1f: %+.2f [%+.2f, %+.2f] (n=%d windows, %d positive)%s' % (f, m, lo, hi, len(vals), int((np.asarray(vals) > 0).sum()), (' | earlier: ' + EARLIER[cond][f]) if cond in EARLIER else ''))
    top = d.groupby(['window', 'f']).gap.mean().sort_values(ascending=False).head(6); print('  largest cells: ' + '; '.join('%s f=%.1f %+.2f' % (w, f, v) for (w, f), v in top.items()))
print('\nFiles in', OUT, ':', sorted(os.listdir(OUT)))
print('REMINDER: disconnect the L4 runtime when this step is finished (Runtime > Disconnect and delete runtime).')


environment ok: {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3', 'xgboost': '3.3.0'} | pandas 2.2.3 | python 3.13.16
cleaned rows: 2827876 (Step 0: 2,827,876) | features 78 | 40s
windows: {'SSH-Patator': 'SSH-Patator', 'DoS Hulk': 'DoS Hulk', 'Bot': 'Bot', 'PortScan': 'PortScan', 'DDoS': 'DDoS', 'GoldenEye': 'DoS GoldenEye', 'Slowloris': 'DoS slowloris', 'Slowhttptest': 'DoS Slowhttptest', 'WebBruteForce': 'Web Attack � Brute Force', 'WebXSS': 'Web Attack � XSS'}
resuming, rows already done: 0

WINDOW SSH-Patator [D1] | block rows 283677 -> kept 268316 (removed 5.4%)
    D1 SSH-Patator f=0.5 done | elapsed 115s
    D1 SSH-Patator f=0.6 done | elapsed 148s
    D1 SSH-Patator f=0.7 done | elapsed 185s
    D1 SSH-Patator f=0.8 done | elapsed 228s
    D1 SSH-Patator f=0.9 done | elapsed 273s

WINDOW DoS Hulk [D1] | block rows 255205 -> kept 197648 (removed 22.6%)
    D1 DoS Hulk f=0.5 done | elapsed 293s
    D1 DoS Hulk f=0.6 done | elapsed 31

In [ ]:
# ===== STEP 6 (final run v10): Friday proximity analysis - are unseen Friday families nearer to a seen attack than to benign? (no training; matches Table 17) =====
import os, sys, json, time, gc, warnings
warnings.filterwarnings('ignore')
# ---------------- CONFIG ----------------
OUT = '/content/drive/MyDrive/UNSW-NB15/final_run_v10'
CONDITIONS = ['D0', 'D1', 'D2']
SEEDS = list(range(42, 52))
FAMILIES = ['Bot', 'DDoS', 'PortScan']; CONTROL = 'BENIGN'
N_REF, N_QUERY = 20_000, 1_000
N_TRAIN, N_TEST = 199_999, 99_999
TRAIN_DAYS = ['monday', 'tuesday', 'wednesday', 'thursday']
PINS = {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3', 'xgboost': '3.3.0'}
RUNS_CSV, PRED_DIR = OUT + '/step6_proximity_runs.csv', OUT + '/step2_preds'
import numpy as np, pandas as pd, sklearn, lightgbm, imblearn, scipy, xgboost
loaded = {'scikit-learn': sklearn.__version__, 'lightgbm': lightgbm.__version__, 'imbalanced-learn': imblearn.__version__, 'scipy': scipy.__version__, 'xgboost': xgboost.__version__}
assert loaded == PINS, ('ENVIRONMENT MISMATCH - restart session and run Step 0 again first', loaded, PINS)
print('environment ok:', loaded, '| pandas', pd.__version__, '| python', sys.version.split()[0])
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
D = pd.read_parquet(OUT + '/cleaned_float32.parquet'); M = pd.read_parquet(OUT + '/dedup_masks.parquet')
assert (D.rid.values == M.rid.values).all(); FEATS = json.load(open(OUT + '/feature_columns.json'))
FE_MODEL = [c for c in FEATS if c != 'Destination Port']
X_ALL = D[FE_MODEL].values; LAB = D.Label.values; DAY = D.day.values; RID = D.rid.values; del D; gc.collect()
print('data:', X_ALL.shape, '| model features:', len(FE_MODEL), '(Destination Port excluded, as in Step 2)')
def prop_sample(pos, N, seed):
    lab = LAB[pos]; n_tot = len(pos); out = []
    for c in np.unique(lab):
        pg = pos[lab == c]; k = (len(pg) * N) // n_tot
        if k > 0: out.append(pg[np.random.RandomState(seed).choice(len(pg), size=k, replace=False)])
    return np.concatenate(out)
done = set()
if os.path.exists(RUNS_CSV):
    r0 = pd.read_csv(RUNS_CSV); done = set(zip(r0.cond, r0.seed))
t0 = time.time()
for cond in CONDITIONS:
    keep = M['keep_' + cond].values
    for seed in SEEDS:
        if (cond, seed) in done: continue
        rng = np.random.RandomState(1000 + seed)
        tr = prop_sample(np.where(keep & np.isin(DAY, TRAIN_DAYS))[0], N_TRAIN, seed); te = prop_sample(np.where(keep & (DAY == 'friday'))[0], N_TEST, seed)
        pf = pd.read_parquet('%s/%s_s%d.parquet' % (PRED_DIR, cond, seed)); assert (pf.rid.values == RID[te]).all(), ('Friday sample differs from Step 2', cond, seed)     # same flows as Step 2
        Xtr, Xte = X_ALL[tr].astype('float64'), X_ALL[te].astype('float64')                      # Step 2 feature space: variance >= 0.01, earlier-feature correlation filter, standardise on the training sample
        kv = np.where(Xtr.var(axis=0) >= 0.01)[0]; Xtr, Xte = Xtr[:, kv], Xte[:, kv]
        C = np.abs(np.corrcoef(Xtr, rowvar=False)); drop = ((C >= 0.95) & np.triu(np.ones(C.shape, bool), k=1)).any(axis=0); kc = np.where(~drop)[0]; Xtr, Xte = Xtr[:, kc], Xte[:, kc]
        sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte); ltr = LAB[tr]; lte = LAB[te]
        ia = np.where(ltr != 'BENIGN')[0]; ib = np.where(ltr == 'BENIGN')[0]
        ia = rng.choice(ia, size=min(N_REF, len(ia)), replace=False); ib = rng.choice(ib, size=min(N_REF, len(ib)), replace=False)
        nn_a = NearestNeighbors(n_neighbors=1, algorithm='brute', n_jobs=-1).fit(Zte[:0] if False else Ztr[ia]); nn_b = NearestNeighbors(n_neighbors=1, algorithm='brute', n_jobs=-1).fit(Ztr[ib]); fam_ref = ltr[ia]
        rows = []
        for fam in FAMILIES + [CONTROL]:
            qi = np.where(lte == fam)[0]
            if len(qi) == 0: continue
            qi = rng.choice(qi, size=min(N_QUERY, len(qi)), replace=False); da, ja = nn_a.kneighbors(Zte[qi]); db, _ = nn_b.kneighbors(Zte[qi]); da, db, ja = da[:, 0], db[:, 0], ja[:, 0]
            ratio = (db + 1e-9) / (da + 1e-9); nearest = pd.Series(fam_ref[ja]).value_counts().to_dict()
            det = {m: float((pf.loc[qi, m].values != 'BENIGN').mean()) for m in ['RandomForest', 'LightGBM', 'XGBoost']}
            rows.append(dict(cond=cond, seed=seed, family=fam, n_query=len(qi), n_family_in_sample=int((lte == fam).sum()), n_ref_attack=len(ia), n_ref_benign=len(ib), n_features=Zte.shape[1],
                             closer_share=float((da < db).mean()), tie_share=float((da == db).mean()), median_ratio=float(np.median(ratio)), median_d_attack=float(np.median(da)), median_d_benign=float(np.median(db)),
                             nearest_attack_family=json.dumps({k: int(v) for k, v in nearest.items()}), det_RandomForest=det['RandomForest'], det_LightGBM=det['LightGBM'], det_XGBoost=det['XGBoost']))
        pd.DataFrame(rows).to_csv(RUNS_CSV, mode='a', header=not os.path.exists(RUNS_CSV), index=False); done.add((cond, seed))
        print(cond, 'seed', seed, '| features', Zte.shape[1], '| closer', {r['family']: round(r['closer_share'], 3) for r in rows}, '| %.0fs' % (time.time() - t0)); del Ztr, Zte; gc.collect()
# ---------------- report ----------------
R = pd.read_csv(RUNS_CSV); assert len(R.groupby(['cond', 'seed'])) == 30
def ms(v):
    v = np.asarray(v, float); k = len(v); m_ = v.mean(); se = v.std(ddof=1) / np.sqrt(k); tq = stats.t.ppf(0.975, k - 1); return m_, m_ - tq * se, m_ + tq * se
for cond in CONDITIONS:
    G = R[R.cond == cond]; print('\n## Proximity, condition %s: %d seeds; reference up to %d attack and %d benign training flows per seed; query up to %d Friday flows per family (the Step 2 test-sample flows)' % (cond, G.seed.nunique(), N_REF, N_REF, N_QUERY))
    print('| Friday family | queries per seed (mean) | closer to a seen attack than to benign (mean over seeds [95% CI]) | median ratio nearest-benign / nearest-attack (mean of seed medians) | nearest seen attack family (pooled share) | detected as any attack: RF / LightGBM / XGBoost (mean) |'); print('|---|---|---|---|---|---|')
    for fam in FAMILIES + [CONTROL]:
        g = G[G.family == fam]
        if not len(g): continue
        m_, lo, hi = ms(g.closer_share); agg = {}
        for s_ in g.nearest_attack_family:
            for k, v in json.loads(s_).items(): agg[k] = agg.get(k, 0) + v
        tot = sum(agg.values()); top = sorted(agg.items(), key=lambda kv: -kv[1])[:3]
        print('| %s%s | %.0f | %.3f [%.3f, %.3f] | %.2f | %s | %.3f / %.3f / %.3f |' % (fam, ' (control)' if fam == CONTROL else '', g.n_query.mean(), m_, lo, hi, g.median_ratio.mean(), ', '.join('%s %.1f%%' % (k.replace('Web Attack �', 'WebAttack'), 100 * v / tot) for k, v in top), g.det_RandomForest.mean(), g.det_LightGBM.mean(), g.det_XGBoost.mean()))
    print('Seed-to-seed spread of the closer share (sd): ' + '; '.join('%s %.3f' % (f, G[G.family == f].closer_share.std()) for f in FAMILIES))
    print('Median distances (attack | benign), mean of seed medians: ' + '; '.join('%s %.2f | %.2f' % (f, G[G.family == f].median_d_attack.mean(), G[G.family == f].median_d_benign.mean()) for f in FAMILIES))
    print('Seed-level association between closer share and mean detection across the 3 families (descriptive, n=3 families, no correlation reported): ' + '; '.join('%s closer %.3f, detected %.3f' % (f, G[G.family == f].closer_share.mean(), G[G.family == f][['det_RandomForest', 'det_LightGBM', 'det_XGBoost']].values.mean()) for f in FAMILIES))
print('\nFiles in', OUT, ':', sorted(os.listdir(OUT)))
print('REMINDER: disconnect the L4 runtime now (Runtime > Disconnect and delete runtime).')


environment ok: {'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'imbalanced-learn': '0.14.2', 'scipy': '1.16.3', 'xgboost': '3.3.0'} | pandas 2.2.3 | python 3.13.16
data: (2827876, 77) | model features: 77 (Destination Port excluded, as in Step 2)
D0 seed 42 | features 43 | closer {'Bot': 0.0, 'DDoS': 0.604, 'PortScan': 0.489, 'BENIGN': 0.01} | 3s
D0 seed 43 | features 42 | closer {'Bot': 0.0, 'DDoS': 0.591, 'PortScan': 0.097, 'BENIGN': 0.015} | 7s
D0 seed 44 | features 41 | closer {'Bot': 0.0, 'DDoS': 0.611, 'PortScan': 0.118, 'BENIGN': 0.014} | 10s
D0 seed 45 | features 43 | closer {'Bot': 0.0, 'DDoS': 0.597, 'PortScan': 0.478, 'BENIGN': 0.004} | 14s
D0 seed 46 | features 41 | closer {'Bot': 0.0, 'DDoS': 0.642, 'PortScan': 0.468, 'BENIGN': 0.011} | 17s
D0 seed 47 | features 43 | closer {'Bot': 0.0, 'DDoS': 0.549, 'PortScan': 0.474, 'BENIGN': 0.018} | 20s
D0 seed 48 | features 42 | closer {'Bot': 0.0, 'DDoS': 0.616, 'PortScan': 0.451, 'BENIGN': 0.02} | 24s
D0 seed 49 | features 44 | cl

In [ ]:
# ===== E2 (self-contained): benign-only baseline, captured and dedup (duplicates defined WITH Destination Port), 10 seeds, same samples as Table 6 =====
# CONFIG
import os, gc, time, warnings
MERGED = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017/preprocessed/all_days_merged.csv'
OUT = '/content/drive/MyDrive/UNSW-NB15'
RES = OUT + '/cic2017_e2_benign_only_runs.csv'
CTL = OUT + '/cic2017_e2_benign_shift_control.csv'
SEEDS = list(range(42, 52))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, sklearn, lightgbm, xgboost, imblearn, scipy
from scipy import stats
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score, roc_auc_score, average_precision_score
from sklearn.ensemble import IsolationForest
from sklearn.neural_network import MLPRegressor
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
print('sklearn', sklearn.__version__, '| lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| imblearn', imblearn.__version__, '| scipy', scipy.__version__, '| pandas', pd.__version__)
TRAIN_DAYS = ['monday', 'tuesday', 'wednesday', 'thursday']
# ---- load, flag duplicates (identical float64 vector INCLUDING Destination Port + identical label, first kept), then cast as in the paper pipeline ----
cic = pd.read_csv(MERGED); cic.columns = [c.strip() for c in cic.columns]
cic['day'] = cic['day'].str.lower(); cic['Label'] = cic['Label'].str.strip()
FEATS = [c for c in cic.columns if c not in ('Label', 'day', 'Destination Port')]      # model features (ports excluded)
FE_P = FEATS + ['Destination Port']                                                      # duplicate definition (port included)
hh = pd.util.hash_pandas_object(cic[FE_P], index=False).values
dupP = pd.DataFrame({'h': hh, 'y': cic['Label'].values}).duplicated().values
print('removed %d of %d (%.2f%%)' % (dupP.sum(), len(cic), 100 * dupP.mean()))
del hh; gc.collect()
for c in cic.columns:
    if cic[c].dtype == 'float64': cic[c] = cic[c].astype('float32')
cic_full = cic; print('captured rows', len(cic_full), '| duplicate rows (ports in key)', int(dupP.sum()))
def prop_sample(df, n, seed):
    frac = n / len(df); parts = []
    for _, g in df.groupby('Label'):
        k = int(frac * len(g))
        if k > 0: parts.append(g.sample(n=k, random_state=seed))
    return pd.concat(parts)
def make_partitions(proto, seed):
    if proto == 'temporal':
        return prop_sample(cic[cic['day'].isin(TRAIN_DAYS)], 199_999, seed), prop_sample(cic[cic['day'] == 'friday'], 99_999, seed)
    pooled = prop_sample(cic, 199_999, seed)
    return train_test_split(pooled, test_size=0.2, stratify=(pooled['Label'] != 'BENIGN'), random_state=seed)
def prepare(tr, te, smote=True, seed=42):
    Xtr, Xte = tr[FEATS].astype('float64'), te[FEATS].astype('float64')
    vt = VarianceThreshold(0.01).fit(Xtr); cols = Xtr.columns[vt.get_support()]
    Xtr, Xte = Xtr[cols], Xte[cols]
    cm = Xtr.corr().abs(); up = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool))
    drop = [c for c in up.columns if (up[c] > 0.95).any()]; Xtr, Xte = Xtr.drop(columns=drop), Xte.drop(columns=drop)
    sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
    ytr, yte = (tr['Label'] != 'BENIGN').astype(int).values, (te['Label'] != 'BENIGN').astype(int).values
    n_syn = 0
    if smote:
        n_min, n_maj = int(ytr.sum()), int((ytr == 0).sum()); target = max(n_min, min(30000, n_maj)); k = max(1, min(5, n_min - 1))
        if target > n_min:
            Ztr, ytr = SMOTE(sampling_strategy={1: target}, k_neighbors=k, random_state=seed).fit_resample(Ztr, ytr); n_syn = target - n_min
    return Ztr, ytr, Zte, yte, Xtr.shape[1], n_syn
def metrics(yte, p): return dict(f1_macro=f1_score(yte, p, average='macro'), recall=recall_score(yte, p), fpr=((p == 1) & (yte == 0)).sum() / max((yte == 0).sum(), 1))


def score_ae(Zb, Zs, seed):
    ae = MLPRegressor(hidden_layer_sizes=(32, 16, 32), activation='relu', max_iter=30, batch_size=512, random_state=seed, early_stopping=False).fit(Zb, Zb)
    f = lambda Z: ((ae.predict(Z) - Z) ** 2).mean(axis=1)
    return f(Zb), [f(Z) for Z in Zs]
def score_if(Zb, Zs, seed):
    m = IsolationForest(n_estimators=100, random_state=seed, n_jobs=-1).fit(Zb)
    return -m.score_samples(Zb), [-m.score_samples(Z) for Z in Zs]
DETS = {'IsolationForest': score_if, 'Autoencoder': score_ae}
res = pd.read_csv(RES) if os.path.exists(RES) else pd.DataFrame(); ctl = pd.read_csv(CTL) if os.path.exists(CTL) else pd.DataFrame()
done = set(zip(res.variant, res.seed, res.proto, res.model)) if len(res) else set(); t0 = time.time()
for var in ['captured', 'dedup_ports']:
    cic = cic_full if var == 'captured' else cic_full[~dupP].reset_index(drop=True)
    for seed in SEEDS:
        for proto in ['random', 'temporal']:
            todo = [m for m in DETS if (var, seed, proto, m) not in done]
            if not todo: continue
            tr, te = make_partitions(proto, seed); Ztr, ytr, Zte, yte, nf, _ = prepare(tr, te, False, seed)
            Zb = Ztr[ytr == 0]; lab = te['Label'].values; new = []; newc = []
            for name in todo:
                s_tr, (s_te,) = DETS[name](Zb, [Zte], seed); thr = np.percentile(s_tr, 99); p = (s_te > thr).astype(int)
                row = dict(variant=var, seed=seed, proto=proto, model=name, f1_macro=round(f1_score(yte, p, average='macro'), 4), recall=round(recall_score(yte, p), 4),
                           fpr=round(((p == 1) & (yte == 0)).sum() / max((yte == 0).sum(), 1), 4), roc_auc=round(roc_auc_score(yte, s_te), 4), pr_auc=round(average_precision_score(yte, s_te), 4),
                           never_alert_f1=round((yte == 0).mean() / (1 + (yte == 0).mean()), 4), n_benign_train=len(Zb), n_test=len(yte), att_test=int(yte.sum()), n_feat=nf)
                for fam in ['Bot', 'DDoS', 'PortScan']:
                    mk = lab == fam; row['recall_' + fam] = round(float(p[mk].mean()), 4) if mk.any() else float('nan'); row['n_' + fam] = int(mk.sum())
                new.append(row); done.add((var, seed, proto, name))
                if proto == 'temporal':      # benign-shift control: fit on 80% of the training-day benign flows, FPR on the other 20% vs on Friday benign
                    bi = np.where(ytr == 0)[0]; perm = np.random.RandomState(seed).permutation(len(bi)); k = int(0.8 * len(bi)); fit_i, hold_i = bi[perm[:k]], bi[perm[k:]]
                    s_fit, (s_hold, s_fri) = DETS[name](Ztr[fit_i], [Ztr[hold_i], Zte[yte == 0]], seed); t2 = np.percentile(s_fit, 99)
                    newc.append(dict(variant=var, seed=seed, model=name, fpr_heldout_train_days=round(float((s_hold > t2).mean()), 4), fpr_friday_benign=round(float((s_fri > t2).mean()), 4), n_hold=len(hold_i), n_fri_benign=int((yte == 0).sum())))
            pd.DataFrame(new).to_csv(RES, mode='a', header=not os.path.exists(RES), index=False)
            if newc: pd.DataFrame(newc).to_csv(CTL, mode='a', header=not os.path.exists(CTL), index=False)
            print(var, 'seed', seed, proto, '| elapsed %.0fs' % (time.time() - t0)); del Ztr, Zte, Zb; gc.collect()
    cic = cic_full
R = pd.read_csv(RES); C = pd.read_csv(CTL)
def ci(v):
    v = np.asarray(v, float); k = len(v); m = v.mean(); se = v.std(ddof=1) / np.sqrt(k); t = stats.t.ppf(0.975, k - 1); return m, m - t * se, m + t * se
print('\n## E2 benign-only baseline. Runs completed: %d of 80 | libraries: sklearn %s, scipy %s, pandas %s' % (len(R), sklearn.__version__, scipy.__version__, pd.__version__))
print('| variant | detector | F1 random | F1 temporal | gap pp | 95% CI (t, df 9) | Wilcoxon p | recall rnd/tmp | FPR rnd/tmp | ROC-AUC rnd/tmp | PR-AUC rnd/tmp | never-alert F1 rnd/tmp |'); print('|---|---|---|---|---|---|---|---|---|---|---|---|')
for var in ['captured', 'dedup_ports']:
    for mn in DETS:
        g = R[(R.variant == var) & (R.model == mn)]; a = g[g.proto == 'random'].set_index('seed'); b = g[g.proto == 'temporal'].set_index('seed'); s = a.index.intersection(b.index)
        if len(s) < 2: continue
        d = (a.f1_macro.loc[s] - b.f1_macro.loc[s]) * 100; m, lo, hi = ci(d); pw = stats.wilcoxon(d, method='exact').pvalue
        f = lambda c: '%.3f/%.3f' % (a[c].loc[s].mean(), b[c].loc[s].mean())
        print('| %s | %s | %.4f | %.4f | %+.2f | [%+.2f, %+.2f] | %.5f | %s | %s | %s | %s | %s |' % (var, mn, a.f1_macro.loc[s].mean(), b.f1_macro.loc[s].mean(), m, lo, hi, pw, f('recall'), f('fpr'), f('roc_auc'), f('pr_auc'), f('never_alert_f1')))
print('\n### Per-Friday-family attack recall at the 99th-percentile threshold (mean over seeds; temporal | random)')
print('| variant | detector | Bot tmp | DDoS tmp | PortScan tmp | Bot rnd | DDoS rnd | PortScan rnd |'); print('|---|---|---|---|---|---|---|---|')
for var in ['captured', 'dedup_ports']:
    for mn in DETS:
        g = R[(R.variant == var) & (R.model == mn)]; t_ = g[g.proto == 'temporal']; r_ = g[g.proto == 'random']
        print('| %s | %s | %.3f | %.3f | %.3f | %.3f | %.3f | %.3f |' % (var, mn, t_.recall_Bot.mean(), t_.recall_DDoS.mean(), t_.recall_PortScan.mean(), r_.recall_Bot.mean(), r_.recall_DDoS.mean(), r_.recall_PortScan.mean()))
print('\n### Benign-shift control (detector fit on 80% of Mon-Thu benign, threshold = its 99th percentile; FPR on held-out 20% Mon-Thu benign vs on Friday benign; mean over seeds)')
print('| variant | detector | FPR held-out Mon-Thu benign | FPR Friday benign | runs |'); print('|---|---|---|---|---|')
for var in ['captured', 'dedup_ports']:
    for mn in DETS:
        g = C[(C.variant == var) & (C.model == mn)]; print('| %s | %s | %.4f | %.4f | %d |' % (var, mn, g.fpr_heldout_train_days.mean(), g.fpr_friday_benign.mean(), len(g)))
print('\nSaved ->', RES, 'and', CTL)


sklearn 1.6.1 | lightgbm 4.6.0 | xgboost 3.4.1 | imblearn 0.14.2 | scipy 1.16.3 | pandas 2.2.3
removed 307078 of 2827876 (10.86%)
captured rows 2827876 | duplicate rows (ports in key) 307078
captured seed 42 random | elapsed 22s
captured seed 42 temporal | elapsed 68s
captured seed 43 random | elapsed 89s
captured seed 43 temporal | elapsed 134s
captured seed 44 random | elapsed 155s
captured seed 44 temporal | elapsed 200s
captured seed 45 random | elapsed 224s
captured seed 45 temporal | elapsed 271s
captured seed 46 random | elapsed 290s
captured seed 46 temporal | elapsed 335s
captured seed 47 random | elapsed 355s
captured seed 47 temporal | elapsed 401s
captured seed 48 random | elapsed 423s
captured seed 48 temporal | elapsed 468s
captured seed 49 random | elapsed 490s
captured seed 49 temporal | elapsed 537s
captured seed 50 random | elapsed 557s
captured seed 50 temporal | elapsed 599s
captured seed 51 random | elapsed 619s
captured seed 51 temporal | elapsed 664s
dedup_ports 